# ASTRA · Notebook 05 — Benchmark, family 3 of 4: diffusion (DDPM)

**CS787 (Generative AI) project, Group 4.** One of four notebooks (03 GAN, 04 VAE, 05 diffusion, 06 flow matching) that train four kinds of generative models for variable-star light curves **under the same conditions** and score them with the same harness.

**This notebook's model: a conditional denoising diffusion model** (DDPM, Ho et al. 2020). Diffusion is not in the course syllabus, so here is the idea in full:
- **Forward process (no learning).** Take a real light curve and add a little Gaussian noise, 1000 times. After the last step nothing of the curve is left: it is pure noise. The noisy curve at any step t can be written in one line: x_t = √ᾱ_t · x_0 + √(1 − ᾱ_t) · ε, where ε is the noise and ᾱ_t falls from 1 to 0.
- **Training.** Pick a real curve and a step t at random, make x_t, and train a network to predict the noise ε that was added (mean squared error). The network also sees the condition: phases, amplitude, class.
- **Sampling.** Start from pure noise and remove the predicted noise step by step. We use the DDIM sampler, which can skip steps: a new curve costs 32 network passes here instead of 1000.
- Compared with a VAE, there is no encoder and no small latent vector: the "latent" is a noise curve of the same size as the data. Compared with a GAN, there is no second network to fight: the loss is a plain regression.

**Jobs.** The search runs the six configurations of `GRID["ddpm"]` in `astra/bench_grid.py` (ε- or v-prediction × learning rate 1e-4, 2e-4, 4e-4) on split 0; the final stage runs the winner (`FINAL["ddpm"]`) on splits 0–4, each run fresh, and sweeps the sampling steps and guidance at its best checkpoint.

## The benchmark protocol (identical in notebooks 03–06)

**Question.** Do diffusion and flow matching give better synthetic light curves than a GAN or a VAE for training a classifier, especially for the rare classes, when all four are tuned, trained to their best point and scored in the same way? And what does each family cost in compute?

| What | Setting | Where it comes from |
|---|---|---|
| Data | **Catalina only** (6 classes, 100 points per curve), pre-processed as in notebook 01 | García-Jara et al. 2022; one dataset by group decision (15 Oct) |
| What a model generates | the magnitudes of one curve, given a real star's phases, amplitude and class | the paper |
| Target | the amplitude-normalised curve, magnitude / amplitude, in [−2, 2]; the amplitude is given as log₁₀, standardised | ours |
| Split | over **stars**, stratified by class: 70% train, 10% validation, 20% test; five splits (seeds 0–4); the search uses split 0 | ours |
| Batches | the paper's resampling block with γ = 0.25 (rare classes drawn more often than their share) | the paper |
| Search | 6 configurations per family on split 0, scored on **validation** stars only; the best goes to the final stage | ours |
| Training length | until 4 checkpoints in a row bring no new best validation score (at least 1 h), at most 6 h in the search and 20 h in the final stage | ours |
| Checkpoints | after 5, 10, 20, 30, 45, 60, 90, 120, 180 minutes, then every hour (every 2 h after 10 h); the final stage also after 1, 2, 3, 7.5 and 15 minutes (the diffusion-type models peak within 10 minutes, then copy); each is the EMA (0.999) of the weights | EMA: the paper |
| Hours | always "hours of training on one T4 alone": iterations ÷ the configuration's measured speed, so sharing a GPU changes nothing | ours |
| Model selection | the checkpoint with the best **validation** TSTR balanced accuracy | the paper's row G, on our metric |
| Compute-matched view | the best checkpoint within 5, 10, 15, 30, 60 minutes and 2, 5, 10 h is also scored on test | ours |
| New conditions | TSTR for conditions no training star has (one star's phases, another's amplitude, same class), at the selected and at the last checkpoint | ours |
| Classifier | the paper's CNN, 15 runs × an ensemble of 5, settings of notebook 01 | the paper + notebook 01 |
| Size | generator / decoder / denoising network: about 4 M parameters each | ours |

**Scores.**
- **TSTR**: the classifier is trained on synthetic curves only and tested on real test stars.
- **Real + synthetic**: every training curve is synthetic with probability 0.25, 0.5 or 0.75; the mix is chosen on validation stars.
- **Real only**: rows A–D of the paper's Table 5 (oversampling, soft warping) on the same split. Augmentation has to beat row A.
- The headline number is **balanced accuracy**, the mean of the per-class recalls. Macro-F1, accuracy and per-class recall are reported too, and recall is also averaged over the **rare tier** (classes with fewer than 1,000 training curves: LPV, DSCT, CEP).
- Uncertainty: the ± of a row is the standard deviation over the 15 classifier runs. Differences between two rows come with a 95% interval from a paired bootstrap over test stars (1,000 resamples). Across splits: mean ± std of 3.
- Fidelity of the synthetic curves in a classifier's feature space: per-class density and precision (do they look real?), recall and coverage (do they cover the variety of the real ones?), and FID.
- **Memorisation**, next to every classifier score (ours). For each generated curve: its distance to the real curve of the star whose condition it was given. The *ratio* compares training stars with validation stars (1 = none). The *copy share* is the share of training-star curves closer to their source than 95% of the validation-star curves are to theirs: 5% = no copying, 100% = every curve a copy. Notebook 01 found late GAN checkpoints copying their rare-class training curves.

**How to run** (the steps per account are in `notebooks/RUN_INSTRUCTIONS.md`)
- Accelerator **GPU T4 ×2** · Internet **On**.
- **SMOKE** first (`SMOKE = True`, about 30 minutes): tests, a tiny search and a tiny final stage, the speed of every grid configuration, and a test of two jobs per GPU. It must end with `ALL DELIVERABLES PRESENT`; the search measures its own speeds.
- **Search** (`STAGE = "search"`, `SMOKE = False`): six jobs on `SLOTS_PER_GPU` × 2 slots. **Final** (`STAGE = "final"`, after the search's winner and its speed are set in `astra/bench_grid.py` and the notebook rebuilt): no input needed; every split trains fresh in `runs_<family>_<dataset>_final`.
- **Next session** (if the last cell lists unfinished jobs): *Add Input* → this notebook's previous version, `SESSION` + 1, Save & Run All. Finished jobs are skipped; paused ones continue.
- **The four-way comparison:** attach the final outputs of the other benchmark notebooks; the results section then adds the comparison tables.

In [ ]:
# ---- Configuration ----
import os, sys, json, time, shutil, subprocess
from pathlib import Path

FAMILY = "ddpm"   # this notebook's model family (astra/bench_ddpm.py)
SMOKE = True            # True: tests, tiny search and final jobs, speeds (run first); False: a real run
STAGE = "search"        # "search": every grid configuration on split 0 (validation only); "final": the winner on
                        # FINAL_SPLITS with test scores (fresh runs: no search output needed)
DATASET = "catalina"    # the benchmark uses Catalina only (user decision of 2026-10-15)
SESSION = 1             # 1, 2, 3, ... only labels this run's summary
SESSION_HOURS = 10.5    # time limit of the training (jobs pause ~25 min before it); lower it if the quota is short
SLOTS_PER_GPU = "auto"  # jobs sharing one GPU: "auto" = measured in the first session (2 if it pays), or 1 / 2
SEARCH_CONFIGS = "all"  # "all", or a list of grid keys (to split one family's search over two accounts)
FINAL_SPLITS = [0, 1, 2, 3, 4]   # final stage: the splits this account runs (to split them over accounts)

# Jobs (the grid, the winner and the speeds are in astra/bench_grid.py). Search: one job per grid configuration on
# split 0. Final: the winner on FINAL_SPLITS plus the reference rows real_s<k> (A = oversampling, B-D = warping).
GRID_TAGS = ['eps_lr1e-4', 'eps_lr2e-4', 'eps_lr4e-4', 'v_lr1e-4', 'v_lr2e-4', 'v_lr4e-4']
FINAL_TAG = 'eps_lr1e-4'
SEARCH_JOBS = [f"{FAMILY}_{t}_s0" for t in (GRID_TAGS if SEARCH_CONFIGS == "all" else SEARCH_CONFIGS)]
FINAL_JOBS = [f"{FAMILY}_{FINAL_TAG}_s{k}" for k in FINAL_SPLITS] if FINAL_TAG else []
JOBS_ALL = SEARCH_JOBS if STAGE == "search" else FINAL_JOBS + [f"real_s{k}" for k in FINAL_SPLITS]


def make_queues(jobs, slots_per_gpu):
    """Slots 0a, 0b run on GPU 0 and 1a, 1b on GPU 1; jobs are dealt out in turn (the longest are listed first)."""
    slots = [f"{g}{chr(97 + k)}" for g in "01" for k in range(slots_per_gpu)]
    return {s: jobs[i::len(slots)] for i, s in enumerate(slots) if jobs[i::len(slots)]}

L = {"catalina": 100, "ztf": 40}[DATASET]     # points per curve
CLASS_RANGE_K, DATA_SEED = 2.0, 0             # Catalina pre-processing, as in notebook 01

LOCAL_TEST = os.environ.get("ASTRA_LOCAL_TEST") == "1"     # only my offline test run sets this
RUN_STARTED = time.time()
WORK = Path(os.environ.get("ASTRA_WORK", "/kaggle/working"))
INPUT = Path(os.environ.get("ASTRA_INPUT", "/kaggle/input"))
TAG = f"{FAMILY}_{DATASET}"
# one runs folder per stage, so final runs start fresh (smoke outputs never mix with the real ones)
STAGE_RUNS = {s: WORK / (f"runs_{TAG}" + ("_final" if s == "final" else "") + ("_smoke" if SMOKE else ""))
              for s in ("search", "final")}
RUNS = STAGE_RUNS[STAGE]
RESULTS = WORK / (f"results_{DATASET}" + ("_smoke" if SMOKE else ""))   # one file per finished job (all notebooks alike)
FIGS, TABLES = WORK / f"figures_{TAG}", WORK / f"tables_{TAG}"
for d in (RUNS, RESULTS, FIGS, TABLES, WORK / "data", WORK / "astra"):
    d.mkdir(parents=True, exist_ok=True)
os.chdir(WORK)
sys.path.insert(0, str(WORK))
assert STAGE in ("search", "final") and (STAGE == "search" or FINAL_JOBS or SMOKE), \
    "STAGE = 'final' needs the search's winner: FINAL in astra/bench_grid.py, then a rebuilt notebook"
assert SEARCH_CONFIGS == "all" or set(SEARCH_CONFIGS) <= set(GRID_TAGS), f"unknown grid keys; choose from {GRID_TAGS}"
SEEDS = [0, 1] if SMOKE else sorted({int(j.rsplit("_s", 1)[1]) for j in JOBS_ALL})   # SMOKE: tiny search + final
print(f"{FAMILY} on {DATASET}: {STAGE} stage, session {SESSION} ({SESSION_HOURS} h), "
      f"{'SMOKE' if SMOKE else 'REAL RUN'} | {len(JOBS_ALL)} jobs: {JOBS_ALL}")

In [ ]:
# ---- GPUs, and restoring the previous session's output ----
import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt
print("torch", torch.__version__, "| GPUs:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
assert LOCAL_TEST or torch.cuda.device_count() >= 2, "Set Accelerator to GPU T4 x2"
DEV = "cuda" if torch.cuda.is_available() else "cpu"


def restore_previous():
    """Copies the folders of an attached previous version of THIS notebook and stage (recognised by this
    stage's runs folder), so finished jobs are skipped and unfinished ones resume. Files already present are
    kept. Outputs of the other benchmark notebooks, and a search output attached to a final session, are not
    copied: the results section reads them where they are."""
    n = 0
    for runs in INPUT.glob(f"*/**/{RUNS.name}"):
        if not runs.is_dir() or WORK in runs.parents:
            continue
        for name in (RUNS.name, "data", f"results_{DATASET}", FIGS.name, TABLES.name):
            root = runs.parent / name
            for src in (root.rglob("*") if root.is_dir() else []):
                dest = WORK / name / src.relative_to(root)
                if src.is_file() and not dest.exists():
                    dest.parent.mkdir(parents=True, exist_ok=True)
                    shutil.copy2(src, dest)
                    n += 1
    return n


n = restore_previous()
print(f"restored {n} file(s) from the previous session" if n else "fresh start (nothing to restore)")
for p in sorted(RUNS.glob("*/status.json")):
    print(f"  {p.parent.name:<14} {json.loads(p.read_text())}")

## The `astra` package

The code lives in small modules written to `/kaggle/working/astra/` by the next cells, so that the GPU workers (separate processes, one per GPU) can import it. The first group is shared with notebook 01 and unchanged; the `bench_*` modules are the benchmark.
- `common`, `catalina`, `ztf`, `sampling`, `classifier`, `warping`, `metrics`: paths and the time guard, the two datasets, the γ-resampling block, the paper's CNN (75 copies trained at once), the soft-warping baselines, density / recall / FID.
- `bench_data`: the split over stars and the amplitude-normalised target.
- `bench_eval`: **the harness**, the same file in all four notebooks: the classifier fits, balanced accuracy, fidelity, the memorisation ratio, the evaluation of a checkpoint and the final test scores.
- `bench_run`: the trainer with the time budget, checkpoints, resume and model selection, and the per-GPU worker.
- `bench_net`: the 1-D U-Net shared by diffusion and flow matching.
- `bench_ddpm`: **this notebook's model**: the noise schedule, the training step and the DDIM sampler.
- `bench_report`: tables, paired bootstrap and figures.

In [ ]:
%%writefile astra/__init__.py
"""ASTRA: generative models for variable-star light curves (CS787 project, Group 4)."""


In [ ]:
%%writefile astra/common.py
"""Small helpers shared by every notebook: seeds, paths, time guard, safe saving, logging."""
import json
import os
import random
import time
import urllib.request
from pathlib import Path

import numpy as np
import torch

WORK = Path(os.environ.get("ASTRA_WORK", "/kaggle/working"))
INPUT = Path(os.environ.get("ASTRA_INPUT", "/kaggle/input"))


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def save_json(obj, path):
    """Write to a temp file first, so a kill mid-write never leaves a broken file."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(json.dumps(obj, indent=1, default=float), encoding="utf-8")
    os.replace(tmp, path)


def load_json(path, default=None):
    path = Path(path)
    return json.loads(path.read_text(encoding="utf-8")) if path.exists() else default


def torch_save(obj, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    torch.save(obj, tmp)
    os.replace(tmp, path)


def restore_rng(state, device):
    """Put back the random-number state saved in a resumable checkpoint. torch.load(map_location=GPU) moves
    the saved states to the GPU, but the generators only accept CPU tensors, so they are moved back first.
    Never fatal: without it a resumed run just continues with fresh random numbers. Returns True on success."""
    try:
        torch.set_rng_state(state["rng_cpu"].cpu())
        if state.get("rng_cuda") is not None and str(device).startswith("cuda"):
            torch.cuda.set_rng_state_all([s.cpu() for s in state["rng_cuda"]])
        return True
    except Exception as e:
        print(f"could not restore the random-number state ({type(e).__name__}: {e}); continuing with a fresh one",
              flush=True)
        return False


class Deadline:
    """Session time guard: Kaggle kills a session at 12 h, we stop cleanly before that."""

    def __init__(self, deadline_unix):
        self.t = float(deadline_unix)

    def left(self):
        return self.t - time.time()

    def passed(self, reserve_s=0.0):
        return self.left() < reserve_s


class Logger:
    def __init__(self, path):
        self.path = Path(path)
        self.path.parent.mkdir(parents=True, exist_ok=True)

    def __call__(self, *msg):
        line = time.strftime("%H:%M:%S ") + " ".join(str(m) for m in msg)
        print(line, flush=True)
        with open(self.path, "a", encoding="utf-8") as f:
            f.write(line + "\n")


def find_input(name):
    """A file attached under /kaggle/input (e.g. a previous version's output), if any."""
    hits = sorted(INPUT.glob(f"**/{name}")) if INPUT.exists() else []
    return hits[0] if hits else None


def download(url, dest, tries=5):
    """Download once; reuse a copy attached under /kaggle/input if there is one."""
    dest = Path(dest)
    if dest.exists() and dest.stat().st_size > 0:
        return dest
    attached = find_input(dest.name)
    if attached is not None:
        return attached
    dest.parent.mkdir(parents=True, exist_ok=True)
    for i in range(tries):
        try:
            tmp = dest.with_suffix(dest.suffix + ".part")
            urllib.request.urlretrieve(url, tmp)
            os.replace(tmp, dest)
            return dest
        except Exception as e:  # network hiccup: wait and retry
            print(f"download failed ({e}), retry {i + 1}/{tries}")
            time.sleep(10 * (i + 1))
    raise RuntimeError(f"could not download {url}")


def peak_gpu_mem_gb():
    if not torch.cuda.is_available():
        return 0.0
    return torch.cuda.max_memory_allocated() / 1e9


In [ ]:
%%writefile astra/catalina.py
"""Catalina Surveys DR1 periodic variables, pre-processed exactly as in Garcia-Jara et al. (2022), Sec. 3.

Steps: period folding -> outlier filtering (sliding z-score, then a class-range filter) -> keep curves with
at least L points -> random sub-sampling to L points -> median centering. Then the paper's split: validation
and test each hold 20% of the smallest class, for every class (Sec. 4.1).
"""
from pathlib import Path

import numpy as np
import pandas as pd

from .common import download

CATALOG_URL = "http://nesssi.cacr.caltech.edu/DataRelease/CatalinaVars.tbl"
PHOT_URL = "http://nesssi.cacr.caltech.edu/DataRelease/AllVar.phot.gz"

# Table 1 of the paper: Drake et al. (2014) type id -> new class
CLASS_MAP = {1: "EBSD/D", 3: "EBSD/D",                      # EW, beta Lyrae
             4: "RRL", 5: "RRL", 6: "RRL", 7: "RRL",        # RRab, RRc, RRd, Blazhko
             2: "EBC",                                      # EA
             13: "LPV",
             11: "DSCT", 12: "DSCT",                        # HADS, LADS
             9: "CEP", 10: "CEP"}                           # ACEP, Cep-II
CLASSES = ["EBSD/D", "RRL", "EBC", "LPV", "DSCT", "CEP"]
PAPER_COUNTS = {"EBSD/D": 28980, "RRL": 7533, "EBC": 4500, "LPV": 483, "DSCT": 241, "CEP": 182}   # Table 2
TRAIN, VAL, TEST = 0, 1, 2


def read_catalog(path):
    cols = ["css_id", "numid", "ra", "dec", "vmag", "period", "amp_cat", "nobs", "vartype"]
    cat = pd.read_csv(path, sep=r"\s+", comment="#", header=None, names=cols)
    cat = cat[cat.vartype.isin(list(CLASS_MAP))].copy()
    cat["period"] = pd.to_numeric(cat.period)          # "\nodata" only occurs in type 17, dropped above
    cat["cls"] = cat.vartype.map(CLASS_MAP)
    return cat.sort_values("numid").reset_index(drop=True)


def read_photometry(path, numids):
    """All (MJD, mag, err) rows of the wanted objects, grouped per object."""
    ph = pd.read_csv(path, header=None, usecols=[0, 1, 2, 3], names=["numid", "mjd", "mag", "err"],
                     dtype={"numid": np.int64, "mjd": np.float64, "mag": np.float64, "err": np.float64})
    ph = ph[ph.numid.isin(set(numids))].sort_values(["numid", "mjd"], kind="stable")
    ids, start = np.unique(ph.numid.values, return_index=True)
    end = np.append(start[1:], len(ph))
    t, m, e = ph.mjd.values, ph.mag.values, ph.err.values
    return {i: (t[a:b], m[a:b], e[a:b]) for i, a, b in zip(ids, start, end)}


def fold(t, period):
    """Eqs. 8-9: phase = (t mod T) / T, in [0, 1)."""
    return np.mod(t, period) / period


def zscore_keep(m, ws=20, z=3.0, passes=2):
    """Sliding-window z-score filter on a phase-sorted curve (window 20, drop z > 3, two passes)."""
    keep = np.ones(len(m), bool)
    for _ in range(passes):
        s = pd.Series(m[keep])
        mu = s.rolling(ws, center=True, min_periods=2).mean().values
        sd = s.rolling(ws, center=True, min_periods=2).std().values
        with np.errstate(invalid="ignore", divide="ignore"):
            bad = np.abs(s.values - mu) / sd > z
        idx = np.flatnonzero(keep)
        keep[idx[bad]] = False
    return keep


def preprocess(data_dir, L=100, class_range_k=3.0, seed=0, log=print):
    """Returns a dict of arrays (one row per kept star) plus a per-class count table."""
    data_dir = Path(data_dir)
    cat = read_catalog(download(CATALOG_URL, data_dir / "CatalinaVars.tbl"))
    log(f"catalog: {len(cat):,} stars in the 6 classes")
    phot = read_photometry(download(PHOT_URL, data_dir / "AllVar.phot.gz"), cat.numid.values)
    log(f"photometry found for {len(phot):,} of them")

    # 1-2. fold, sort by phase, sliding z-score filter
    curves = []
    for r in cat.itertuples():
        if r.numid not in phot:
            continue
        t, m, e = phot[r.numid]
        ph = fold(t, r.period)
        o = np.argsort(ph, kind="stable")
        ph, m, e = ph[o], m[o], e[o]
        k = zscore_keep(m)
        curves.append((r.numid, r.cls, r.period, ph[k], m[k], e[k]))

    # 2b. class-range filter: drop a curve if > 90% of its magnitudes lie outside median +- k*std of its class
    stats = {}
    for c in CLASSES:
        allm = np.concatenate([cv[4] for cv in curves if cv[1] == c])
        stats[c] = (np.median(allm), np.std(allm))
    kept, n_range, n_short = [], 0, 0
    for cv in curves:
        med, sd = stats[cv[1]]
        if np.mean(np.abs(cv[4] - med) > class_range_k * sd) > 0.9:
            n_range += 1
        elif len(cv[4]) < L:       # 3. need at least L points to sub-sample
            n_short += 1
        else:
            kept.append(cv)
    log(f"dropped {n_range} by the class-range filter and {n_short} with fewer than {L} points")

    # 4-5. random sub-sampling to L points (kept in phase order), then median centering
    rng = np.random.default_rng(seed)
    n = len(kept)
    out = dict(numid=np.zeros(n, np.int64), y=np.zeros(n, np.int64), period=np.zeros(n),
               phase=np.zeros((n, L), np.float32), mag=np.zeros((n, L), np.float32),
               err=np.zeros((n, L), np.float32), amp=np.zeros(n, np.float32))
    for i, (numid, c, period, ph, m, e) in enumerate(kept):
        s = np.sort(rng.choice(len(m), L, replace=False))
        x = m[s] - np.median(m[s])
        out["numid"][i], out["y"][i], out["period"][i] = numid, CLASSES.index(c), period
        out["phase"][i], out["mag"][i], out["err"][i] = ph[s], x, e[s]
        out["amp"][i] = (x.max() - x.min()) / 2          # half peak-to-peak of the L points

    # split (Sec. 4.1): val and test each get 20% of the smallest class, for every class
    counts = np.bincount(out["y"], minlength=len(CLASSES))
    n_hold = int(0.2 * counts.min())
    split = np.full(n, TRAIN, np.int8)
    for c in range(len(CLASSES)):
        idx = rng.permutation(np.flatnonzero(out["y"] == c))
        split[idx[:n_hold]] = VAL
        split[idx[n_hold:2 * n_hold]] = TEST
    out["split"] = split
    out["classes"] = np.array(CLASSES)
    table = pd.DataFrame({"class": CLASSES, "ours": counts,
                          "paper": [PAPER_COUNTS[c] for c in CLASSES]})
    table["ours/paper"] = (table.ours / table.paper).round(3)
    table["train"] = [int(((out["y"] == c) & (split == TRAIN)).sum()) for c in range(len(CLASSES))]
    table["val"] = n_hold
    table["test"] = n_hold
    return out, table


In [ ]:
%%writefile astra/ztf.py
"""The ZTF dataset of Garcia-Jara et al. (2022), rebuilt from public sources.

What the paper states (Sec. 3): the periodic variable stars of ALeRCE's ZTF training set (Sanchez-Saez et al.
2021; classes EB, RRL, LPV, DSCT, CEP), light curves "processed by the ALeRCE broker", anomalous observations
(marked with magnitude 100) removed with a threshold of 30, folded with the provided period, 40 randomly
chosen observations per curve, median-centred. Table 2 gives the class counts after pre-processing.

What it does not state: the photometric band, the date of the data, the period's origin, how stars were
selected. We recovered these from the paper itself (notes/ztf_exact_dataset_leads.md): the real light curves
plotted in the paper and in the first author's thesis were digitised and matched to ALeRCE's detections, which
identified the 21 stars shown. They tell us that
  - a light curve is the detections of one star in ONE band, and the dataset holds g curves and r curves
    (14 and 7 of the 21);
  - the period is the `period` column of ALeRCE's internal label file, i.e. the period of the catalogue the
    label came from (equal to 1e-5 for all 21), with phase = (MJD mod T) / T. Most Gaia-labelled stars have no
    period there (78% of the RR Lyrae, 74% of the LPVs), cannot be folded, and drop out;
  - no plotted point is later than MJD 59048.3 (2020-07-19), and many are that late: one snapshot of that date;
  - only the magnitude threshold is applied; the 40 points are centred on their own median.
With one curve per (star, band), stars that have a period, detections up to that date and repeated epochs
dropped, "at least 40 points" gives EB 31,174 / RRL 18,748 / LPV 5,470, against Table 2's 31,477 / 18,729 /
5,245. DSCT and CEP come out at 768 and 716 against 507 and 471, which we cannot explain. So:
  - light curves = ALeRCE detections (TAP table ztf.detection), corrected magnitudes magpsf_corr < 30, per band;
  - keep curves with at least L points, then draw at random exactly the paper's number of curves per class
    (all of them where fewer qualify: EB), so the class imbalance and the split sizes match Table 2;
  - split as in Sec. 4.1: validation and test each hold 20% of the smallest class, for every class. As in the
    paper the split is over curves, so the g and the r curve of a star can fall on different sides.
"""
import numpy as np
import pandas as pd

CLASSES = ["EB", "RRL", "LPV", "DSCT", "CEP"]                    # the paper's names, in Table 2 order
ALERCE_NAME = {"EB": "E", "RRL": "RRL", "LPV": "LPV", "DSCT": "DSCT", "CEP": "CEP"}
PAPER_COUNTS = {"EB": 31477, "RRL": 18729, "LPV": 5245, "DSCT": 507, "CEP": 471}
TRAIN, VAL, TEST = 0, 1, 2
MAG_THR = 30.0
MJD_MAX = 59048.4                                                 # 2020-07-19
BANDS = (1, 2)                                                    # g, r


def preprocess(raw_npz, L=40, bands=BANDS, seed=0, match_paper_counts=True, period_key="period_label",
               mjd_max=MJD_MAX, log=print):
    """raw_npz: the file written by build/ztf/download_ztf.py and add_label_periods.py. bands: 1 = g, 2 = r.
    period_key: "period_label" (the label file's catalogue period) or "period" (ALeRCE's Multiband_period,
    which is half the orbital period for most eclipsing binaries).
    mjd_max: the last date of the detections used (None = all of them)."""
    z = np.load(raw_npz)
    period, cls = z[period_key], z["cls"]
    ok = np.isin(z["fid"], bands) & (z["mag"] < MAG_THR)
    if mjd_max is not None:
        ok &= z["mjd"] <= mjd_max
    # one light curve per (star, band): sort its detections by time and drop repeated epochs
    cur, mjd, mag, err = z["obj"][ok].astype(np.int64) * 4 + z["fid"][ok], z["mjd"][ok], z["mag"][ok], z["err_ext"][ok]
    order = np.lexsort((mjd, cur))
    cur, mjd, mag, err = cur[order], mjd[order], mag[order], err[order]
    new = np.r_[True, (cur[1:] != cur[:-1]) | (mjd[1:] != mjd[:-1])]
    cur, mjd, mag, err = cur[new], mjd[new], mag[new], err[new]
    ids, start, cnt = np.unique(cur, return_index=True, return_counts=True)
    star, band = ids // 4, ids % 4
    good = (cnt >= L) & np.isfinite(period[star]) & (period[star] > 0)
    rng = np.random.default_rng(seed)
    chosen, qualify = [], {}
    for c in CLASSES:
        idx = np.flatnonzero(good & (cls[star] == ALERCE_NAME[c]))
        qualify[c] = len(idx)
        k = min(PAPER_COUNTS[c], len(idx)) if match_paper_counts else len(idx)
        chosen.append(np.sort(rng.choice(idx, k, replace=False)))
    y = np.concatenate([np.full(len(v), i) for i, v in enumerate(chosen)])
    chosen = np.concatenate(chosen)
    n = len(chosen)
    out = dict(oid=z["oid"][star[chosen]], band=band[chosen].astype(np.int8), y=y, period=period[star[chosen]],
               phase=np.zeros((n, L), np.float32), mag=np.zeros((n, L), np.float32),
               err=np.zeros((n, L), np.float32), amp=np.zeros(n, np.float32))
    for i, k in enumerate(chosen):
        s = rng.choice(cnt[k], L, replace=False) + start[k]       # L random observations
        ph = np.mod(mjd[s], out["period"][i]) / out["period"][i]  # Eqs. 8-9
        o = np.argsort(ph, kind="stable")
        x = mag[s][o] - np.median(mag[s])                         # median centring
        out["phase"][i], out["mag"][i], out["err"][i] = ph[o], x, err[s][o]
        out["amp"][i] = (x.max() - x.min()) / 2
    counts = np.bincount(y, minlength=len(CLASSES))
    n_hold = int(0.2 * counts.min())
    split = np.full(n, TRAIN, np.int8)
    for c in range(len(CLASSES)):
        idx = rng.permutation(np.flatnonzero(y == c))
        split[idx[:n_hold]] = VAL
        split[idx[n_hold:2 * n_hold]] = TEST
    out["split"] = split
    out["classes"] = np.array(CLASSES)
    has_p = np.isfinite(period) & (period > 0)
    table = pd.DataFrame({"class": CLASSES, "labeled stars": [int(np.sum(cls == ALERCE_NAME[c])) for c in CLASSES],
                          "with a period": [int(np.sum((cls == ALERCE_NAME[c]) & has_p)) for c in CLASSES],
                          f"curves >= {L} points": [qualify[c] for c in CLASSES], "ours": counts,
                          "paper": [PAPER_COUNTS[c] for c in CLASSES]})
    for b, name in ((1, "g"), (2, "r")):
        table[name] = [int(((y == c) & (out["band"] == b)).sum()) for c in range(len(CLASSES))]
    table["train"] = [int(((y == c) & (split == TRAIN)).sum()) for c in range(len(CLASSES))]
    table["val"] = n_hold
    table["test"] = n_hold
    log(f"{n:,} light curves of {L} points ({int((out['band'] == 1).sum()):,} in g, {int((out['band'] == 2).sum()):,} in r)")
    return out, table


In [ ]:
%%writefile astra/sampling.py
"""The resampling block of Garcia-Jara et al. (2022), Sec. 4.6.

Each class keeps its own shuffled queue. A class is picked with probability
    p_i = gamma / N + (1 - gamma) * S_i / S          (Eq. 12)
and the next sample is taken from its queue; an empty queue is reshuffled.
gamma = 0 keeps the original class mix, gamma = 1 gives a uniformly balanced stream.
"""
import numpy as np


class ClassResampler:
    def __init__(self, y, gamma, seed=0, n_classes=None):
        y = np.asarray(y)
        self.n_classes = n_classes or int(y.max()) + 1
        self.rng = np.random.default_rng(seed)
        self.pools = [np.flatnonzero(y == c) for c in range(self.n_classes)]
        sizes = np.array([len(p) for p in self.pools], float)
        self.p = gamma / self.n_classes + (1 - gamma) * sizes / sizes.sum()
        self.p[sizes == 0] = 0
        self.p /= self.p.sum()
        self.queues = [self.rng.permutation(p) for p in self.pools]
        self.pos = np.zeros(self.n_classes, int)

    def _take(self, c, k):
        out = []
        while k > 0:
            if self.pos[c] == len(self.queues[c]):
                self.queues[c] = self.rng.permutation(self.pools[c])
                self.pos[c] = 0
            n = min(k, len(self.queues[c]) - self.pos[c])
            out.append(self.queues[c][self.pos[c]:self.pos[c] + n])
            self.pos[c] += n
            k -= n
        return np.concatenate(out)

    def sample(self, batch):
        """Indices of one batch (classes drawn with p, samples without replacement inside a class)."""
        cls = self.rng.choice(self.n_classes, size=batch, p=self.p)
        idx = np.empty(batch, np.int64)
        for c in np.unique(cls):
            where = np.flatnonzero(cls == c)
            idx[where] = self._take(c, len(where))
        return idx


In [ ]:
%%writefile astra/classifier.py
"""The downstream classifier of Garcia-Jara et al. (2022), Sec. 4.3 and Table 3.

Base classifier: 5 conv blocks (Conv k=3 + BatchNorm, MaxPool 2 + ReLU), channels 2->32->64->128->64->64,
then Dense -> 100, BN + ReLU + Dropout, Dense -> N. Input = magnitudes and phases (2 channels).
The classifier is an ensemble of 5 base CNNs, and every score is repeated over 15 independent runs,
i.e. 75 small CNNs per table row. We train them side by side as ONE network with a leading "member"
axis: every member has its own weights and its own mini-batches, so they are exactly as independent
as 75 separate trainings. Each layer is one batched matrix product over the members (a convolution
with kernel 3 is a matrix product on the 3 shifted copies of its input), which a GPU runs far faster
than 75 small networks or a grouped convolution (measured on a T4: 133 s per epoch with grouped
convolutions). Mixed precision (fp16) is used on the GPU for the same reason.
"""
import math

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from .sampling import ClassResampler


class MemberLinear(nn.Module):
    """M independent Linear layers. x: (M, B, fin) -> (M, B, fout)."""

    def __init__(self, M, fin, fout):
        super().__init__()
        bound = 1 / math.sqrt(fin)                     # same init as nn.Linear
        self.w = nn.Parameter(torch.empty(M, fin, fout).uniform_(-bound, bound))
        self.b = nn.Parameter(torch.empty(M, 1, fout).uniform_(-bound, bound))

    def forward(self, x):
        return torch.baddbmm(self.b, x, self.w)


class MemberConv(nn.Module):
    """M independent Conv1d(kernel 3, padding 1) layers. x: (M, cin, B, L) -> (M, cout, B, L)."""

    def __init__(self, M, cin, cout, k=3):
        super().__init__()
        self.k = k
        bound = 1 / math.sqrt(cin * k)                 # same init as nn.Conv1d
        self.w = nn.Parameter(torch.empty(M, cout, cin * k).uniform_(-bound, bound))
        self.b = nn.Parameter(torch.empty(M, cout, 1).uniform_(-bound, bound))

    def forward(self, x):
        M, C, B, L = x.shape
        xp = F.pad(x, (self.k // 2, self.k // 2))
        cols = torch.stack([xp[..., i:i + L] for i in range(self.k)], 2)       # (M, cin, k, B, L)
        return torch.baddbmm(self.b, self.w, cols.reshape(M, C * self.k, B * L)).view(M, -1, B, L)


class CNNEnsemble(nn.Module):
    """M copies of the paper's base classifier (Table 3), run side by side. Input (M, B, in_ch, L)."""

    def __init__(self, M, n_classes, L, in_ch=2, dropout=0.5):
        super().__init__()
        self.M, self.in_ch = M, in_ch
        chans = [in_ch, 32, 64, 128, 64, 64]
        self.convs = nn.ModuleList(MemberConv(M, a, b) for a, b in zip(chans[:-1], chans[1:]))
        self.bns = nn.ModuleList(nn.BatchNorm2d(M * b) for b in chans[1:])
        l = L
        for _ in range(5):
            l = math.ceil(l / 2)                       # ceil(L/32) as in Table 3
        self.fc1 = MemberLinear(M, 64 * l, 100)
        self.bn = nn.BatchNorm1d(M * 100)
        self.drop = nn.Dropout(dropout)
        self.fc2 = MemberLinear(M, 100, n_classes)

    def features(self, x):
        """Output of the last conv block, flattened: (M, B, 64 * ceil(L/32))."""
        h = x.transpose(1, 2)                                              # (M, C, B, L)
        for conv, bn in zip(self.convs, self.bns):
            h = conv(h)
            M, C, B, L = h.shape
            h = bn(h.reshape(1, M * C, B, L))                              # statistics over (B, L) per member-channel
            h = F.relu(F.max_pool2d(h, (1, 2), ceil_mode=True)).view(M, C, B, -1)
        return h.transpose(1, 2).reshape(h.shape[0], h.shape[2], -1)

    def forward(self, x):
        h = self.fc1(self.features(x))                                     # (M, B, 100)
        M, B, _ = h.shape
        h = self.bn(h.transpose(1, 2).reshape(1, M * 100, B)).view(M, 100, B).transpose(1, 2)
        return self.fc2(self.drop(F.relu(h)))                              # (M, B, n_classes) logits


def reference_cnn(ens, m):
    """Member m of an ensemble rebuilt from standard PyTorch layers (used only by the self-tests)."""
    layers = []
    for conv in ens.convs:
        cout, cin = conv.w.shape[1], conv.w.shape[2] // conv.k
        ref = nn.Conv1d(cin, cout, conv.k, padding=conv.k // 2)
        ref.weight.data, ref.bias.data = conv.w[m].reshape(cout, cin, conv.k).clone(), conv.b[m, :, 0].clone()
        layers += [ref, nn.BatchNorm1d(cout), nn.MaxPool1d(2, ceil_mode=True), nn.ReLU()]
    fcs = []
    for fc in (ens.fc1, ens.fc2):
        ref = nn.Linear(fc.w.shape[1], fc.w.shape[2])
        ref.weight.data, ref.bias.data = fc.w[m].T.clone(), fc.b[m, 0].clone()
        fcs.append(ref)
    return nn.Sequential(*layers, nn.Flatten(), fcs[0], nn.BatchNorm1d(100), nn.ReLU(), fcs[1])


def stack_input(x, ph):
    """(n, L) magnitudes + (n, L) phases -> (n, 2, L)."""
    return torch.stack([x, ph], dim=1)


def amp_on(amp, device):
    return bool(amp) and str(device).startswith("cuda")


def autocast(device, amp):
    """fp16 mixed precision on the GPU when amp is on; a no-op otherwise (e.g. in the offline CPU tests)."""
    on = amp_on(amp, device)
    return torch.autocast("cuda" if on else "cpu", enabled=on)


@torch.no_grad()
def predict_proba(model, X, batch=512, amp=True):
    """Softmax outputs of every member: (M, n, N). X: (n, 2, L) on the model's device."""
    model.eval()
    out = []
    for i in range(0, len(X), batch):
        xb = X[i:i + batch].unsqueeze(0).expand(model.M, -1, -1, -1)
        with autocast(X.device.type, amp):
            logits = model(xb)
        out.append(F.softmax(logits.float(), -1))
    return torch.cat(out, 1)


def macro_f1(y, pred, n_classes):
    f1 = []
    for c in range(n_classes):
        tp = np.sum((pred == c) & (y == c))
        p = tp / max(np.sum(pred == c), 1)
        r = tp / max(np.sum(y == c), 1)
        f1.append(0.0 if p + r == 0 else 2 * p * r / (p + r))
    return float(np.mean(f1))


def fit_ensembles(train, val, test, n_classes, runs=15, ens=5, max_epochs=10, patience=3, batch=128,
                  lr=1e-4, seed=0, warp=None, device="cuda", log=print, tag="", amp=True, checks_per_epoch=4):
    """Train `runs` independent ensembles of `ens` CNNs and score each run on val and test.

    train/val/test: dicts of torch tensors x (n, L), ph (n, L), y (n,). Training batches are class-balanced
    (the resampling block with gamma = 1, one independent stream per member), one epoch = len(train) samples.
    Each run is scored on val and test 4 times per epoch; we keep the check with the lowest ensemble validation
    loss (early stopping) and report its test scores. Measured on Catalina: the validation loss is lowest
    after ~2 epochs and test accuracy then falls from 74% to 67% by epoch 10, because the balanced batches
    repeat the rarest class ~66 times per epoch. So training stops once no run improved for `patience` epochs.
    `warp` optionally augments every training batch (the soft-warping baselines).
    """
    torch.manual_seed(seed)
    M = runs * ens
    L = train["x"].shape[1]
    model = CNNEnsemble(M, n_classes, L).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr, betas=(0.9, 0.999))
    scaler = torch.amp.GradScaler(enabled=amp_on(amp, device))
    Xtr = stack_input(train["x"], train["ph"]).to(device)
    ytr = train["y"].to(device)
    Xva, yva = stack_input(val["x"], val["ph"]).to(device), val["y"].cpu().numpy()
    Xte, yte = stack_input(test["x"], test["ph"]).to(device), test["y"].cpu().numpy()
    y_np = train["y"].cpu().numpy()
    samplers = [ClassResampler(y_np, gamma=1.0, seed=seed * 1000 + m, n_classes=n_classes) for m in range(M)]
    steps = math.ceil(len(y_np) / batch / checks_per_epoch)            # training steps between two checks

    best_loss = np.full(runs, np.inf)
    best = {"epoch": np.zeros(runs), "val_acc": np.zeros(runs), "test_acc": np.zeros(runs),
            "test_f1": np.zeros(runs), "test_pred": np.zeros((runs, len(yte)), int)}
    curves = {"epoch": [], "train_loss": [], "val_loss": [], "val_acc": [], "test_acc": []}
    since = np.zeros(runs, int)
    for check in range(1, int(max_epochs * checks_per_epoch) + 1):
        epoch = check / checks_per_epoch
        model.train()
        idx = torch.from_numpy(np.stack([s.sample(steps * batch) for s in samplers])).to(device)   # (M, S*B)
        tot = torch.zeros((), device=device)                            # summed on the GPU (no sync per step)
        for s in range(steps):
            ib = idx[:, s * batch:(s + 1) * batch]                    # (M, B)
            xb, yb = Xtr[ib], ytr[ib]                                  # (M, B, 2, L), (M, B)
            if warp is not None:
                xb = warp(xb)
            with autocast(device, amp):
                logits = model(xb)
            loss = F.cross_entropy(logits.float().reshape(-1, n_classes), yb.reshape(-1)) * M   # sum over members
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            tot += loss.detach() / M
        tot = tot.item()
        # scores of every run (= average of its members' softmax) on val and test: (runs, n, N)
        pv = predict_proba(model, Xva, amp=amp).reshape(runs, ens, len(yva), n_classes).mean(1).cpu().numpy()
        pt = predict_proba(model, Xte, amp=amp).reshape(runs, ens, len(yte), n_classes).mean(1).cpu().numpy()
        vloss = -np.log(np.take_along_axis(pv, yva[None, :, None], 2)[:, :, 0] + 1e-12).mean(1)   # (runs,)
        vacc = (pv.argmax(2) == yva[None]).mean(1)
        tpred = pt.argmax(2)                                                                    # (runs, n)
        tacc = (tpred == yte[None]).mean(1)
        for r in range(runs):
            if vloss[r] < best_loss[r]:
                best_loss[r], since[r] = vloss[r], 0
                best["epoch"][r], best["val_acc"][r], best["test_acc"][r] = epoch, vacc[r], tacc[r]
                best["test_f1"][r] = macro_f1(yte, tpred[r], n_classes)
                best["test_pred"][r] = tpred[r]
            else:
                since[r] += 1
        curves["epoch"].append(epoch)
        curves["train_loss"].append(tot / steps)
        curves["val_loss"].append(float(vloss.mean()))
        curves["val_acc"].append(float(vacc.mean()))
        curves["test_acc"].append(float(tacc.mean()))
        if check % checks_per_epoch == 0:
            log(f"  {tag} epoch {epoch:.0f}: train loss {tot / steps:.3f} | val loss {vloss.mean():.3f} "
                f"acc {vacc.mean():.3f} | test acc {tacc.mean():.3f}")
        if (since >= patience * checks_per_epoch).all():
            break
    recall = np.array([[np.mean(best["test_pred"][r][yte == c] == c) for c in range(n_classes)]
                       for r in range(runs)])
    return {"test_acc": best["test_acc"].tolist(), "test_f1": best["test_f1"].tolist(),
            "val_acc": best["val_acc"].tolist(), "best_epoch": best["epoch"].tolist(),
            "test_recall_per_class": recall.tolist(), "epochs_run": epoch, "curves": curves}


def train_feature_cnn(train, val, n_classes, max_epochs=20, patience=4, batch=128, lr=1e-4, seed=0,
                      device="cuda", log=print):
    """One base classifier trained on the balanced real training set (D_train^u). Its last conv block is the
    feature space for density, recall, FID and the G-score (Sec. 4.3). Keeps the lowest-val-loss weights.
    (Full precision: it is a single small network and its features are reused by every metric.)"""
    torch.manual_seed(seed)
    L = train["x"].shape[1]
    model = CNNEnsemble(1, n_classes, L).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr, betas=(0.9, 0.999))
    Xtr, ytr = stack_input(train["x"], train["ph"]).to(device), train["y"].to(device)
    Xva, yva = stack_input(val["x"], val["ph"]).to(device), val["y"].to(device)
    sampler = ClassResampler(train["y"].cpu().numpy(), gamma=1.0, seed=seed, n_classes=n_classes)
    steps = math.ceil(len(ytr) / batch)
    best, best_state, since, curve = np.inf, None, 0, []
    for epoch in range(1, max_epochs + 1):
        model.train()
        idx = torch.from_numpy(sampler.sample(steps * batch)).to(device)
        tot = torch.zeros((), device=device)
        for s in range(steps):
            ib = idx[s * batch:(s + 1) * batch]
            loss = F.cross_entropy(model(Xtr[ib].unsqueeze(0))[0], ytr[ib])
            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()
            tot += loss.detach()
        p = predict_proba(model, Xva, amp=False)[0]
        vloss = F.nll_loss(torch.log(p + 1e-12), yva).item()
        vacc = (p.argmax(1) == yva).float().mean().item()
        curve.append((tot.item() / steps, vloss, vacc))
        if vloss < best:
            best, since = vloss, 0
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        else:
            since += 1
        if since >= patience:
            break
    model.load_state_dict(best_state)
    model.eval()
    log(f"feature CNN: {epoch} epochs, best val loss {best:.3f}, val acc at best "
        f"{[c[2] for c in curve][int(np.argmin([c[1] for c in curve]))]:.3f}")
    return model, curve


@torch.no_grad()
def extract_features(model, x, ph, batch=1024):
    """Feature vectors (n, 64 * ceil(L/32)) of light curves x, ph (torch tensors on the model's device)."""
    model.eval()
    out = []
    for i in range(0, len(x), batch):
        out.append(model.features(stack_input(x[i:i + batch], ph[i:i + batch]).unsqueeze(0))[0])
    return torch.cat(out)


In [ ]:
%%writefile astra/warping.py
"""Soft window-warping baselines of Garcia-Jara et al. (2022), Sec. 4.8.3, Eqs. 17-18.

A random phase window [t1, t2] is picked; values v inside it are mapped by f(v) = a + b*tanh(k*(v - c)),
with a = c = window centre and b = half-width, so the window never grows (f(lo) >= lo, f(hi) <= hi).
k ~ U[1/(2b), 2/b]: small k*b contracts towards the centre, large k*b (saturated tanh) expands.
(The paper prints the interval with `a`; with `a` the magnitude version breaks, because the centre of
median-centred magnitudes can be 0 or negative, so we read it as the half-width b.)
Time-warping maps the phases, magnitude-warping maps the magnitudes (window limits m1, m2 = min/max
magnitude inside the phase window), mixed-warping does both.
"""
import torch


def _tanh_map(v, lo, hi, inside):
    a = (lo + hi) / 2
    b = (hi - lo) / 2
    ok = inside & (b > 0)
    b_safe = torch.where(b > 0, b, torch.ones_like(b))
    k = (0.5 + 1.5 * torch.rand_like(b)) / b_safe          # U[1/(2b), 2/b]
    return torch.where(ok, a + b * torch.tanh(k * (v - a)), v)


def soft_warp(xb, mode):
    """xb: (..., 2, L) with channel 0 = magnitudes, channel 1 = phases. Returns a warped copy."""
    x, ph = xb[..., 0, :], xb[..., 1, :]
    shape = x.shape[:-1] + (1,)
    t = torch.sort(torch.rand(shape[:-1] + (2,), device=xb.device), dim=-1).values
    t1, t2 = t[..., :1], t[..., 1:]
    inside = (ph >= t1) & (ph <= t2)
    if mode in ("time", "mixed"):
        ph = _tanh_map(ph, t1, t2, inside)
    if mode in ("mag", "mixed"):
        big = torch.finfo(x.dtype).max
        m1 = torch.where(inside, x, torch.full_like(x, big)).min(-1, keepdim=True).values
        m2 = torch.where(inside, x, torch.full_like(x, -big)).max(-1, keepdim=True).values
        enough = inside.sum(-1, keepdim=True) >= 2
        m1 = torch.where(enough, m1, torch.zeros_like(m1))
        m2 = torch.where(enough, m2, torch.zeros_like(m2))
        x = _tanh_map(x, m1, m2, inside & enough)
    return torch.stack([x, ph], dim=-2)


def make_warp(mode):
    return None if mode is None else (lambda xb: soft_warp(xb, mode))


In [ ]:
%%writefile astra/metrics.py
"""Feature-space metrics used by Garcia-Jara et al. (2022): density & recall (Eqs. 5-6), the G-score
(Eqs. 13-16) and FID (Eq. 3). Precision and coverage are computed too, for completeness.

A ball B(phi, r) is the sphere around phi whose radius r is the distance to its k-th nearest neighbour
inside the same set (Eq. 4).
"""
import numpy as np
import torch
from scipy import linalg, stats


def knn_radii(f, k, chunk=4096):
    """Distance from every row of f to its k-th nearest neighbour in f (itself excluded)."""
    out = []
    for i in range(0, len(f), chunk):
        d = torch.cdist(f[i:i + chunk], f)
        out.append(d.topk(k + 1, dim=1, largest=False).values[:, k])      # column 0 is the point itself
    return torch.cat(out)


def prdc(real, fake, k=5, real_radii=None, chunk=4096):
    """Precision, recall, density, coverage of fake features against real features (torch, same device)."""
    rr = knn_radii(real, k) if real_radii is None else real_radii
    rf = knn_radii(fake, k)
    in_real_balls = torch.zeros(len(fake), device=real.device)       # how many real balls hold each fake
    real_covered = torch.zeros(len(real), dtype=torch.bool, device=real.device)
    real_recalled = torch.zeros(len(real), dtype=torch.bool, device=real.device)
    for i in range(0, len(real), chunk):
        d = torch.cdist(real[i:i + chunk], fake)                     # (chunk, n_fake)
        inside = d <= rr[i:i + chunk, None]
        in_real_balls += inside.sum(0)
        real_covered[i:i + chunk] = inside.any(1)
        real_recalled[i:i + chunk] = (d <= rf[None, :]).any(1)
    return {"precision": (in_real_balls > 0).float().mean().item(),
            "recall": real_recalled.float().mean().item(),
            "density": (in_real_balls.sum() / (k * len(fake))).item(),
            "coverage": real_covered.float().mean().item()}


def g_score(D, R):
    """D, R: arrays (n_checkpoints, n_classes). Per-class min-max normalisation over the checkpoints of one
    run (Eqs. 13-14), per-class F-score (Eq. 15), macro average (Eq. 16). Returns one score per checkpoint."""
    D, R = np.asarray(D, float), np.asarray(R, float)

    def mm(a):
        lo, hi = a.min(0, keepdims=True), a.max(0, keepdims=True)
        return (a - lo) / np.where(hi > lo, hi - lo, 1.0)

    Dn, Rn = mm(D), mm(R)
    F = np.where(Dn + Rn > 0, 2 * Dn * Rn / np.where(Dn + Rn > 0, Dn + Rn, 1.0), 0.0)
    return F.mean(1)


def gauss_stats(f):
    """Mean and covariance of a feature set (numpy, float64)."""
    return f.mean(0), np.cov(f, rowvar=False)


def fid_from_stats(mu1, s1, mu2, s2):
    """Frechet distance between two Gaussians (Eq. 3)."""
    covmean = linalg.sqrtm(s1 @ s2)
    if not np.isfinite(covmean).all():                                 # standard fix for a singular product
        eps = np.eye(len(s1)) * 1e-6
        covmean = linalg.sqrtm((s1 + eps) @ (s2 + eps))
    return float(np.sum((mu1 - mu2) ** 2) + np.trace(s1 + s2 - 2 * np.real(covmean)))


def fid(real, fake):
    """Frechet distance between Gaussians fitted to two feature sets (numpy arrays)."""
    return fid_from_stats(*gauss_stats(real), *gauss_stats(fake))


def welch_p(a, b):
    """Two-sided Welch's t-test p-value, as in Table 5."""
    return float(stats.ttest_ind(a, b, equal_var=False).pvalue)


In [ ]:
%%writefile astra/bench_data.py
"""Data for the ASTRA benchmark: the pre-processed light curves of the GAN notebook, split over stars.

What stays as in Garcia-Jara et al. (2022): the curves (L magnitudes at L sorted phases, median-centred), and
what a generator is given, the condition (phases, amplitude, class) of a real star.

What changes for the benchmark (ours):
  - the split. The paper holds out 20% of the smallest class for every class (36 curves per class on Catalina),
    and on ZTF it splits over curves, so the g and the r curve of a star can sit on both sides. Here the split
    is over STARS and stratified by class: 70% train, 10% validation, 20% test. Curves that share class and
    period are treated as one star (ZTF gives a star seen in two fields two object ids, with the same
    catalogue period), so no star is on two sides;
  - the target. Every model generates the amplitude-normalised curve u = magnitude / amplitude, which lies in
    [-2, 2] for every class, and is told the amplitude as log10(amplitude), standardised. A generated curve is
    turned back into magnitudes by multiplying with the amplitude of its condition. rep = "raw" gives the
    paper's own target instead (magnitudes, amplitude in mag), for one control run of the GAN.
"""
import hashlib

import numpy as np
import torch
import torch.nn.functional as F

from .sampling import ClassResampler

TRAIN, VAL, TEST = 0, 1, 2
FRACTIONS = (0.7, 0.1, 0.2)          # train, validation, test (of the stars of every class)
RARE_BELOW = 1000                    # a class with fewer training curves than this is in the "rare tier"
LATENT = {100: 16, 40: 8}            # noise / latent size per curve length: Catalina, ZTF (Sec. 4.2 of the paper)
G_PARAMS = {100: 3_990_785, 40: 3_708_161}   # parameters of the paper's generator; the other families match it


def star_ids(y, period):
    """One id per (class, period): the curves of one star, whatever band or object id they come from."""
    key = np.stack([y.astype(np.float64), period.astype(np.float64)], 1)
    return np.unique(key, axis=0, return_inverse=True)[1].reshape(-1)


def star_split(y, star, seed, fractions=FRACTIONS):
    """Stratified split over stars: every class keeps the same train/val/test shares, a star is on one side."""
    rng = np.random.default_rng(1000 + seed)
    stars, first = np.unique(star, return_index=True)
    ys = y[first]
    side = np.full(len(stars), TRAIN, np.int8)
    for c in np.unique(ys):
        idx = rng.permutation(np.flatnonzero(ys == c))
        n_val, n_test = int(round(fractions[1] * len(idx))), int(round(fractions[2] * len(idx)))
        side[idx[:n_val]] = VAL
        side[idx[n_val:n_val + n_test]] = TEST
    return side[np.searchsorted(stars, star)]


def split_table(npz_path, seed):
    """Curves per class and side, for the notebook's data section."""
    z = np.load(npz_path)
    split = star_split(z["y"], star_ids(z["y"], z["period"]), seed)
    n = len(z["classes"])
    return {name: np.bincount(z["y"][split == s], minlength=n) for name, s in (("train", TRAIN), ("val", VAL),
                                                                               ("test", TEST))}


def load(npz_path, seed, device, rep="unit"):
    """Everything a job needs, as tensors on `device`. Index arrays always refer to rows of the file."""
    z = np.load(npz_path)
    y = z["y"].astype(np.int64)
    star = star_ids(y, z["period"])
    split = star_split(y, star, seed)
    n_classes = len(z["classes"])
    t = lambda a, dt=torch.float32: torch.from_numpy(np.ascontiguousarray(a)).to(device=device, dtype=dt)
    data = {"x": t(z["mag"]), "ph": t(z["phase"]), "amp": t(z["amp"]), "y": t(y, torch.long),
            "n_classes": n_classes, "classes": [str(c) for c in z["classes"]], "L": z["mag"].shape[1],
            "y_np": y, "split": split, "star": star, "seed": seed, "rep": rep, "device": device}
    data["onehot"] = F.one_hot(data["y"], n_classes).float()
    for name, s in (("train", TRAIN), ("val", VAL), ("test", TEST)):
        idx = np.flatnonzero(split == s)
        data[f"{name}_idx_np"] = idx
        data[f"{name}_idx"] = torch.from_numpy(idx).to(device)
    tr = data["train_idx"]
    assert (data["amp"] > 0).all(), "every curve needs a positive amplitude"
    if rep == "unit":                 # u = x / amplitude in [-2, 2]; condition = standardised log-amplitude
        la = torch.log10(data["amp"])
        data["u"] = data["x"] / data["amp"][:, None]
        data["camp"] = (la - la[tr].mean()) / la[tr].std()
        data["mag_scale"] = data["amp"].clone()
        data["out_scale"] = 2.0
    elif rep == "raw":                # the paper's target: magnitudes, amplitude in mag, tanh scale = largest |mag|
        data["u"], data["camp"] = data["x"], data["amp"]
        data["mag_scale"] = torch.ones_like(data["amp"])
        data["out_scale"] = float(data["x"][tr].abs().max())
    else:
        raise ValueError(rep)
    counts = np.bincount(y[split == TRAIN], minlength=n_classes)
    data["train_counts"] = counts
    data["rare"] = counts < RARE_BELOW
    h = hashlib.md5()
    for a in (y, split, z["mag"], z["phase"]):
        h.update(np.ascontiguousarray(a).tobytes())
    data["hash"] = h.hexdigest()[:12]            # two notebooks may pair their results only if this agrees
    return data


def balanced_idx(data, n, seed, gamma=1.0, side="train"):
    """n row indices of one side, drawn by the resampling block (gamma = 1: every class equally often)."""
    pool = data[f"{side}_idx_np"]
    rs = ClassResampler(data["y_np"][pool], gamma, seed=seed, n_classes=data["n_classes"])
    return data[f"{side}_idx"][torch.from_numpy(rs.sample(n)).to(data["device"])]


def subset(data, idx, x=None):
    """A set of curves as the classifier wants it: magnitudes x, phases ph, labels y."""
    return {"x": data["x"][idx] if x is None else x, "ph": data["ph"][idx], "y": data["y"][idx]}


In [ ]:
%%writefile astra/bench_eval.py
"""The evaluation harness of the ASTRA benchmark. It is the same code for every model family: a family only
hands over sample(idx, seed) -> magnitudes of one generated curve per condition idx (rows of real stars).

What is measured, and where it comes from:
  - TSTR (train on synthetic, test on real) and real + synthetic training, with the classifier of Garcia-Jara
    et al. (2022): 15 runs of an ensemble of 5 CNNs. Reported on the real test stars of the split.
  - Balanced accuracy = the mean of the per-class recalls. On the paper's class-balanced test set this is its
    plain accuracy; on our stratified test set it keeps every class equally important. Macro-F1, accuracy and
    the per-class recalls are reported next to it.
  - Fidelity in the feature space of a classifier trained on real data (the paper's Sec. 4.3): per-class
    precision, recall, density and coverage with k-NN balls, and FID.
  - Memorisation (ours), two numbers per class, from the distance between a generated curve and the real curve
    of its own condition star. The ratio: median distance for training stars / the same for validation stars;
    near 1, training stars are treated like new stars, far below 1, the model reproduces the training curves it
    was conditioned on. The copy share: the share of training-star curves that are closer to their source than
    95% of the validation-star curves are to theirs; 5% means no copying, 100% means every curve is a copy.
    Notebook 01 showed why it is needed: late GAN checkpoints return the training curve of the condition for the
    rare classes, which density and recall cannot see.
  - TSTR for conditions no training star has (ours, 2026-10-22): one star's phases with another star's
    amplitude, same class; at the selected and at the last checkpoint. It tells whether a model that copies its
    training curves has still learnt the classes.
Model selection uses validation stars only. Test scores of the non-selected settings are saved for the
analysis but never used for a choice.
"""
import math
import os
import time

import numpy as np
import torch
import torch.nn.functional as F

from .bench_data import balanced_idx, subset
from .classifier import CNNEnsemble, amp_on, autocast, extract_features, predict_proba, stack_input
from .common import load_json, save_json
from .metrics import fid_from_stats, knn_radii, prdc
from .sampling import ClassResampler
from .warping import make_warp

# Classifier settings are those of the replication notebook (the "ours" choices are explained there).
EVAL = dict(k=5, ens=5,
            n_sel=20_000, sel_runs=3, sel_max_epochs=6, sel_patience=2,      # per checkpoint: 3 runs x 5 CNNs
            final_runs=15, final_max_epochs=10, final_patience=3,            # test scores: 15 runs x 5 CNNs
            p_syn=(0.25, 0.5, 0.75),      # share of synthetic curves in real + synthetic training
            n_final=None,                 # synthetic curves for the test scores (None = one per training curve)
            max_sweeps=None,              # how many of the family's other sampling settings to score (None = all)
            n_mem=2_000, n_loss=2_000)
EVAL_TINY = dict(n_sel=600, sel_runs=1, sel_max_epochs=1, sel_patience=1, final_runs=2, final_max_epochs=1,
                 final_patience=1, p_syn=(0.5,), n_final=1200, max_sweeps=2, n_mem=200, n_loss=200)
REAL_ROWS = {"A_oversampling": None, "B_soft_time_warping": "time", "C_soft_mag_warping": "mag",
             "D_soft_mixed_warping": "mixed"}


def save_npz(path, compressed=False, **arrays):
    """Write to a temp file first, so a kill mid-write never leaves a broken file."""
    tmp = path.with_name(path.stem + ".tmp.npz")
    (np.savez_compressed if compressed else np.savez)(tmp, **arrays)
    os.replace(tmp, path)


# ---------------------------------------------------------------- scores
def confusion(y, pred, n_classes, w=None):
    """pred: (runs, n) predicted classes -> (runs, N, N) counts [true, predicted]; w = optional curve weights."""
    pred = np.atleast_2d(pred)
    return np.stack([np.bincount(y * n_classes + p, weights=w, minlength=n_classes ** 2)
                     .reshape(n_classes, n_classes) for p in pred])


def scores_from_conf(conf):
    """conf[..., true, predicted] -> balanced accuracy (mean recall over the classes present), macro-F1,
    accuracy, and the per-class recall, precision and F1."""
    conf = np.asarray(conf, float)
    tp = np.diagonal(conf, axis1=-2, axis2=-1)
    n_true, n_pred = conf.sum(-1), conf.sum(-2)
    rec, prec = tp / np.maximum(n_true, 1e-12), tp / np.maximum(n_pred, 1e-12)
    f1 = 2 * prec * rec / np.maximum(prec + rec, 1e-12)
    has = n_true > 0
    k = np.maximum(has.sum(-1), 1)
    return {"bal_acc": (rec * has).sum(-1) / k, "macro_f1": (f1 * has).sum(-1) / k,
            "acc": tp.sum(-1) / np.maximum(conf.sum((-2, -1)), 1e-12), "recall": rec, "precision": prec, "f1": f1}


def summarise(y, pred, n_classes, rare=None):
    """Scores of every run (pred: runs x n), then their mean and std over the runs."""
    s = scores_from_conf(confusion(y, pred, n_classes))
    out = {}
    for k in ("bal_acc", "macro_f1", "acc"):
        out[k], out[k + "_std"], out[k + "_runs"] = float(s[k].mean()), float(s[k].std()), s[k].tolist()
    for k in ("recall", "precision", "f1"):
        out[k] = s[k].mean(0).tolist()
    if rare is not None and rare.any() and (~rare).any():
        out["rare_recall"], out["common_recall"] = float(s["recall"][:, rare].mean()), float(s["recall"][:, ~rare].mean())
    return out


# ---------------------------------------------------------------- the classifier
def fit_classifiers(pools, val, test, n_classes, runs=15, ens=5, max_epochs=10, patience=3, batch=128, lr=1e-4,
                    seed=0, p_syn=0.5, warp=None, device="cuda", log=print, tag="", amp=True, checks_per_epoch=4):
    """Train `runs` independent ensembles of `ens` CNNs side by side and return each run's test predictions.

    pools: one training set, or [real, synthetic]; with two, every training curve is synthetic with
    probability p_syn. Batches are class-balanced inside each pool (the resampling block with gamma = 1),
    one epoch = as many curves as the first pool holds. Each run is checked on the validation stars 4 times
    per epoch and keeps the check with the lowest class-balanced validation loss (early stopping); training
    ends once no run has improved for `patience` epochs. This is astra.classifier.fit_ensembles with two
    changes for the stratified split: the validation loss weights every class equally, and the test
    predictions are returned (for balanced accuracy, bootstrap intervals and paired tests).
    """
    torch.manual_seed(seed)
    M = runs * ens
    model = CNNEnsemble(M, n_classes, pools[0]["x"].shape[1]).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr, betas=(0.9, 0.999))
    scaler = torch.amp.GradScaler(enabled=amp_on(amp, device))
    Xtr = torch.cat([stack_input(p["x"], p["ph"]) for p in pools]).to(device)
    ytr = torch.cat([p["y"] for p in pools]).to(device)
    off = len(pools[0]["y"])
    samplers = [[ClassResampler(p["y"].cpu().numpy(), 1.0, seed=seed * 1000 + m + 500_000 * k, n_classes=n_classes)
                 for m in range(M)] for k, p in enumerate(pools)]
    coin = np.random.default_rng(seed + 77)
    Xva, yva = stack_input(val["x"], val["ph"]).to(device), val["y"].cpu().numpy()
    Xte, yte = stack_input(test["x"], test["ph"]).to(device), test["y"].cpu().numpy()
    cnt = np.bincount(yva, minlength=n_classes)
    wva = 1.0 / (np.sum(cnt > 0) * cnt[yva])                           # every class weighs the same, sum = 1
    steps = math.ceil(off / batch / checks_per_epoch)                  # training steps between two checks

    best_loss, since = np.full(runs, np.inf), np.zeros(runs, int)
    best = {"epoch": np.zeros(runs), "val_bal_acc": np.zeros(runs), "val_loss": np.zeros(runs),
            "test_pred": np.zeros((runs, len(yte)), np.int8)}
    curves = {"epoch": [], "train_loss": [], "val_loss": [], "val_bal_acc": []}
    for check in range(1, int(max_epochs * checks_per_epoch) + 1):
        epoch = check / checks_per_epoch
        model.train()
        idx = np.stack([s.sample(steps * batch) for s in samplers[0]])                         # (M, S*B)
        if len(pools) == 2:
            idx_syn = np.stack([s.sample(steps * batch) for s in samplers[1]]) + off
            idx = np.where(coin.random(idx.shape) < p_syn, idx_syn, idx)
        idx = torch.from_numpy(idx).to(device)
        tot = torch.zeros((), device=device)
        for s in range(steps):
            ib = idx[:, s * batch:(s + 1) * batch]
            xb, yb = Xtr[ib], ytr[ib]                                  # (M, B, 2, L), (M, B)
            if warp is not None:
                xb = warp(xb)
            with autocast(device, amp):
                logits = model(xb)
            loss = F.cross_entropy(logits.float().reshape(-1, n_classes), yb.reshape(-1)) * M   # sum over members
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            tot += loss.detach() / M
        tot = tot.item() / steps
        # every run = the average of its members' softmax outputs
        pv = predict_proba(model, Xva, amp=amp).reshape(runs, ens, len(yva), n_classes).mean(1).cpu().numpy()
        vloss = (-np.log(np.take_along_axis(pv, yva[None, :, None], 2)[:, :, 0] + 1e-12) * wva).sum(1)   # (runs,)
        vbal = ((pv.argmax(2) == yva[None]) * wva).sum(1)
        better = vloss < best_loss
        if better.any():                                               # test predictions only when a run improved
            tp = predict_proba(model, Xte, amp=amp).reshape(runs, ens, len(yte), n_classes).mean(1).argmax(2).cpu().numpy()
            best["test_pred"][better] = tp[better]
            best["epoch"][better], best["val_bal_acc"][better], best["val_loss"][better] = epoch, vbal[better], vloss[better]
            best_loss[better] = vloss[better]
        since = np.where(better, 0, since + 1)
        for k, v in zip(curves, (epoch, tot, float(vloss.mean()), float(vbal.mean()))):
            curves[k].append(v)
        if check % checks_per_epoch == 0:
            log(f"  {tag} epoch {epoch:.0f}: train loss {tot:.3f} | val loss {vloss.mean():.3f} "
                f"balanced acc {vbal.mean():.3f}")
        if (since >= patience * checks_per_epoch).all():
            break
    return {"test_pred": best["test_pred"], "val_bal_acc": best["val_bal_acc"].tolist(),
            "val_loss": best["val_loss"].tolist(), "best_epoch": best["epoch"].tolist(), "epochs_run": epoch,
            "curves": curves}


# ---------------------------------------------------------------- fidelity and memorisation
class Fidelity:
    """Feature-space scores of a synthetic set against the real training curves of one split."""

    def __init__(self, data, feat, k=5):
        self.data, self.feat, self.k = data, feat, k
        tr = data["train_idx"]
        f, ytr = extract_features(feat, data["x"][tr], data["ph"][tr]), data["y"][tr]
        self.per = []                                   # per class: real features and their k-NN radii
        for c in range(data["n_classes"]):
            fc = f[ytr == c]
            self.per.append((fc, knn_radii(fc, k)))
        f64 = f.cpu().numpy().astype(np.float64)
        self.real_stats = (f64.mean(0), np.cov(f64, rowvar=False))
        self.prior = data["train_counts"] / data["train_counts"].sum()

    def __call__(self, x_syn, idx):
        d = self.data
        fg, yg = extract_features(self.feat, x_syn, d["ph"][idx]), d["y"][idx]
        out = {m: [] for m in ("precision", "recall", "density", "coverage")}
        for c in range(d["n_classes"]):
            fc = fg[yg == c]
            r = prdc(self.per[c][0], fc, self.k, real_radii=self.per[c][1]) if len(fc) > self.k else {}
            for m in out:
                out[m].append(r.get(m, float("nan")))
        # FID at the natural class mix: the synthetic set is class-balanced, so every curve is weighted by
        # (share of its class in the training set) / (share of its class in this set)
        yg = yg.cpu().numpy()
        w = (self.prior / np.maximum(np.bincount(yg, minlength=d["n_classes"]) / len(yg), 1e-12))[yg]
        w = w / w.sum()
        f64 = fg.cpu().numpy().astype(np.float64)
        mu = (w[:, None] * f64).sum(0)
        cov = (w[:, None] * (f64 - mu)).T @ (f64 - mu)
        try:
            out["fid"] = fid_from_stats(*self.real_stats, mu, cov)
        except Exception:                   # a failed matrix square root: no value, but the job goes on
            out["fid"] = float("nan")
        # the amplitude condition: half the peak-to-peak range of a generated curve should equal it
        half_range = (x_syn.max(1).values - x_syn.min(1).values) / 2
        out["amp_err"] = float((half_range / d["amp"][idx] - 1).abs().mean())
        return out


def own_distance(data, x_syn, idx):
    """RMS difference between a generated curve and the real curve of its condition star, in amplitudes."""
    return ((x_syn - data["x"][idx]) / data["amp"][idx][:, None]).pow(2).mean(1).sqrt()


def memorisation(data, sample, x_syn, idx, n, seed, **opts):
    """Per class: (median own-distance for training conditions / the same for validation conditions,
    share of training-condition curves closer to their source than the 5th percentile of the validation ones)."""
    idx_v = balanced_idx(data, n, seed, side="val")
    d_t, d_v = own_distance(data, x_syn, idx), own_distance(data, sample(idx_v, seed + 1, **opts), idx_v)
    yt, yv = data["y"][idx], data["y"][idx_v]
    ratio, copy = [], []
    for c in range(data["n_classes"]):
        if (yt == c).any() and (yv == c).any():
            a, b = d_t[yt == c], d_v[yv == c]
            ratio.append(float(a.median() / b.median()))
            copy.append(float((a < torch.quantile(b.float(), 0.05)).float().mean()))
        else:
            ratio.append(float("nan")); copy.append(float("nan"))
    return ratio, copy


def fixed_rows(data, side, n, seed=0):
    idx = data[f"{side}_idx_np"]
    pick = np.random.default_rng(seed).choice(len(idx), min(n, len(idx)), replace=False)
    return data[f"{side}_idx"][torch.from_numpy(np.sort(pick)).to(data["device"])]


# ---------------------------------------------------------------- one checkpoint (validation side only)
def evaluate_checkpoint(model, fidelity, data, cfg, k, device):
    t0 = time.time()
    idx = balanced_idx(data, cfg["n_sel"], seed=50_000 + k)
    x = model.sample(idx, seed=60_000 + k)
    res = {"ckpt": k, "nonfinite": model.n_bad, **fidelity(x, idx)}
    res["mem_ratio"], res["copy_share"] = memorisation(data, model.sample, x, idx, cfg["n_mem"], seed=65_000 + k)
    r = fit_classifiers([subset(data, idx, x)], subset(data, data["val_idx"]), subset(data, data["test_idx"]),
                        data["n_classes"], runs=cfg["sel_runs"], ens=cfg["ens"], max_epochs=cfg["sel_max_epochs"],
                        patience=cfg["sel_patience"], seed=k, device=device, log=lambda *a: None,
                        amp=cfg.get("amp", True))
    res["val_bal_acc"], res["val_bal_acc_runs"] = float(np.mean(r["val_bal_acc"])), r["val_bal_acc"]
    # logged for the analysis only: never used to choose a checkpoint
    res["test_bal_acc_info"] = summarise(data["y_np"][data["test_idx_np"]], r["test_pred"], data["n_classes"])["bal_acc"]
    # the model's own loss on training stars and on validation stars (over/under-fitting check)
    res["loss_train"] = model.losses(fixed_rows(data, "train", cfg["n_loss"]), seed=1)
    res["loss_val"] = model.losses(fixed_rows(data, "val", cfg["n_loss"]), seed=1)
    res["eval_s"] = time.time() - t0
    return res


# ---------------------------------------------------------------- the selected checkpoint (test side)
def final_evaluation(model, fidelity, data, cfg, out, deadline, device, log, sweeps=()):
    """Test scores of the selected checkpoint. Every part is saved when it is done, so a session that runs out
    of time continues here in the next one. Returns the dict of parts, or None if it had to stop early."""
    parts = load_json(out / "final_partial.json", {})
    preds = dict(np.load(out / "final_pred.npz")) if (out / "final_pred.npz").exists() else {}
    N, rare = data["n_classes"], data["rare"]
    va, te, real = (subset(data, data[f"{s}_idx"]) for s in ("val", "test", "train"))
    yte = data["y_np"][data["test_idx_np"]]

    def sync():
        if str(device).startswith("cuda"):
            torch.cuda.synchronize()

    def fit(name, pools, runs, max_epochs, patience, **kw):
        t0 = time.time()
        r = fit_classifiers(pools, va, te, N, runs=runs, ens=cfg["ens"], max_epochs=max_epochs, patience=patience,
                            seed=1, device=device, log=log, tag=name, amp=cfg.get("amp", True), **kw)
        s = summarise(yte, r["test_pred"], N, rare)
        s.update(val_bal_acc=float(np.mean(r["val_bal_acc"])), best_epoch=float(np.mean(r["best_epoch"])),
                 curves=r["curves"], minutes=(time.time() - t0) / 60)
        preds[name] = r["test_pred"]
        log(f"{name}: test balanced accuracy {100 * s['bal_acc']:.2f} +- {100 * s['bal_acc_std']:.2f} | "
            f"macro-F1 {100 * s['macro_f1']:.2f} | val balanced accuracy {100 * s['val_bal_acc']:.2f}")
        return s

    def save():
        save_npz(out / "final_pred.npz", **preds)
        save_json(parts, out / "final_partial.json")

    # one synthetic curve per (balanced) training condition; the same conditions for every family
    idx = balanced_idx(data, cfg.get("n_final") or len(data["train_idx"]), seed=70_000)
    if "samples" not in parts:
        sync(); t0 = time.time()
        x = model.sample(idx, seed=80_000)
        sync(); dt = time.time() - t0
        save_npz(out / "samples.npz", True, idx=idx.cpu().numpy(), x=x.cpu().numpy().astype(np.float16))
        mem, copy = memorisation(data, model.sample, x, idx, cfg["n_mem"], seed=85_000)
        parts["samples"] = {"n": len(idx), "sampling_s_per_10k": dt / len(idx) * 1e4, "nonfinite": model.n_bad,
                            **fidelity(x, idx), "mem_ratio": mem, "copy_share": copy}
        save()
    x = torch.from_numpy(np.load(out / "samples.npz")["x"].astype(np.float32)).to(device)
    syn = subset(data, idx, x)
    fin = dict(runs=cfg["final_runs"], max_epochs=cfg["final_max_epochs"], patience=cfg["final_patience"])
    todo = [("tstr", [syn], {})] + [(f"aug_p{round(100 * p)}", [real, syn], {"p_syn": p}) for p in cfg["p_syn"]]
    for name, pools, kw in todo:
        if name in parts:
            continue
        if deadline.passed(900):
            log("time budget reached during the final evaluation; it continues next session")
            return None
        parts[name] = fit(name, pools, **fin, **kw)
        save()
    # other sampling settings of the same checkpoint (smaller: n_sel curves, 3 runs)
    sel = dict(runs=cfg["sel_runs"], max_epochs=cfg["sel_max_epochs"], patience=cfg["sel_patience"])
    idx_s = balanced_idx(data, cfg["n_sel"], seed=75_000)
    for i, opts in enumerate([{}] + list(sweeps)):
        name = "sweep_" + ("default" if not opts else "_".join(f"{k}{v}" for k, v in opts.items()))
        if name in parts:
            continue
        if deadline.passed(900):
            log("time budget reached during the sampling sweeps; they continue next session")
            return None
        sync(); t0 = time.time()
        xs = model.sample(idx_s, seed=76_000, **opts)
        sync(); dt = time.time() - t0
        parts[name] = {"opts": opts, "sampling_s_per_10k": dt / len(idx_s) * 1e4, "fidelity": fidelity(xs, idx_s),
                       **fit(name, [subset(data, idx_s, xs)], **sel)}
        save()
    return parts


def tstr_only(model, data, cfg, out, deadline, device, log, name):
    """The test score of another checkpoint (the compute-matched budgets): TSTR with the same conditions,
    seeds and classifier runs as the headline. Saved as one more part, so it resumes like the others."""
    parts = load_json(out / "final_partial.json", {})
    if name in parts:
        return parts[name]
    if deadline.passed(900):
        log(f"time budget reached before {name}; it continues next session")
        return None
    preds = dict(np.load(out / "final_pred.npz")) if (out / "final_pred.npz").exists() else {}
    idx = balanced_idx(data, cfg.get("n_final") or len(data["train_idx"]), seed=70_000)
    t0 = time.time()
    x = model.sample(idx, seed=80_000)
    va, te = (subset(data, data[f"{s}_idx"]) for s in ("val", "test"))
    r = fit_classifiers([subset(data, idx, x)], va, te, data["n_classes"], runs=cfg["final_runs"], ens=cfg["ens"],
                        max_epochs=cfg["final_max_epochs"], patience=cfg["final_patience"], seed=1, device=device,
                        log=log, tag=name, amp=cfg.get("amp", True))
    s = summarise(data["y_np"][data["test_idx_np"]], r["test_pred"], data["n_classes"], data["rare"])
    s.update(val_bal_acc=float(np.mean(r["val_bal_acc"])), best_epoch=float(np.mean(r["best_epoch"])),
             minutes=(time.time() - t0) / 60, nonfinite=model.n_bad)
    parts[name], preds[name] = s, r["test_pred"]
    save_npz(out / "final_pred.npz", **preds)
    save_json(parts, out / "final_partial.json")
    log(f"{name}: test balanced accuracy {100 * s['bal_acc']:.2f} +- {100 * s['bal_acc_std']:.2f}")
    return s


# ---------------------------------------------------------------- conditions that no training star has
def novel_conditions(data, idx, seed):
    """For every training-condition row of idx: the phases of ANOTHER training star of the same class and the
    amplitude of a third one; the class is kept. No training star has that combination, so a model that has
    learnt to look up a training curve by its condition cannot copy one; what it generates then shows how much
    of the class it has learnt. Returns a view of the data whose rows 0 .. len(idx)-1 are these conditions (all
    that a family's cond() and to_mag() read), and the two donor rows of every condition."""
    rng = np.random.default_rng(seed)
    rows, y, star, tr = idx.cpu().numpy(), data["y_np"], data["star"], data["train_idx_np"]

    def donors(avoid):
        pick = np.empty_like(rows)
        for c in np.unique(y[rows]):
            k = np.flatnonzero(y[rows] == c)
            pool = tr[y[tr] == c]
            assert len(np.unique(star[pool])) >= 3, f"class {c}: too few training stars for new conditions"
            pick[k] = rng.choice(pool, len(k))
            for _ in range(100):           # another star than every row in `avoid` (redraw the clashes)
                bad = np.any([star[pick[k]] == star[a[k]] for a in avoid], axis=0)
                if not bad.any():
                    break
                pick[k[bad]] = rng.choice(pool, int(bad.sum()))
            assert not np.any([star[pick[k]] == star[a[k]] for a in avoid]), "no different donor star found"
        return pick

    ph_from = donors([rows])
    amp_from = donors([rows, ph_from])
    p, a = (torch.from_numpy(v).to(idx.device) for v in (ph_from, amp_from))
    view = {k: v for k, v in data.items() if k not in ("x", "u")}     # no curves: a family must not read them here
    view.update(ph=data["ph"][p], y=data["y"][idx], onehot=data["onehot"][idx], camp=data["camp"][a],
                amp=data["amp"][a], mag_scale=data["mag_scale"][a])
    return view, ph_from, amp_from


class conditions_of:
    """Inside this block the model generates for the condition rows of `view` instead of its own data."""

    def __init__(self, model, view):
        self.model, self.view = model, view

    def __enter__(self):
        self.old, self.model.data = self.model.data, self.view

    def __exit__(self, *exc):
        self.model.data = self.old


def condition_test(model, data, cfg, out, deadline, device, log, name, novel, ckpt):
    """TSTR of the loaded checkpoint with the sampling sweeps' sizes (n_sel balanced training conditions, sel_runs
    classifier runs, the same seeds): for the training stars' own conditions (at the selected checkpoint this is
    the part sweep_default), or, with novel, for the same rows' new conditions (novel_conditions). No
    memorisation score here: a new condition has no source curve. Saved as one more part, so it resumes."""
    parts = load_json(out / "final_partial.json", {})
    if name in parts:
        return parts[name]
    if deadline.passed(900):
        log(f"time budget reached before {name}; it continues next session")
        return None
    preds = dict(np.load(out / "final_pred.npz")) if (out / "final_pred.npz").exists() else {}
    idx = balanced_idx(data, cfg["n_sel"], seed=75_000)
    t0 = time.time()
    if novel:
        view, _, _ = novel_conditions(data, idx, seed=77_000)
        with conditions_of(model, view):
            x = model.sample(torch.arange(len(idx), device=idx.device), seed=76_000)
        pool = {"x": x, "ph": view["ph"], "y": view["y"]}
    else:
        pool = subset(data, idx, model.sample(idx, seed=76_000))
    va, te = (subset(data, data[f"{s}_idx"]) for s in ("val", "test"))
    r = fit_classifiers([pool], va, te, data["n_classes"], runs=cfg["sel_runs"], ens=cfg["ens"],
                        max_epochs=cfg["sel_max_epochs"], patience=cfg["sel_patience"], seed=1, device=device,
                        log=log, tag=name, amp=cfg.get("amp", True))
    s = summarise(data["y_np"][data["test_idx_np"]], r["test_pred"], data["n_classes"], data["rare"])
    s.update(val_bal_acc=float(np.mean(r["val_bal_acc"])), best_epoch=float(np.mean(r["best_epoch"])),
             minutes=(time.time() - t0) / 60, nonfinite=model.n_bad, ckpt=ckpt, novel=novel)
    parts[name], preds[name] = s, r["test_pred"]
    save_npz(out / "final_pred.npz", **preds)
    save_json(parts, out / "final_partial.json")
    log(f"{name} (checkpoint {ckpt}): test balanced accuracy {100 * s['bal_acc']:.2f} +- {100 * s['bal_acc_std']:.2f}")
    return s


def run_real(data, cfg, out, deadline, device, log):
    """The reference rows on real data only: A = oversampling, B-D = + soft warping (Table 5 of the paper)."""
    out.mkdir(parents=True, exist_ok=True)
    parts = load_json(out / "final_partial.json", {})
    preds = dict(np.load(out / "final_pred.npz")) if (out / "final_pred.npz").exists() else {}
    va, te, real = (subset(data, data[f"{s}_idx"]) for s in ("val", "test", "train"))
    yte = data["y_np"][data["test_idx_np"]]
    for row, mode in REAL_ROWS.items():
        if row in parts:
            continue
        if deadline.passed(900):
            log("time budget reached; the remaining rows run next session")
            return None
        t0 = time.time()
        r = fit_classifiers([real], va, te, data["n_classes"], runs=cfg["final_runs"], ens=cfg["ens"],
                            max_epochs=cfg["final_max_epochs"], patience=cfg["final_patience"], seed=1,
                            warp=make_warp(mode), device=device, log=log, tag=row, amp=cfg.get("amp", True))
        s = summarise(yte, r["test_pred"], data["n_classes"], data["rare"])
        s.update(val_bal_acc=float(np.mean(r["val_bal_acc"])), best_epoch=float(np.mean(r["best_epoch"])),
                 curves=r["curves"], minutes=(time.time() - t0) / 60)
        parts[row], preds[row] = s, r["test_pred"]
        save_npz(out / "final_pred.npz", **preds)
        save_json(parts, out / "final_partial.json")
        log(f"{row}: test balanced accuracy {100 * s['bal_acc']:.2f} +- {100 * s['bal_acc_std']:.2f} | "
            f"macro-F1 {100 * s['macro_f1']:.2f} ({s['minutes']:.1f} min)")
    return parts


In [ ]:
%%writefile astra/bench_grid.py
"""The benchmark's search grids, the configurations chosen by the search, and the measured speeds.

This is the only file to edit between the stages (then rebuild the notebooks with build/build_notebooks.py):
after the search, put each family's winner into FINAL[family] and the speed its search used into SPEED[family].
The winner: the highest mean of the three best checkpoints' validation scores, read against the copy share
(a configuration whose good checkpoints copy its training curves loses; group decision, 2026-10-22).
Pure Python without torch, so the notebook builder can read it as well.

The search runs every configuration on split 0 and scores validation stars only. What is held fixed for all
families (the controls): network size (about 4 M parameters), EMA 0.999, resampling gamma = 0.25, mixed
precision, the schedule and the stopping rule (astra/bench_run.py). Sampling settings (steps, guidance, solver,
temperature) need no training and are swept at the selected checkpoint in the final stage.
"""
GRID = {
    # the paper's WGAN-GP (5 critic steps, gradient penalty 10, Adam betas 0.5 / 0.9): learning rate x batch size
    "gan": {f"lr{lr}_b{b}": dict(lr=float(lr), batch=b) for lr in ("5e-5", "1e-4", "2e-4") for b in (64, 256)},
    # KL weight beta (< 1: sharper curves, codes further from the prior; > 1: closer to the prior) x learning rate;
    # beta = 16 added for the search's session 2 (group decision, 2026-10-22): the score rose with beta up to the grid edge
    "vae": {f"b{b}_lr{lr}": dict(beta=float(b), lr=float(lr)) for b in ("0.1", "1", "4", "16") for lr in ("2e-4", "1e-3")},
    # what the network predicts (eps, or v = sqrt(ab) eps - sqrt(1 - ab) x0), cosine schedule x learning rate
    "ddpm": {f"{p}_lr{lr}": dict(pred=p, lr=float(lr)) for p in ("eps", "v") for lr in ("1e-4", "2e-4", "4e-4")},
    # how training times are drawn (uniform, or logit-normal: more at middle times, Esser et al. 2024) x learning rate
    "fm": {f"{t[:2]}_lr{lr}": dict(t_sampling=t, lr=float(lr)) for t in ("uniform", "logit_normal")
           for lr in ("1e-4", "2e-4", "4e-4")},
}

# the search's winner per family (a key of GRID[family]); None until the search is read
# (GAN, DDPM, FM: group decision, 2026-10-22; VAE: 2026-10-29, the rule and the copy-aware reading agree;
# notes/benchmark_results.md §5-§9)
FINAL = {"gan": "lr5e-5_b64", "vae": "b16_lr1e-3", "ddpm": "eps_lr1e-4", "fm": "un_lr4e-4"}

# training steps per second of each configuration ALONE on one T4 (fp16). They turn the schedule (minutes of
# training) into iterations, so checkpoints, caps and the compute budgets do not depend on how many jobs share a
# GPU. None here = the search notebook measures them in its first session and keeps them in runs_*/speeds.json;
# before the final stage they are copied here from the search output (the values that search used).
SPEED = {fam: {tag: None for tag in grid} for fam, grid in GRID.items()}
# GAN: runs_gan_catalina/speeds.json of the search (KM, 21 Oct)
SPEED["gan"].update({"lr1e-4_b256": 9.06, "lr1e-4_b64": 11.97, "lr2e-4_b256": 9.02, "lr2e-4_b64": 12.13,
                     "lr5e-5_b256": 8.96, "lr5e-5_b64": 11.68})
# DDPM, FM: the SMOKE speeds that were typed into the search notebooks (KT, 19 and 21 Oct)
SPEED["ddpm"].update({"eps_lr1e-4": 11.6, "eps_lr2e-4": 11.5, "eps_lr4e-4": 11.6, "v_lr1e-4": 11.5,
                      "v_lr2e-4": 11.5, "v_lr4e-4": 11.4})
SPEED["fm"].update({"un_lr1e-4": 11.7, "un_lr2e-4": 11.7, "un_lr4e-4": 11.7, "lo_lr1e-4": 11.7,
                    "lo_lr2e-4": 11.6, "lo_lr4e-4": 11.6})
# VAE: the winner's exact value from runs_vae_catalina/speeds.json of its search (part D, 25 Oct); the other
# configurations stay None (their speeds are in the search outputs)
SPEED["vae"].update({"b16_lr1e-3": 77.42449013496925})


def search_jobs(family):
    return [f"{family}_{tag}_s0" for tag in GRID[family]]


FINAL_SPLITS = (0, 1, 2, 3, 4)          # the final stage runs the winner on these splits (5 since 2026-10-15)


def final_jobs(family, splits=FINAL_SPLITS):
    """The winner on the given splits; every final run starts fresh in the final stage's own runs folder."""
    tag = FINAL[family]
    return [f"{family}_{tag}_s{s}" for s in splits] if tag else []


In [ ]:
%%writefile astra/bench_run.py
"""Trainer and per-GPU worker of the ASTRA benchmark, shared by the four model families.

python -m astra.bench_run --family fm --stage search --queue fm_un_lr2e-4_s0 --deadline <unix time> --data <npz>
                          --dataset catalina --runs-dir <dir> --results-dir <dir> [--smoke]
python -m astra.bench_run --family fm --speedtest 300 --data <npz>        (SMOKE: training steps per second)

The controls of the comparison live here, so they are identical for GAN, VAE, DDPM and flow matching:
  - the same schedule: checkpoints after 5, 10, 20, 30, 45, 60, 90, ... minutes of training ON ONE T4 ALONE
    (search), after 1, 2, 3, 5, 7.5, 10, 15, 20, 30, 45, 60, 90, ... minutes (final: the diffusion-type models
    peak within 10 minutes, before they start copying their training curves). The minutes are turned into
    iterations with the configuration's measured speed (SPEED in astra/bench_grid.py), so the schedule does not
    change when two jobs share a GPU and a resumed run repeats exactly;
  - the same stopping rule: a run trains until 4 checkpoints in a row bring no new best validation score (and at
    least 1 h), or until the cap of its stage (search 6 h, final 20 h);
  - the same batches: the paper's resampling block with gamma = 0.25; every checkpoint is the EMA (0.999);
  - the same model selection: the checkpoint with the best validation TSTR balanced accuracy;
  - the same compute-matched view: the best checkpoint within 5, 10, 15, 30, 60 min and 2, 5, 10 h is also
    scored on test;
  - the same final evaluation (astra.bench_eval), including TSTR for conditions no training star has, at the
    selected and at the last checkpoint.
Two stages. "search": every configuration of the family's GRID on split 0, validation scores only. "final": the
winning configuration on splits 0-4 with test scores; every final run starts fresh (the notebooks keep the final
runs in their own folder, so split 0 does not continue the search run; since 2026-10-22). A family module
supplies DEFAULTS, GRID, FINAL, SPEED, JOBS, TINY and a Family subclass named MODEL.
"""
import argparse
import copy
import hashlib
import importlib
import json
import os
import shutil
import time
import traceback
from pathlib import Path

import numpy as np
import torch

from . import bench_data
from .bench_eval import (EVAL, EVAL_TINY, Fidelity, condition_test, evaluate_checkpoint, final_evaluation, run_real,
                         save_npz, tstr_only)
from .classifier import CNNEnsemble, amp_on, train_feature_cnn
from .common import WORK, Deadline, Logger, load_json, restore_rng, save_json, torch_save
from .metrics import g_score
from .sampling import ClassResampler

STAGES = ("search", "final")
LATER_MIN = (60, 90, 120, 180, 240, 300, 360, 420, 480, 540, 600, 720, 840, 960, 1080, 1200)
COMMON = dict(gamma=0.25, ema=0.999, amp=True, rep="unit",
              # checkpoints of each stage, in minutes of training on one T4 alone (turned into iterations with
              # `speed`); the final stage looks closer at the first 10 minutes (decided 2026-10-21)
              ckpt_min={"search": (5, 10, 20, 30, 45) + LATER_MIN,
                        "final": (1, 2, 3, 5, 7.5, 10, 15, 20, 30, 45) + LATER_MIN},
              cap_h={"search": 6.0, "final": 20.0},     # training cap of each stage, same units
              patience=4, min_h=1.0,          # stop when 4 checkpoints in a row bring no new best (after >= 1 h)
              # compute-matched view: the best checkpoint within each budget, in minutes on one T4 alone
              budgets_min=(5, 10, 15, 30, 60, 120, 300, 600),
              speed=None,                     # steps per second alone on a T4 (from the family's SPEED table)
              iters=None, n_ckpt=None,        # tests only: a fixed number of steps with n_ckpt equal checkpoints
              chunk=20, save_every_s=1800)
# smoke and offline tests: search checkpoints at 12, 24, 36 steps, final at 6, 12, 24, 36, 48
TINY = dict(ckpt_min={"search": (0.2, 0.4, 0.6, 0.8), "final": (0.1, 0.2, 0.4, 0.6, 0.8)},
            cap_h={"search": 0.6 / 60, "final": 0.8 / 60}, patience=1, min_h=0.0, budgets_min=(0.3,), speed=1.0,
            save_every_s=20)


def grid_jobs(prefix, grid, final):
    """Job names of a family: every grid configuration on split 0 (the search), and the final configuration
    on splits 1-4 (its split 0 has the search run's name; the final stage runs it fresh in its own folder)."""
    jobs = {f"{prefix}_{tag}_s0": dict(seed=0, **cfg) for tag, cfg in grid.items()}
    if final:
        jobs.update({f"{prefix}_{final}_s{s}": dict(seed=s, **grid[final]) for s in (1, 2, 3, 4)})
    return jobs


def budget_key(minutes):
    """The name of a compute budget in the results: 30 -> '30min' (astra.bench_report.budget_hours reads it)."""
    return f"{minutes:g}min"


def job_tag(job, family):
    """gan_lr1e-4_b64_s0 -> lr1e-4_b64"""
    return job[len(family) + 1:job.rindex("_s")]


class Family:
    """What a model family provides to the trainer. A subclass builds its networks in __init__, registers
    them with track() and make_ema(), and implements step(), sample_batch(), losses() and eval_nets()."""
    SWEEPS = ()                        # other sampling settings to score at the selected checkpoint
    STATS = ("loss",)                  # names of the numbers step() returns

    def __init__(self, cfg, data, device):
        self.cfg, self.data, self.dev = cfg, data, device
        self.amp = amp_on(cfg.get("amp", True), device)
        self._tracked, self._ema, self.n_bad = {}, [], 0

    def track(self, **objs):
        """Networks, optimisers and gradient scalers: everything a resumed run must restore."""
        self._tracked.update(objs)

    def make_ema(self, net):
        ema = copy.deepcopy(net).eval().requires_grad_(False)
        self._ema.append((ema, net))
        return ema

    def update_ema(self):
        with torch.no_grad():
            for ema, net in self._ema:
                for pe, p in zip(ema.parameters(), net.parameters()):
                    pe.lerp_(p, 1 - self.cfg["ema"])

    def state_dict(self):
        return {k: v.state_dict() for k, v in self._tracked.items()}

    def load_state_dict(self, s):
        for k, v in self._tracked.items():
            v.load_state_dict(s[k])

    def eval_nets(self):
        """name -> network, for everything sample() and losses() use (this is what a checkpoint stores)."""
        raise NotImplementedError

    def sampler_state(self):
        return {k: copy.deepcopy(v.state_dict()) for k, v in self.eval_nets().items()}

    def load_sampler_state(self, s):
        for k, v in self.eval_nets().items():
            v.load_state_dict(s[k])

    def n_params(self):
        return {k: sum(p.numel() for p in v.parameters()) for k, v in self._tracked.items()
                if isinstance(v, torch.nn.Module)}

    def cond(self, idx):
        d = self.data
        return d["ph"][idx], d["camp"][idx], d["y"][idx]

    def to_mag(self, u, idx):
        return u.float() * self.data["mag_scale"][idx][:, None]

    @torch.no_grad()
    def sample(self, idx, seed, batch=4096, **opts):
        """Magnitudes of one generated curve per condition row idx[i] (EMA weights, full precision output)."""
        g = torch.Generator(device=self.dev).manual_seed(seed)
        x = torch.cat([self.sample_batch(idx[i:i + batch], g, **opts) for i in range(0, len(idx), batch)])
        bad = ~torch.isfinite(x)
        self.n_bad = int(bad.sum())        # reported with the scores; a non-finite value is replaced by 0
        return torch.where(bad, torch.zeros_like(x), x) if self.n_bad else x


class preserve_rng:
    """Evaluation must not change the random numbers of training (a resumed run then repeats exactly)."""

    def __enter__(self):
        self.cpu = torch.get_rng_state()
        self.cuda = torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None

    def __exit__(self, *exc):
        torch.set_rng_state(self.cpu)
        if self.cuda is not None:
            torch.cuda.set_rng_state_all(self.cuda)


class BenchRun:
    """One training run of one family: schedule, checkpoints, their evaluation, stopping rule, resume, scores."""

    def __init__(self, cfg, data, fidelity, family_cls, out, deadline, device):
        self.cfg, self.data, self.fid, self.dev = cfg, data, fidelity, device
        self.out, self.deadline = out, deadline
        (out / "ckpt").mkdir(parents=True, exist_ok=True)
        self.log = Logger(out / "log.txt")
        self.metrics = load_json(out / "metrics.json", [])
        self.ckpts = load_json(out / "ckpts.json", [])           # iteration and training seconds of each checkpoint
        torch.manual_seed(cfg["seed"])
        self.model = family_cls(cfg, data, device)
        self.sampler = ClassResampler(data["y_np"][data["train_idx_np"]], cfg["gamma"], seed=cfg["seed"],
                                      n_classes=data["n_classes"])
        self.it, self.train_s, self.n_ckpt = 0, 0.0, 0
        self.hist, self.n_hist = None, 0
        self.sched, self.cap = cfg["ckpt_its"], cfg["cap_its"]

    def draw(self):
        """Row indices of one training batch from the resampling block."""
        return self.data["train_idx"][torch.from_numpy(self.sampler.sample(self.cfg["batch"])).to(self.dev)]

    def sync(self):
        if str(self.dev).startswith("cuda"):
            torch.cuda.synchronize()

    def solo_h(self, it):
        return it / self.cfg["speed"] / 3600 if self.cfg.get("speed") else float("nan")

    # ---------------- state ----------------
    def save_state(self):
        torch_save({"model": self.model.state_dict(), "sampler": self.sampler, "it": self.it,
                    "train_s": self.train_s, "n_ckpt": self.n_ckpt, "hist": self.hist, "n_hist": self.n_hist,
                    "rng_cpu": torch.get_rng_state(),
                    "rng_cuda": torch.cuda.get_rng_state_all() if str(self.dev).startswith("cuda") else None},
                   self.out / "last.pt")

    def load_state(self):
        p = self.out / "last.pt"
        if not p.exists():
            return False
        s = torch.load(p, map_location=self.dev, weights_only=False)
        self.model.load_state_dict(s["model"])
        self.sampler, self.it, self.train_s, self.n_ckpt = s["sampler"], s["it"], s["train_s"], s["n_ckpt"]
        self.hist, self.n_hist = s["hist"], s["n_hist"]
        restore_rng(s, self.dev)
        return True

    # ---------------- training ----------------
    def next_ckpt(self):
        return self.sched[self.n_ckpt] if self.n_ckpt < len(self.sched) else self.cap

    def train_chunk(self):
        n = min(self.cfg["chunk"], self.next_ckpt() - self.it)     # land exactly on the checkpoint's step
        self.sync()
        t0 = time.time()
        for _ in range(n):
            s = self.model.step(self.draw, self.it)
            self.hist = s if self.hist is None else self.hist + s
            self.n_hist += 1
            self.it += 1
        self.sync()
        self.train_s += time.time() - t0   # only training counts; evaluation does not

    def checkpoint(self):
        self.n_ckpt += 1
        torch_save(self.model.sampler_state(), self.out / "ckpt" / f"ema_{self.n_ckpt:02d}.pt")
        self.ckpts = [c for c in self.ckpts if c["ckpt"] < self.n_ckpt] + [
            {"ckpt": self.n_ckpt, "it": self.it, "train_s": self.train_s, "solo_h": self.solo_h(self.it)}]
        save_json(self.ckpts, self.out / "ckpts.json")
        h = (self.hist / max(self.n_hist, 1)).tolist()
        stats = ", ".join(f"{n} {v:.4f}" for n, v in zip(self.model.STATS, h))
        self.log(f"checkpoint {self.n_ckpt}: iteration {self.it:,} = {self.solo_h(self.it):.2f} h on one T4 alone "
                 f"({self.train_s / 60:.1f} min here, {self.it / max(self.train_s, 1e-9):.1f} it/s) | {stats}")
        with open(self.out / "train_curve.csv", "a") as f:
            f.write(f"{self.n_ckpt},{self.it},{self.train_s:.1f}," + ",".join(f"{v:.6f}" for v in h) + "\n")
        self.hist, self.n_hist = None, 0
        self.save_state()

    def evaluate_pending(self):
        done = {m["ckpt"] for m in self.metrics}
        todo = [k for k in range(1, self.n_ckpt + 1) if k not in done]
        if not todo:
            return
        live = self.model.sampler_state()
        meta = {c["ckpt"]: c for c in self.ckpts}
        with preserve_rng():
            for k in todo:
                self.model.load_sampler_state(torch.load(self.out / "ckpt" / f"ema_{k:02d}.pt", map_location=self.dev))
                m = evaluate_checkpoint(self.model, self.fid, self.data, self.cfg, k, self.dev)
                m.update({key: meta[k][key] for key in ("it", "train_s", "solo_h")})
                self.metrics.append(m)
                self.metrics.sort(key=lambda a: a["ckpt"])
                save_json(self.metrics, self.out / "metrics.json")
                self.log(f"checkpoint {k}: val TSTR balanced acc {100 * m['val_bal_acc']:.2f} | FID {m['fid']:.2f} | "
                         f"density {np.nanmean(m['density']):.3f} recall {np.nanmean(m['recall']):.3f} | "
                         f"memorisation ratio {np.nanmean(m['mem_ratio']):.2f}, copies "
                         f"{100 * np.nanmean(m['copy_share']):.1f}% | loss train {m['loss_train']:.4f} "
                         f"val {m['loss_val']:.4f} ({m['eval_s']:.0f} s)"
                         + (f" | !! {m['nonfinite']} non-finite generated values set to 0" if m["nonfinite"] else ""))
        self.model.load_sampler_state(live)

    def stop_reason(self):
        """None while the run goes on; 'cap' or 'no gain' when it ends (the metrics are up to date here)."""
        c, m = self.cfg, self.metrics
        if c.get("patience") and len(m) > c["patience"] and self.it >= c["min_its"]:
            best = int(np.argmax([a["val_bal_acc"] for a in m]))
            if len(m) - 1 - best >= c["patience"]:
                return "no gain"          # checked first: a run that stops at its cap may continue in the next stage
        return "cap" if self.it >= self.cap else None

    # ---------------- the whole run ----------------
    def run(self, reserve_s):
        c, stage = self.cfg, self.cfg["stage"]
        if (self.out / "final.json").exists() or (stage == "search" and (self.out / "search.json").exists()):
            return "done"
        if stage == "final" and (self.out / "search.json").exists():
            raise RuntimeError(f"{self.out} holds a search run: final runs start fresh in the final stage's own folder")
        if self.load_state():
            self.log(f"{stage} stage resumed at iteration {self.it:,} ({self.solo_h(self.it):.2f} h on one T4 alone)")
        else:            # a fresh run: its random numbers must not depend on what ran between __init__ and here
            torch.manual_seed(c["seed"] + 10_000)
            self.log(f"parameters: {self.model.n_params()} | schedule (iterations): {self.sched} | cap {self.cap:,}")
        self.evaluate_pending()
        t_save = time.time()
        while True:
            why = self.stop_reason()
            if why:
                break
            if self.deadline.passed(reserve_s):
                self.save_state()
                self.log(f"time budget of the session reached at iteration {self.it:,}: state saved, resume next session")
                return "paused"
            self.train_chunk()
            if self.it >= self.next_ckpt():
                self.checkpoint()
                self.evaluate_pending()
                t_save = time.time()
            elif time.time() - t_save > c["save_every_s"]:
                self.save_state()
                t_save = time.time()
        return self.finish_search(why) if stage == "search" else self.finish(why)

    def best(self, max_it=None):
        m = [a for a in self.metrics if max_it is None or a["it"] <= max_it]
        return max(m, key=lambda a: a["val_bal_acc"]) if m else None

    def finish_search(self, why):
        """The search scores validation only. The state is kept (the final stage starts its own runs)."""
        b = self.best()
        self.save_state()
        save_json({"stage": "search", "stop": why, "iters": self.it, "train_s": self.train_s,
                   "solo_h": self.solo_h(self.it), "best_ckpt": b["ckpt"], "best_val_bal_acc": b["val_bal_acc"],
                   "best_it": b["it"], "best_solo_h": b["solo_h"], "n_params": self.model.n_params()},
                  self.out / "search.json")
        self.log(f"search run ends ({why}) at {self.solo_h(self.it):.2f} h: best checkpoint {b['ckpt']} at "
                 f"{b['solo_h']:.2f} h, val TSTR balanced acc {100 * b['val_bal_acc']:.2f}")
        return "done"

    def finish(self, why):
        """Model selection on validation stars, then the test scores of the selected checkpoint, then the
        compute-matched view: the best checkpoint within each budget, scored on test (TSTR), then TSTR for
        conditions no training star has, at the selected and at the last checkpoint."""
        m = self.metrics
        gs = g_score(np.nan_to_num([a["density"] for a in m]), np.nan_to_num([a["recall"] for a in m]))
        for a, g in zip(m, gs):
            a["g_score"] = float(g)
        save_json(m, self.out / "metrics.json")
        best = self.best()
        self.model.load_sampler_state(torch.load(self.out / "ckpt" / f"ema_{best['ckpt']:02d}.pt", map_location=self.dev))
        self.log(f"run ends ({why}) at {self.solo_h(self.it):.2f} h; selected checkpoint {best['ckpt']} at "
                 f"{best['solo_h']:.2f} h (val TSTR balanced acc {100 * best['val_bal_acc']:.2f}); "
                 f"the G-score would pick checkpoint {m[int(np.argmax(gs))]['ckpt']}")
        with preserve_rng():
            parts = final_evaluation(self.model, self.fid, self.data, self.cfg, self.out, self.deadline, self.dev,
                                     self.log, sweeps=self.model.SWEEPS[:self.cfg["max_sweeps"]])
        if parts is None:
            return "paused"
        budgets = {}
        for B in self.cfg["budgets_min"]:
            b = self.best(max_it=round(B * 60 * self.cfg["speed"]))
            if b is None:
                continue
            key = budget_key(B)
            budgets[key] = {"ckpt": b["ckpt"], "it": b["it"], "solo_h": b["solo_h"], "val_bal_acc": b["val_bal_acc"],
                            "mem_ratio": b["mem_ratio"], "copy_share": b["copy_share"]}
            if b["ckpt"] == best["ckpt"]:
                budgets[key]["test"] = "tstr"                  # the same checkpoint as the headline
                continue
            self.model.load_sampler_state(torch.load(self.out / "ckpt" / f"ema_{b['ckpt']:02d}.pt", map_location=self.dev))
            with preserve_rng():
                got = tstr_only(self.model, self.data, self.cfg, self.out, self.deadline, self.dev, self.log,
                                name=f"budget_{key}")
            if got is None:
                return "paused"
            budgets[key]["test"] = f"budget_{key}"
        # does a model that copies still generate useful curves for conditions it has not seen? (n_sel curves,
        # 3 classifier runs, as the sampling sweeps; the selected checkpoint's own conditions = sweep_default)
        last = m[-1]["ckpt"]
        novel = [("novel_selected", best["ckpt"], True)]
        if last != best["ckpt"]:
            novel += [("default_last", last, False), ("novel_last", last, True)]
        for name, k, new in novel:
            self.model.load_sampler_state(torch.load(self.out / "ckpt" / f"ema_{k:02d}.pt", map_location=self.dev))
            with preserve_rng():
                got = condition_test(self.model, self.data, self.cfg, self.out, self.deadline, self.dev, self.log,
                                     name, novel=new, ckpt=k)
            if got is None:
                return "paused"
        parts = load_json(self.out / "final_partial.json", {})
        save_json({"stage": "final", "stop": why, "selected_ckpt": best["ckpt"], "selected_solo_h": best["solo_h"],
                   "last_ckpt": last, "last_solo_h": m[-1]["solo_h"],
                   "g_score_ckpt": m[int(np.argmax(gs))]["ckpt"], "iters": self.it, "train_s": self.train_s,
                   "solo_h": self.solo_h(self.it), "n_params": self.model.n_params(), "budgets": budgets,
                   "parts": parts}, self.out / "final.json")
        (self.out / "last.pt").unlink(missing_ok=True)
        self.log("finished")
        return "done"


def feature_cnn(runs_dir, data, device, max_epochs=20, log=print):
    """The classifier whose last conv block is the feature space of the fidelity scores: trained once per
    split on the real training curves (as in the GAN notebook), then reused by every job of that split."""
    path = runs_dir / f"feature_cnn_s{data['seed']}.pt"
    model = CNNEnsemble(1, data["n_classes"], data["L"]).to(device)
    if not path.exists():
        tr, va = (bench_data.subset(data, data[f"{s}_idx"]) for s in ("train", "val"))
        model, curve = train_feature_cnn(tr, va, data["n_classes"], max_epochs=max_epochs, seed=data["seed"],
                                         device=device, log=log)
        torch_save(model.state_dict(), path)
        save_json(curve, runs_dir / f"feature_cnn_s{data['seed']}_curve.json")
    model.load_state_dict(torch.load(path, map_location=device))
    h = hashlib.md5()
    for v in model.state_dict().values():
        h.update(v.detach().cpu().numpy().tobytes())
    data["feat_hash"] = h.hexdigest()[:12]      # stored with every result that uses this feature space
    return model.eval()


def job_config(job, fam, stage, tiny, speeds=None):
    """real_s<seed> = the reference rows on real data; any other name is looked up in the family's JOBS.
    The schedule in minutes becomes iterations here, with the configuration's measured speed."""
    assert stage in STAGES, stage
    cfg = dict(COMMON, **EVAL, seed=int(job.rsplit("_s", 1)[1]), job=job, stage=stage)
    if job.startswith("real_s"):
        return dict(cfg, **(dict(TINY, **EVAL_TINY, tiny=True) if tiny else {}))
    cfg.update(fam.DEFAULTS)
    cfg["tag"] = job_tag(job, fam.__name__.rsplit("_", 1)[1])
    cfg.update(fam.JOBS.get(job) or fam.GRID[cfg["tag"]])     # any grid configuration on any split (SMOKE, tests)
    if tiny:
        cfg.update(TINY, **EVAL_TINY, tiny=True)
        cfg.update(getattr(fam, "TINY", {}))
    else:
        cfg["speed"] = fam.SPEED.get(cfg["tag"]) or (speeds or {}).get(cfg["tag"])
        if not cfg["speed"]:
            raise ValueError(f"{job}: no measured speed for {cfg['tag']} (SPEED in astra/bench_grid.py, or "
                             f"runs_*/speeds.json of the search output)")
    spd = cfg["speed"]
    if cfg.get("iters"):                       # tests: a fixed number of steps, n_ckpt equal checkpoints, no early stop
        n = cfg["n_ckpt"]
        cfg["ckpt_its"] = [round(cfg["iters"] * (k + 1) / n) for k in range(n)]
        cfg["cap_its"], cfg["patience"] = cfg["iters"], None
    else:
        cap = round(cfg["cap_h"][stage] * 3600 * spd)
        its = sorted({max(1, round(m * 60 * spd)) for m in cfg["ckpt_min"][stage]})
        cfg["ckpt_its"] = [i for i in its if i < cap] + [cap]
        cfg["cap_its"] = cap
    cfg["min_its"] = round(cfg["min_h"] * 3600 * spd)
    cfg["kl_warmup_its"] = round(cfg.get("kl_warmup_min", 0) * 60 * spd)
    return cfg


def write_result(job, family, dataset, cfg, data, out, results_dir):
    """One self-contained file per finished job; the notebooks' reports and the final comparison read these.
    A search run writes its validation scores only (no test predictions)."""
    results_dir.mkdir(parents=True, exist_ok=True)
    real = job.startswith("real_s")
    res = {"job": job, "family": "real" if real else family, "dataset": dataset, "stage": cfg["stage"],
           "tag": cfg.get("tag"), "seed": cfg["seed"], "data_hash": data["hash"], "feat_hash": data.get("feat_hash"),
           "classes": data["classes"], "train_counts": data["train_counts"].tolist(), "rare": data["rare"].tolist(),
           "config": {k: v for k, v in cfg.items() if isinstance(v, (int, float, str, bool, tuple, list, dict, type(None)))},
           "metrics": load_json(out / "metrics.json", [])}
    if not real and cfg["stage"] == "search":
        res.update(load_json(out / "search.json"))
        save_json(res, results_dir / f"{job}.json")
        return
    res.update(load_json(out / "final.json") or {"parts": load_json(out / "final_partial.json", {})})
    if real:
        res["stage"] = "final"
    save_json(res, results_dir / f"{job}.json")
    te = data["test_idx_np"]
    preds = dict(np.load(out / "final_pred.npz"))
    save_npz(results_dir / f"{job}_pred.npz", True, y_test=data["y_np"][te], star_test=data["star"][te], **preds)
    if (out / "samples.npz").exists():        # the synthetic curves, for the side-by-side figure and one-space fidelity
        shutil.copy2(out / "samples.npz", results_dir / f"{job}_samples.npz")


def speedtest(fam, cfg_over, data, device, n_steps=30, warm=10):
    """Training steps per second of one configuration (the batches, fp16 and EMA of a real run)."""
    cfg = dict(COMMON, **EVAL, **fam.DEFAULTS, seed=0, kl_warmup_its=0)
    cfg.update(cfg_over)
    torch.manual_seed(0)
    model = fam.MODEL(cfg, data, device)
    rs = ClassResampler(data["y_np"][data["train_idx_np"]], cfg["gamma"], seed=0, n_classes=data["n_classes"])
    draw = lambda: data["train_idx"][torch.from_numpy(rs.sample(cfg["batch"])).to(device)]
    sync = (lambda: torch.cuda.synchronize()) if str(device).startswith("cuda") else (lambda: None)
    for i in range(warm):
        model.step(draw, 10 ** 9)
    sync(); t = time.time()
    for i in range(n_steps):
        model.step(draw, 10 ** 9)
    sync()
    return n_steps / (time.time() - t), model


def benchmark(fam, data, device, quick=False):
    """SMOKE only: the steps per second of EVERY grid configuration alone on this GPU (they become the family's
    SPEED table), the sampling speed, and how long the classifier fits of an evaluation take.
    quick (my offline CPU test): fewer steps and curves, a quarter of a classifier epoch."""
    n_steps, n_sample, frac = (3, 256, 0.25) if quick else (40, 4096, 1.0)
    from .bench_eval import fit_classifiers
    speeds, model = {}, None
    for tag, over in fam.GRID.items():
        speeds[tag], m = speedtest(fam, over, data, device, n_steps=n_steps, warm=2 if quick else 10)
        model = model or m
    cfg = dict(COMMON, **EVAL, **fam.DEFAULTS, seed=0)
    sync = (lambda: torch.cuda.synchronize()) if str(device).startswith("cuda") else (lambda: None)
    out = {"speeds": speeds, "n_params": model.n_params()}
    idx = bench_data.balanced_idx(data, n_sample, seed=0)
    model.sample(idx[:64], seed=0)
    sync(); t = time.time()
    model.sample(idx, seed=0)
    sync()
    out["curves_per_s"] = len(idx) / (time.time() - t)
    tr, va, te = (bench_data.subset(data, data[f"{s}_idx"]) for s in ("train", "val", "test"))
    for runs in (cfg["sel_runs"], cfg["final_runs"]):
        t = time.time()
        fit_classifiers([tr], va, te, data["n_classes"], runs=runs, ens=cfg["ens"], max_epochs=frac, device=device,
                        log=lambda *a: None)
        out[f"clf_epoch_s_{runs * cfg['ens']}cnn"] = (time.time() - t) / frac
    out["peak_gpu_gb"] = torch.cuda.max_memory_allocated() / 1e9 if str(device).startswith("cuda") else 0.0
    return out


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--family", required=True)
    ap.add_argument("--stage", choices=STAGES, default="search")
    ap.add_argument("--queue", default="")
    ap.add_argument("--deadline", type=float, default=1e12)
    ap.add_argument("--data", required=True)
    ap.add_argument("--dataset", default="catalina")
    ap.add_argument("--runs-dir", default=str(WORK / "runs"))
    ap.add_argument("--results-dir", default=str(WORK / "results"))
    ap.add_argument("--smoke", action="store_true")
    ap.add_argument("--speedtest", type=int, default=0, help="only measure: this many steps of the default config")
    ap.add_argument("--speeds", default="", help="json file: grid key -> steps per second alone on a T4")
    a = ap.parse_args()
    runs_dir, results_dir = Path(a.runs_dir), Path(a.results_dir)
    tiny = a.smoke or os.environ.get("ASTRA_LOCAL_TEST") == "1"      # my offline CPU test runs tiny jobs
    dev = "cuda" if torch.cuda.is_available() else "cpu"              # cpu only in my offline tests
    deadline = Deadline(a.deadline)
    torch.backends.cudnn.benchmark = True
    fam = importlib.import_module(f"astra.bench_{a.family}")
    if a.speedtest:                       # SMOKE: is it worth running two jobs on one GPU?
        data = bench_data.load(a.data, 0, dev, "unit")
        sps, _ = speedtest(fam, {}, data, dev, n_steps=a.speedtest)
        print(json.dumps({"family": a.family, "it_per_s": sps}), flush=True)
        return
    for job in [j for j in a.queue.split(",") if j]:
        out = runs_dir / job
        status_path = out / "status.json"
        st = load_json(status_path, {}) or {}
        real = job.startswith("real_s")
        if st.get("state") == "done" and (real or st.get("stage") in ("final", a.stage)):
            print(f"[{job}] already done, skipping", flush=True)
            continue
        log = Logger(out / "log.txt")
        log(f"start {job}, {a.stage} stage ({'tiny' if tiny else 'real run'})")
        t0 = time.time()
        if dev == "cuda":
            torch.cuda.reset_peak_memory_stats()
        try:
            speeds = json.loads(Path(a.speeds).read_text()) if a.speeds and Path(a.speeds).exists() else None
            cfg = job_config(job, fam, a.stage, tiny, speeds)
            save_json(cfg, out / "config.json")
            data = bench_data.load(a.data, cfg["seed"], dev, cfg["rep"])
            if real:
                state = "done" if run_real(data, cfg, out, deadline, dev, log) is not None else "paused"
            else:
                fid = Fidelity(data, feature_cnn(runs_dir, data, dev, 3 if tiny else 20, log), cfg["k"])
                state = BenchRun(cfg, data, fid, fam.MODEL, out, deadline, dev).run(reserve_s=900)
            if state == "done":
                write_result(job, a.family, a.dataset, cfg, data, out, results_dir)
            del data
        except Exception:
            log(traceback.format_exc())
            state = "failed"
        save_json({"state": state, "stage": "final" if real else a.stage, "hours_this_session": (time.time() - t0) / 3600,
                   "hours_total": st.get("hours_total", 0) + (time.time() - t0) / 3600,
                   "peak_gpu_gb": torch.cuda.max_memory_allocated() / 1e9 if dev == "cuda" else 0}, status_path)
        log(f"{job}: {state}")
        if state == "paused":
            break                 # out of time: the rest of the queue waits for the next session


if __name__ == "__main__":
    main()


In [ ]:
%%writefile astra/bench_net.py
"""The network shared by the diffusion model and the flow-matching model of the benchmark.

Both models learn a function net(x_t, t | condition) that takes a noisy light curve and says how to clean it.
They use exactly this network, with the same size, so a difference between them comes from the training
objective and the sampler, not from the architecture.

A 1-D U-Net (ours; 3.99M parameters, sized to the paper's GAN generator: 3.99M on Catalina, 3.71M on ZTF):
  - input channels: the noisy curve, and sin / cos features of the phase of every point. The points of a
    folded light curve sit at irregular phases, and phase is periodic, so the network is told each point's
    phase through periodic features, and its convolutions wrap around from the last point to the first;
  - three resolutions (L, L/2, L/4) with residual blocks and skip connections;
  - the time t, the class and the amplitude enter every block as one added embedding vector. One extra
    class row means "no class": dropping the class in training allows classifier-free guidance at sampling.
"""
import math

import torch
import torch.nn as nn
import torch.nn.functional as F


def conv3(cin, cout, stride=1):
    return nn.Conv1d(cin, cout, 3, stride=stride, padding=1, padding_mode="circular")


def sinusoidal(t, dim):
    """The usual sinusoidal embedding of a scalar per sample."""
    half = dim // 2
    f = torch.exp(-math.log(10000.0) * torch.arange(half, device=t.device) / half)
    a = t[:, None].float() * f[None]
    return torch.cat([a.sin(), a.cos()], 1)


class ResBlock(nn.Module):
    def __init__(self, cin, cout, emb):
        super().__init__()
        self.n1, self.c1 = nn.GroupNorm(8, cin), conv3(cin, cout)
        self.e = nn.Linear(emb, cout)
        self.n2, self.c2 = nn.GroupNorm(8, cout), conv3(cout, cout)
        nn.init.zeros_(self.c2.weight); nn.init.zeros_(self.c2.bias)       # every block starts as the identity
        self.skip = nn.Conv1d(cin, cout, 1) if cin != cout else nn.Identity()

    def forward(self, h, e):
        r = self.c1(F.silu(self.n1(h))) + self.e(e)[:, :, None]
        return self.skip(h) + self.c2(F.silu(self.n2(r)))


class UNet1d(nn.Module):
    def __init__(self, n_classes, base=88, n_mid=2, n_freq=4, use_phase=True):
        super().__init__()
        self.n_classes, self.base, self.n_freq = n_classes, base, n_freq
        self.use_phase = use_phase      # False = the ablation: the network is not told where the points are in phase
        b, emb = base, 4 * base
        self.t_mlp = nn.Sequential(nn.Linear(b, emb), nn.SiLU(), nn.Linear(emb, emb))
        self.y_emb = nn.Embedding(n_classes + 1, emb)               # row n_classes = "no class"
        self.a_mlp = nn.Sequential(nn.Linear(1, emb), nn.SiLU(), nn.Linear(emb, emb))
        self.inp = conv3(1 + 2 * n_freq, b)
        self.enc0 = nn.ModuleList([ResBlock(b, b, emb), ResBlock(b, b, emb)])
        self.down1 = conv3(b, 2 * b, stride=2)
        self.enc1 = nn.ModuleList([ResBlock(2 * b, 2 * b, emb), ResBlock(2 * b, 2 * b, emb)])
        self.down2 = conv3(2 * b, 4 * b, stride=2)
        self.mid = nn.ModuleList([ResBlock(4 * b, 4 * b, emb) for _ in range(n_mid)])
        self.up1 = conv3(4 * b, 2 * b)
        self.dec1 = nn.ModuleList([ResBlock(4 * b, 2 * b, emb), ResBlock(2 * b, 2 * b, emb)])
        self.up0 = conv3(2 * b, b)
        self.dec0 = nn.ModuleList([ResBlock(2 * b, b, emb), ResBlock(b, b, emb)])
        self.out_norm, self.out = nn.GroupNorm(8, b), conv3(b, 1)
        nn.init.zeros_(self.out.weight); nn.init.zeros_(self.out.bias)

    def forward(self, x, t, ph, camp, y):
        """x (B, L) noisy curve, t (B,) in [0, 1], ph (B, L) phases, camp (B,) amplitude condition,
        y (B,) class ids (n_classes = no class). Returns (B, L)."""
        e = self.t_mlp(sinusoidal(t * 1000, self.base)) + self.y_emb(y) + self.a_mlp(camp[:, None])
        k = 2 * math.pi * torch.arange(1, self.n_freq + 1, device=x.device)[None, :, None] * ph[:, None, :]
        feats = torch.cat([k.sin(), k.cos()], 1)
        if not self.use_phase:           # same network, same size, but the phase features are blanked
            feats = torch.zeros_like(feats)
        h0 = self.inp(torch.cat([x[:, None], feats], 1))
        for blk in self.enc0:
            h0 = blk(h0, e)
        h1 = self.down1(h0)
        for blk in self.enc1:
            h1 = blk(h1, e)
        h = self.down2(h1)
        for blk in self.mid:
            h = blk(h, e)
        h = torch.cat([self.up1(F.interpolate(h, size=h1.shape[-1], mode="nearest")), h1], 1)
        for blk in self.dec1:
            h = blk(h, e)
        h = torch.cat([self.up0(F.interpolate(h, size=h0.shape[-1], mode="nearest")), h0], 1)
        for blk in self.dec0:
            h = blk(h, e)
        return self.out(F.silu(self.out_norm(h)))[:, 0]


def guided(net, x, t, ph, camp, y, w):
    """Classifier-free guidance: w = 1 is the plain conditional model; w > 1 pushes the output further
    from what the model would do without knowing the class (2 network calls instead of 1)."""
    out = net(x, t, ph, camp, y)
    if w == 1.0:
        return out
    none = net(x, t, ph, camp, torch.full_like(y, net.n_classes))
    return none + w * (out - none)


In [ ]:
%%writefile astra/bench_ddpm.py
"""Benchmark family 3 — diffusion: a conditional DDPM (Ho et al. 2020) with a DDIM sampler (Song et al. 2021).

Forward process: a real curve x0 is mixed with Gaussian noise in T = 1000 steps,
    x_t = sqrt(ab_t) * x0 + sqrt(1 - ab_t) * eps,    eps ~ N(0, I),
where ab_t falls from 1 (clean) to about 0 (pure noise) along the cosine schedule of Nichol & Dhariwal (2021).
Training: pick t at random, and train the network to predict the noise eps from x_t (mean squared error).
Sampling: start from pure noise and remove it step by step. DDIM does this deterministically and can skip
steps, so a curve costs `nfe` network calls (default 32) instead of 1000.
Each step first estimates the clean curve, x0 = (x_t - sqrt(1 - ab_t) * eps) / sqrt(ab_t). At the very last
steps of the cosine schedule sqrt(ab_t) is about 5e-5, so this divides by almost zero and the estimate is
rounding noise. Sampling therefore starts at the noisiest step with ab_t >= 0.001 (step 979 of 1000 for the
cosine schedule), where x_t is still 99.9% noise. Training uses all 1000 steps.

The network (astra.bench_net.UNet1d) and its conditioning are shared with the flow-matching model.
In 10% of the training samples the class is hidden, which allows classifier-free guidance at sampling time.
"""
import math

import torch
import torch.nn as nn
import torch.nn.functional as F

from .bench_net import UNet1d, guided
from . import bench_grid
from .bench_run import Family, grid_jobs
from .classifier import autocast

DEFAULTS = dict(batch=256, lr=2e-4, base=88, T=1000, schedule="cosine", pred="eps", p_uncond=0.1, grad_clip=1.0,
                nfe=32, guidance=1.0, chunk=20, use_phase=True,
                ab_min=1e-3)       # ours: sampling starts at the noisiest step that still has ab_t >= ab_min
GRID, FINAL, SPEED = bench_grid.GRID["ddpm"], bench_grid.FINAL["ddpm"], bench_grid.SPEED["ddpm"]   # astra/bench_grid.py
JOBS = grid_jobs("ddpm", GRID, FINAL)
TINY = dict(nfe=4)
LOSS_T = (0.05, 0.2, 0.4, 0.6, 0.8, 0.95)             # noise levels (t / T) at which losses() is measured


def alpha_bar(T, schedule):
    """ab_t for t = 0 .. T-1: the share of the signal's variance that is left after t+1 noising steps."""
    if schedule == "cosine":
        t = torch.linspace(0, T, T + 1, dtype=torch.float64) / T
        f = torch.cos((t + 0.008) / 1.008 * math.pi / 2) ** 2
        betas = (1 - f[1:] / f[:-1]).clamp(max=0.999)
    elif schedule == "linear":
        betas = torch.linspace(1e-4, 0.02, T, dtype=torch.float64)
    else:
        raise ValueError(schedule)
    return torch.cumprod(1 - betas, 0).float()


class MODEL(Family):
    SWEEPS = ({"nfe": 8}, {"guidance": 2.0}, {"nfe": 4}, {"nfe": 16}, {"nfe": 64}, {"guidance": 1.5})

    def __init__(self, cfg, data, device):
        super().__init__(cfg, data, device)
        self.T = cfg["T"]
        self.ab = alpha_bar(self.T, cfg["schedule"]).to(device)
        self.t_start = int(torch.nonzero(self.ab >= cfg["ab_min"]).max())      # first step of every sampling run
        self.net = UNet1d(data["n_classes"], cfg["base"], use_phase=cfg["use_phase"]).to(device)
        self.ema = self.make_ema(self.net)
        self.opt = torch.optim.Adam(self.net.parameters(), lr=cfg["lr"])
        self.scaler = torch.amp.GradScaler(enabled=self.amp)
        self.track(net=self.net, ema=self.ema, opt=self.opt, scaler=self.scaler)

    def eval_nets(self):
        return {"ema": self.ema}

    def noised(self, x0, t, eps):
        a, s = self.ab[t].sqrt()[:, None], (1 - self.ab[t]).sqrt()[:, None]
        return a * x0 + s * eps, (eps if self.cfg["pred"] == "eps" else a * eps - s * x0)

    def step(self, draw, it):
        c = self.cfg
        idx = draw()
        x0, (ph, camp, y) = self.data["u"][idx], self.cond(idx)
        t = torch.randint(0, self.T, (len(idx),), device=self.dev)
        xt, target = self.noised(x0, t, torch.randn_like(x0))
        y = torch.where(torch.rand(len(idx), device=self.dev) < c["p_uncond"], torch.full_like(y, self.net.n_classes), y)
        with autocast(self.dev, self.amp):
            out = self.net(xt, t.float() / self.T, ph, camp, y)
        loss = F.mse_loss(out.float(), target)
        self.opt.zero_grad(set_to_none=True)
        self.scaler.scale(loss).backward()
        self.scaler.unscale_(self.opt)
        nn.utils.clip_grad_norm_(self.net.parameters(), c["grad_clip"])
        self.scaler.step(self.opt)
        self.scaler.update()
        self.update_ema()
        return torch.nan_to_num(loss.detach())[None]

    def sample_batch(self, j, g, nfe=None, guidance=None):
        """DDIM with eta = 0: nfe evenly spaced time steps from t_start down to 0."""
        c = self.cfg
        nfe, w = nfe or c["nfe"], c["guidance"] if guidance is None else guidance
        ph, camp, y = self.cond(j)
        x = torch.randn(len(j), self.data["L"], device=self.dev, generator=g)
        ts = torch.linspace(self.t_start, 0, nfe).round().long().tolist()
        clip = self.data["out_scale"]
        for i, t in enumerate(ts):
            ab_t = self.ab[t]
            ab_prev = self.ab[ts[i + 1]] if i + 1 < nfe else torch.ones((), device=self.dev)
            tt = torch.full((len(j),), t / self.T, device=self.dev)
            with autocast(self.dev, self.amp):
                out = guided(self.ema, x, tt, ph, camp, y, w)
            out = out.float()
            x0 = (x - (1 - ab_t).sqrt() * out) / ab_t.sqrt() if c["pred"] == "eps" else ab_t.sqrt() * x - (1 - ab_t).sqrt() * out
            x0 = x0.clamp(-clip, clip)                     # real curves lie in [-2, 2]
            eps = (x - ab_t.sqrt() * x0) / (1 - ab_t).sqrt()
            x = ab_prev.sqrt() * x0 + (1 - ab_prev).sqrt() * eps
        return self.to_mag(x, j)

    @torch.no_grad()
    def losses(self, idx, seed):
        """The training loss of the EMA network at six fixed noise levels, with fixed noise."""
        g = torch.Generator(device=self.dev).manual_seed(seed)
        x0, (ph, camp, y) = self.data["u"][idx], self.cond(idx)
        tot = 0.0
        for frac in LOSS_T:
            t = torch.full((len(idx),), int(frac * self.T), device=self.dev)
            xt, target = self.noised(x0, t, torch.randn(x0.shape, device=self.dev, generator=g))
            tot += float(F.mse_loss(self.ema(xt, t.float() / self.T, ph, camp, y), target))
        return tot / len(LOSS_T)


In [ ]:
%%writefile astra/bench_report.py
"""Tables and figures of the ASTRA benchmark, built from the result files the workers write (one per job).

Every benchmark notebook calls this on its own results. If the outputs of the other notebooks are attached
(Add Input), the same calls give the four-way comparison, because all notebooks use the same split, the same
conditions and the same classifier harness (checked through the data hash stored in every result).
"""
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

FAMILIES = {"real": "real data only", "gan": "GAN", "vae": "VAE", "ddpm": "diffusion (DDPM)", "fm": "flow matching"}
MAIN = ("gan", "vae", "ddpm", "fm")      # every final-stage job of a family is its main (searched) setting


def collect(roots, folder, stage="final"):
    """job -> result of one stage ("final": test scores; "search": validation only), from every <folder>/*.json
    below the given roots. A final result wins over a search result of the same job, then the first root wins."""
    out = {}
    for root in map(Path, roots):
        if not root.exists():
            continue
        for p in sorted(root.glob(f"**/{folder}/*.json")):
            r = json.loads(p.read_text(encoding="utf-8"))
            if "job" not in r:
                continue
            r.setdefault("stage", "final" if "parts" in r else "search")
            if r["stage"] != stage or r["job"] in out or (stage == "final" and "parts" not in r):
                continue
            r["_pred"], r["_from"] = p.with_name(p.stem + "_pred.npz"), str(p.parent)
            out[r["job"]] = r
    return out


def rare_mean(r, values):
    """Mean of a per-class list over the rare tier of the result's split (nan if there is none)."""
    v, rare = np.array(values, float), np.array(r["rare"], bool)
    return float(np.nanmean(v[rare])) if rare.any() else float("nan")


COPY_LIMIT = 0.15    # a checkpoint whose rare-tier copy share is above this counts as copying (5% = none)


def budget_hours(key):
    """'30min' -> 0.5 (astra.bench_run.budget_key); results written before 2026-10-22 use hours ('0.5h')."""
    return float(key[:-3]) / 60 if key.endswith("min") else float(key[:-1])


def budget_label(key):
    h = budget_hours(key)
    return f"{60 * h:g} min" if h < 1 else f"{h:g} h"


def bootstrap_bal_acc(y, star, preds, n_classes, n_boot=1000, seed=0):
    """Balanced accuracy (mean over the runs) on n_boot resamples of the test STARS, drawn with replacement;
    all curves of a star stay together. Every entry of preds (runs x n) sees the same resamples, so the
    difference between two entries is a paired comparison."""
    rng = np.random.default_rng(seed)
    stars, inv = np.unique(star, return_inverse=True)
    W = rng.multinomial(len(stars), np.full(len(stars), 1 / len(stars)), size=n_boot).astype(np.float32)[:, inv.reshape(-1)]
    cls = [np.flatnonzero(y == c) for c in range(n_classes)]
    n_true = np.stack([W[:, i].sum(1) for i in cls], 1)                          # (n_boot, N)
    has = n_true > 0
    out = {}
    for name, p in preds.items():
        ok = (np.atleast_2d(p) == y[None]).astype(np.float32)                    # (runs, n)
        rec = np.stack([W[:, i] @ ok[:, i].T for i in cls], 2) / np.maximum(n_true[:, None, :], 1e-9)
        out[name] = ((rec * has[:, None, :]).sum(2) / np.maximum(has.sum(1), 1)[:, None]).mean(1)
    return out


def ci(d):
    return f"{d.mean():+.2f} [{np.percentile(d, 2.5):+.2f}, {np.percentile(d, 97.5):+.2f}]"


def best_aug(parts):
    """The real + synthetic mix with the best VALIDATION balanced accuracy."""
    augs = {k: v for k, v in parts.items() if k.startswith("aug_p")}
    return max(augs, key=lambda k: augs[k]["val_bal_acc"]) if augs else None


def paired(res, a, b, n_boot=1000):
    """Bootstrap samples (in %) of entries a and b, each given as (job, part name). None if they cannot be paired."""
    (ja, pa), (jb, pb) = a, b
    if ja not in res or jb not in res or res[ja]["data_hash"] != res[jb]["data_hash"]:
        return None
    if not (res[ja]["_pred"].exists() and res[jb]["_pred"].exists()):
        return None
    za, zb = np.load(res[ja]["_pred"]), np.load(res[jb]["_pred"])
    assert np.array_equal(za["y_test"], zb["y_test"]), "the two jobs do not share a test set"
    s = bootstrap_bal_acc(za["y_test"], za["star_test"], {"a": za[pa], "b": zb[pb]}, len(res[ja]["classes"]), n_boot)
    return 100 * s["a"], 100 * s["b"]


def job_table(res, n_boot=1000):
    """One row per generative job: scores of the selected checkpoint and its difference to real data only."""
    rows = []
    for job, r in sorted(res.items()):
        if r["family"] == "real":
            continue
        P, S = r["parts"], r["parts"]["samples"]
        aug, base = best_aug(P), f"real_s{r['seed']}"
        row = {"job": job, "ckpt": r["selected_ckpt"], "selected at (h)": round(r.get("selected_solo_h", np.nan), 2),
               "trained (h)": round(r.get("solo_h", np.nan), 2), "stopped by": r.get("stop", ""),
               "TSTR bal-acc": f"{100 * P['tstr']['bal_acc']:.2f} ± {100 * P['tstr']['bal_acc_std']:.2f}",
               "copies, rare tier %": round(100 * rare_mean(r, S.get("copy_share", [np.nan] * len(r["rare"]))), 1),
               "mem ratio, rare tier": round(rare_mean(r, S["mem_ratio"]), 2),
               "TSTR macro-F1": round(100 * P["tstr"]["macro_f1"], 2)}
        if aug:
            row.update({"real+syn bal-acc": f"{100 * P[aug]['bal_acc']:.2f} ± {100 * P[aug]['bal_acc_std']:.2f}",
                        "p_syn": int(aug[5:]) / 100, "real+syn rare recall": round(100 * P[aug].get("rare_recall", np.nan), 2)})
        if base in res:
            row["real only (A)"] = round(100 * res[base]["parts"]["A_oversampling"]["bal_acc"], 2)
            for part, col in (("tstr", "TSTR - A [95% CI]"), (aug, "real+syn - A [95% CI]")):
                s = paired(res, (job, part), (base, "A_oversampling"), n_boot) if part else None
                row[col] = ci(s[0] - s[1]) if s else ""
        row.update({"FID": round(S["fid"], 2), "density": round(float(np.nanmean(S["density"])), 3),
                    "recall": round(float(np.nanmean(S["recall"])), 3), "coverage": round(float(np.nanmean(S["coverage"])), 3),
                    "mem ratio (all)": round(float(np.nanmean(S["mem_ratio"])), 2), "amp err": round(S["amp_err"], 3),
                    "s per 10k curves": round(S["sampling_s_per_10k"], 2)})
        rows.append(row)
    return pd.DataFrame(rows)


def real_table(res):
    """The reference rows on real data (A = oversampling, B-D = soft warping), one line per split."""
    rows = []
    for job, r in sorted(res.items()):
        if r["family"] != "real":
            continue
        for name, p in r["parts"].items():
            rows.append({"job": job, "row": name, "bal-acc": f"{100 * p['bal_acc']:.2f} ± {100 * p['bal_acc_std']:.2f}",
                         "macro-F1": round(100 * p["macro_f1"], 2), "accuracy": round(100 * p["acc"], 2),
                         "rare recall": round(100 * p.get("rare_recall", np.nan), 2),
                         "common recall": round(100 * p.get("common_recall", np.nan), 2),
                         "epochs (best)": round(p["best_epoch"], 1), "from": Path(r["_from"]).parent.name})
    return pd.DataFrame(rows)


def family_table(res):
    """Mean ± standard deviation over the splits (seeds) of the main setting of every family present."""
    def agg(vals):
        v = np.array([x for x in vals if x is not None and np.isfinite(x)], float)
        return f"{v.mean():.2f} ± {v.std():.2f}" if len(v) else ""

    rows = []
    real = [r for r in res.values() if r["family"] == "real"]
    for name in ("A_oversampling", "B_soft_time_warping", "C_soft_mag_warping", "D_soft_mixed_warping"):
        got = [r["parts"][name] for r in real if name in r["parts"]]
        if got:
            rows.append({"method": f"real only, {name[2:].replace('_', ' ')}", "splits": len(got),
                         "bal-acc": agg(100 * p["bal_acc"] for p in got), "macro-F1": agg(100 * p["macro_f1"] for p in got),
                         "rare recall": agg(100 * p.get("rare_recall", np.nan) for p in got),
                         "common recall": agg(100 * p.get("common_recall", np.nan) for p in got)})
    for fam in MAIN:
        rs = [r for _, r in main_jobs(res, fam)]
        got = [r["parts"] for r in rs]
        if not got:
            continue
        fidelity = {"copies, rare tier %": agg(100 * rare_mean(r, r["parts"]["samples"].get("copy_share", [np.nan] * len(r["rare"])))
                                               for r in rs),
                    "mem ratio, rare tier": agg(rare_mean(r, r["parts"]["samples"]["mem_ratio"]) for r in rs),
                    "FID": agg(P["samples"]["fid"] for P in got),
                    "recall (diversity)": agg(float(np.nanmean(P["samples"]["recall"])) for P in got),
                    "density (fidelity)": agg(float(np.nanmean(P["samples"]["density"])) for P in got),
                    "trained (h, one T4)": agg(r.get("solo_h", np.nan) for r in rs)}
        for label, pick, extra in ((f"{FAMILIES[fam]}, synthetic only (TSTR)", lambda P: P["tstr"], fidelity),
                                   (f"{FAMILIES[fam]}, real + synthetic", lambda P: P[best_aug(P)] if best_aug(P) else None, {})):
            ps = [pick(P) for P in got if pick(P)]
            if ps:
                rows.append({"method": label, "splits": len(ps), "bal-acc": agg(100 * p["bal_acc"] for p in ps),
                             "macro-F1": agg(100 * p["macro_f1"] for p in ps),
                             "rare recall": agg(100 * p.get("rare_recall", np.nan) for p in ps),
                             "common recall": agg(100 * p.get("common_recall", np.nan) for p in ps), **extra})
    return pd.DataFrame(rows).fillna("")


def class_table(res):
    """Per-class test recall (%), mean over the splits; classes from most to least common."""
    cols = {}
    real = [r["parts"]["A_oversampling"]["recall"] for r in res.values()
            if r["family"] == "real" and "A_oversampling" in r["parts"]]
    if real:
        cols["real only (A)"] = 100 * np.mean(real, 0)
    for fam in MAIN:
        got = [r["parts"] for _, r in main_jobs(res, fam)]
        if got:
            cols[f"{FAMILIES[fam]} TSTR"] = 100 * np.mean([P["tstr"]["recall"] for P in got], 0)
            if best_aug(got[0]):
                cols[f"{FAMILIES[fam]} real+syn"] = 100 * np.mean([P[best_aug(P)]["recall"] for P in got], 0)
    if not cols:
        return pd.DataFrame()
    r0 = next(iter(res.values()))
    t = pd.DataFrame(cols, index=r0["classes"]).round(1)
    t.insert(0, "train curves", r0["train_counts"])
    return t


def family_pairs(res, n_boot=1000):
    """Paired comparison of every two families on the splits both have (main settings only)."""
    rows = []
    fams = [f for f in MAIN if main_jobs(res, f)]
    for i, fa in enumerate(fams):
        for fb in fams[i + 1:]:
            ja_by, jb_by = ({s: r["job"] for s, r in main_jobs(res, f)} for f in (fa, fb))
            for seed in sorted(set(ja_by) & set(jb_by)):
                ja, jb = ja_by[seed], jb_by[seed]
                row = {"A": FAMILIES[fa], "B": FAMILIES[fb], "split": seed}
                for label, pa, pb in (("TSTR: A - B [95% CI]", "tstr", "tstr"),
                                      ("real+syn: A - B [95% CI]", best_aug(res[ja]["parts"]), best_aug(res[jb]["parts"]))):
                    s = paired(res, (ja, pa), (jb, pb), n_boot) if pa and pb else None
                    row[label] = ci(s[0] - s[1]) if s else "(different data or no predictions)"
                rows.append(row)
    return pd.DataFrame(rows)


def novel_table(res):
    """Does a model that copies still generate useful curves for conditions it has not seen? Per final job: test
    TSTR (n_sel curves, 3 classifier runs) at the selected and at the last checkpoint, for the training stars'
    own conditions and for new ones (phases of one star, amplitude of another, same class)."""
    rows = []
    for job, r in sorted(res.items()):
        P = r.get("parts", {})
        if "novel_selected" not in P:
            continue
        cells = {"selected": ("sweep_default", "novel_selected"), "last": ("default_last", "novel_last")}
        for which, (own, new) in cells.items():
            if own not in P and new not in P:
                continue
            k = r.get(f"{which}_ckpt")
            sel = next((a for a in r.get("metrics", []) if a["ckpt"] == k), {})
            row = {"job": job, "checkpoint": f"{which} ({k}, {60 * r.get(f'{which}_solo_h', np.nan):.0f} min)",
                   "copies, rare tier % (validation)": round(100 * rare_mean(r, sel.get("copy_share", [np.nan] * len(r["rare"]))), 1)}
            for label, name in (("own conditions", own), ("new conditions", new)):
                p = P.get(name)
                row[f"{label}: test bal-acc"] = round(100 * p["bal_acc"], 2) if p else np.nan
                row[f"{label}: rare recall"] = round(100 * p.get("rare_recall", np.nan), 2) if p else np.nan
            rows.append(row)
    return pd.DataFrame(rows)


def sweep_table(res):
    rows = []
    for job, r in sorted(res.items()):
        for name, p in r["parts"].items():
            if name.startswith("sweep_"):
                rows.append({"job": job, "sampling": "default" if not p["opts"] else ", ".join(f"{k} = {v}" for k, v in p["opts"].items()),
                             "val bal-acc": round(100 * p["val_bal_acc"], 2), "test bal-acc": round(100 * p["bal_acc"], 2),
                             "FID": round(p["fidelity"]["fid"], 2),
                             "density": round(float(np.nanmean(p["fidelity"]["density"])), 3),
                             "recall": round(float(np.nanmean(p["fidelity"]["recall"])), 3),
                             "s per 10k curves": round(p["sampling_s_per_10k"], 2)})
    return pd.DataFrame(rows)


# ---------------------------------------------------------------- figures
def plot_training(res, path=None):
    """Per checkpoint: validation TSTR balanced accuracy, FID, recall and density, memorisation ratio, and the
    model's own loss on training and validation stars (over/under-fitting)."""
    jobs = [(j, r) for j, r in sorted(res.items()) if r.get("metrics")]
    if not jobs:
        return
    fig, ax = plt.subplots(1, 5, figsize=(24, 3.8))
    for j, r in jobs:
        m = r["metrics"]
        x = [a.get("solo_h", a["ckpt"]) for a in m]
        ax[0].plot(x, [100 * a["val_bal_acc"] for a in m], "o-", ms=3, label=j)
        sel = r.get("selected_ckpt") or r.get("best_ckpt")
        for a in m:
            if a["ckpt"] == sel:
                ax[0].plot([a.get("solo_h", a["ckpt"])], [100 * a["val_bal_acc"]], "r*", ms=12)
        ax[1].plot(x, [a["fid"] for a in m], "o-", ms=3, label=j)
        line, = ax[2].plot(x, [np.nanmean(a["recall"]) for a in m], "o-", ms=3, label=f"{j} recall")
        ax[2].plot(x, [np.nanmean(a["density"]) for a in m], "s--", ms=3, c=line.get_color(), label=f"{j} density")
        line, = ax[3].plot(x, [100 * rare_mean(r, a.get("copy_share", [np.nan] * len(r["rare"]))) for a in m], "o-", ms=3,
                           label=f"{j} copies (rare tier)")
        line, = ax[4].plot(x, [a["loss_train"] for a in m], "o-", ms=3, label=f"{j} train")
        ax[4].plot(x, [a["loss_val"] for a in m], "s--", ms=3, c=line.get_color(), label=f"{j} val")
    ax[3].axhline(5, c="k", lw=0.6)
    for a, t in zip(ax, ("validation TSTR balanced accuracy % (star = selected)", "FID (lower = closer to real)",
                         "macro recall (diversity) and density (fidelity)",
                         "copies of the source curve, rare tier % (5 = none)",
                         "the model's own loss: training vs validation stars")):
        a.set_title(t, fontsize=9); a.set_xlabel("hours of training on one T4"); a.set_xscale("log")
        a.legend(fontsize=6)
    fig.tight_layout()
    if path:
        fig.savefig(path, dpi=110)
    plt.show()


def plot_samples(run_dir, Z, title, path=None, per_class=2):
    """Real curves and generated curves with the same condition (phases, amplitude, class), like Fig. 9."""
    s = Path(run_dir) / "samples.npz"
    if not s.exists():
        return
    S = np.load(s)
    classes = [str(c) for c in Z["classes"]]
    fig, ax = plt.subplots(per_class, len(classes), figsize=(3.4 * len(classes), 2.4 * per_class), sharex=True, squeeze=False)
    for c, name in enumerate(classes):
        for r, i in enumerate(np.flatnonzero(Z["y"][S["idx"]] == c)[:per_class]):
            k = S["idx"][i]
            ax[r, c].plot(Z["phase"][k], Z["mag"][k], ".", ms=3, label="real")
            ax[r, c].plot(Z["phase"][k], S["x"][i].astype(float), ".", ms=3, label="generated")
            ax[r, c].invert_yaxis(); ax[r, c].set_title(name, fontsize=9)
    ax[0, 0].legend(fontsize=7); fig.suptitle(title); fig.tight_layout()
    if path:
        fig.savefig(path, dpi=110)
    plt.show()


def plot_classifiers(res, path=None):
    """Training and validation loss of the final classifiers (over/under-fitting of the harness itself)."""
    panels = [(f"{j}: {name}", p["curves"]) for j, r in sorted(res.items()) for name, p in r["parts"].items()
              if name in ("A_oversampling", "tstr") or name == best_aug(r["parts"])]
    if not panels:
        return
    fig, ax = plt.subplots(1, len(panels), figsize=(3.6 * len(panels), 3), squeeze=False)
    for a, (title, c) in zip(ax[0], panels):
        a.plot(c["epoch"], c["train_loss"], label="train"); a.plot(c["epoch"], c["val_loss"], label="val (class-balanced)")
        a.set_title(title, fontsize=8); a.set_xlabel("epoch"); a.legend(fontsize=6)
    fig.tight_layout()
    if path:
        fig.savefig(path, dpi=100)
    plt.show()


def plot_sweeps(res, path=None):
    """Quality against sampling cost for the other sampling settings of the selected checkpoints."""
    t = sweep_table(res)
    if not len(t):
        return
    fig, ax = plt.subplots(figsize=(7.5, 4.2))
    for job, g in t.groupby("job"):
        ax.scatter(g["s per 10k curves"], g["test bal-acc"], label=job, s=25)
        for _, row in g.iterrows():
            ax.annotate(row["sampling"], (row["s per 10k curves"], row["test bal-acc"]), fontsize=6, xytext=(3, 3),
                        textcoords="offset points")
    ax.set_xscale("log"); ax.set_xlabel("seconds to generate 10,000 curves (log scale)")
    ax.set_ylabel("TSTR test balanced accuracy % (3 runs)"); ax.legend(fontsize=7)
    ax.set_title("sampling settings of the selected checkpoint", fontsize=9); fig.tight_layout()
    if path:
        fig.savefig(path, dpi=110)
    plt.show()


# ---------------------------------------------------------------- figures and scores for the four-way comparison
def main_jobs(res, fam):
    """The finished final-stage jobs of a family (its searched setting), one per split: [(seed, result), ...]."""
    return sorted(((r["seed"], r) for r in res.values() if r["family"] == fam and r.get("stage", "final") == "final"),
                  key=lambda t: t[0])


# ---------------------------------------------------------------- the search (validation only) and the budgets
def search_table(res):
    """One row per searched configuration (split 0, validation stars only), best first within a family.
    The rule for the winner: the highest mean of its three best checkpoints' validation scores (less noisy than
    the single best), read against the copy share: "top-3, no copies" counts only checkpoints whose rare-tier
    copy share is at most COPY_LIMIT (n = how many; group decision, 2026-10-22, after the GAN search, where copying
    checkpoints made up the top-3 of the rule's winner); within about 1 point the cheaper run wins."""
    rows = []
    for job, r in res.items():
        m = r.get("metrics") or []
        b = next((a for a in m if a["ckpt"] == r.get("best_ckpt")), None)
        if b is None:
            continue
        top3 = sorted((a["val_bal_acc"] for a in m), reverse=True)[:3]
        clean = sorted((a["val_bal_acc"] for a in m if rare_mean(r, a["copy_share"]) <= COPY_LIMIT), reverse=True)[:3]
        rows.append({"family": r["family"], "config": r.get("tag") or job,
                     "top-3 mean (rule)": round(100 * float(np.mean(top3)), 2),
                     "top-3, no copies (n)": f"{100 * np.mean(clean):.2f} ({len(clean)})" if clean else "-",
                     "best val bal-acc":
                     round(100 * b["val_bal_acc"], 2), "± (3 runs)": round(100 * float(np.std(b["val_bal_acc_runs"])), 2),
                     "at (h, one T4)": round(b.get("solo_h", np.nan), 2), "trained (h)": round(r.get("solo_h", np.nan), 2),
                     "stopped by": r.get("stop", ""), "copies, rare tier %": round(100 * rare_mean(r, b["copy_share"]), 1),
                     "mem ratio, rare tier": round(rare_mean(r, b["mem_ratio"]), 2), "FID": round(b["fid"], 2),
                     "recall": round(float(np.nanmean(b["recall"])), 3), "checkpoints": len(m)})
    t = pd.DataFrame(rows)
    return t.sort_values(["family", "top-3 mean (rule)"], ascending=[True, False]) if len(t) else t


def plot_search(res, path=None):
    """Validation TSTR balanced accuracy and rare-tier copies of every searched configuration against hours."""
    fams = sorted({r["family"] for r in res.values()})
    if not fams:
        return
    fig, ax = plt.subplots(2, len(fams), figsize=(6 * len(fams), 7), squeeze=False)
    for k, fam in enumerate(fams):
        for job, r in sorted(res.items()):
            if r["family"] != fam or not r.get("metrics"):
                continue
            m = r["metrics"]
            x = [a["solo_h"] for a in m]
            ax[0, k].plot(x, [100 * a["val_bal_acc"] for a in m], "o-", ms=3, label=r.get("tag") or job)
            ax[1, k].plot(x, [100 * rare_mean(r, a["copy_share"]) for a in m], "o-", ms=3, label=r.get("tag") or job)
        ax[0, k].set_title(f"{FAMILIES[fam]}: validation TSTR balanced accuracy %", fontsize=9)
        ax[1, k].set_title(f"{FAMILIES[fam]}: copies of the source curve, rare tier % (5 = none)", fontsize=9)
        ax[1, k].axhline(5, c="k", lw=0.6)
        for a in ax[:, k]:
            a.set_xscale("log"); a.set_xlabel("hours of training on one T4"); a.legend(fontsize=7)
    fig.tight_layout()
    if path:
        fig.savefig(path, dpi=110)
    plt.show()


def budget_table(res):
    """The compute-matched view: per family and budget, the test balanced accuracy (TSTR) of the best checkpoint
    reached within that much training on one T4; mean ± std over the splits. 'full' = no limit."""
    rows = []
    for fam in MAIN:
        jobs = main_jobs(res, fam)
        if not jobs:
            continue
        keys = sorted({k for _, r in jobs for k in r.get("budgets", {})}, key=budget_hours)
        for key in keys + ["full"]:
            vals, at, copies = [], [], []
            for _, r in jobs:
                if key == "full":         # the selected checkpoint; copies measured as for the budgets (validation side)
                    sel = next((a for a in r.get("metrics", []) if a["ckpt"] == r.get("selected_ckpt")), {})
                    vals.append(100 * r["parts"]["tstr"]["bal_acc"]); at.append(r.get("selected_solo_h", np.nan))
                    copies.append(100 * rare_mean(r, sel.get("copy_share", [np.nan] * len(r["rare"]))))
                    continue
                b = r.get("budgets", {}).get(key)
                if b and b["test"] in r["parts"]:
                    vals.append(100 * r["parts"][b["test"]]["bal_acc"]); at.append(b["solo_h"])
                    copies.append(100 * rare_mean(r, b["copy_share"]))
            if vals:
                rows.append({"family": FAMILIES[fam], "budget (one T4)": key if key == "full" else budget_label(key),
                             "splits": len(vals),
                             "test bal-acc (TSTR)": f"{np.mean(vals):.2f} ± {np.std(vals):.2f}",
                             "checkpoint at (h)": round(float(np.mean(at)), 2),
                             "copies, rare tier % (validation)": round(float(np.nanmean(copies)), 1)})
    return pd.DataFrame(rows)


def plot_budget(res, path=None):
    """Test balanced accuracy (TSTR) of each family if training had stopped after each budget (5 min ... 10 h),
    and at its own best point; the dashed line is real data only (row A)."""
    fig, ax = plt.subplots(figsize=(7, 4.2))
    drawn = False
    for i, fam in enumerate(MAIN):
        jobs = main_jobs(res, fam)
        if not jobs:
            continue
        pts = {}
        for _, r in jobs:
            for key, b in r.get("budgets", {}).items():
                if b["test"] in r["parts"]:
                    pts.setdefault(budget_hours(key), []).append(100 * r["parts"][b["test"]]["bal_acc"])
        if not pts:
            continue
        xs = sorted(pts)
        ax.errorbar(xs, [np.mean(pts[x]) for x in xs], [np.std(pts[x]) for x in xs], fmt="o-", c=f"C{i}",
                    capsize=3, label=FAMILIES[fam])
        full = [(r.get("selected_solo_h", np.nan), 100 * r["parts"]["tstr"]["bal_acc"]) for _, r in jobs]
        ax.plot([f[0] for f in full], [f[1] for f in full], "*", c=f"C{i}", ms=11)
        drawn = True
    real = [100 * r["parts"]["A_oversampling"]["bal_acc"] for r in res.values()
            if r["family"] == "real" and "A_oversampling" in r["parts"]]
    if real:
        ax.axhline(np.mean(real), c="k", ls="--", lw=1, label="real data only (A)")
    if not drawn:
        plt.close(fig)
        return
    ax.set_xscale("log"); ax.set_xlabel("training budget, hours on one T4"); ax.set_ylabel("test balanced accuracy %")
    ax.set_title("if every model had stopped after this much training (stars = own best point)", fontsize=9)
    ax.legend(fontsize=8); fig.tight_layout()
    if path:
        fig.savefig(path, dpi=130)
    plt.show()


def samples_of(r):
    """The synthetic curves of a job's selected checkpoint (saved next to its result file), or None."""
    p = r["_pred"].with_name(f"{r['job']}_samples.npz")
    return np.load(p) if p.exists() else None


def plot_headline(res, path=None):
    """Test balanced accuracy of every method: one point per split, a bar for the mean. The dashed line is
    real data only (row A); a method has to be above it to count as useful augmentation."""
    entries = []
    real = sorted((r["seed"], 100 * r["parts"]["A_oversampling"]["bal_acc"]) for r in res.values()
                  if r["family"] == "real" and "A_oversampling" in r["parts"])
    if real:
        entries.append(("real only (A)", [v for _, v in real], "0.5"))
    for i, fam in enumerate(f for f in MAIN if main_jobs(res, f)):
        jobs = main_jobs(res, fam)
        entries.append((f"{FAMILIES[fam]}\nsynthetic only", [100 * r["parts"]["tstr"]["bal_acc"] for _, r in jobs], f"C{i}"))
        aug = [100 * r["parts"][best_aug(r["parts"])]["bal_acc"] for _, r in jobs if best_aug(r["parts"])]
        if aug:
            entries.append((f"{FAMILIES[fam]}\nreal + synthetic", aug, f"C{i}"))
    if not entries:
        return
    fig, ax = plt.subplots(figsize=(1.5 * len(entries) + 2, 4.2))
    for k, (label, vals, color) in enumerate(entries):
        ax.bar(k, np.mean(vals), color=color, alpha=0.35 if "synthetic only" in label else 0.8, width=0.7)
        ax.plot([k] * len(vals), vals, "k.", ms=7)
        ax.text(k, max(vals) + 0.4, f"{np.mean(vals):.1f}", ha="center", fontsize=8)
    if real:
        ax.axhline(np.mean([v for _, v in real]), color="k", ls="--", lw=1)
    ax.set_xticks(range(len(entries))); ax.set_xticklabels([e[0] for e in entries], fontsize=8)
    lo = min(min(e[1]) for e in entries)
    ax.set_ylim(max(0, lo - 5), None); ax.set_ylabel("test balanced accuracy %")
    ax.set_title("balanced accuracy per method (points = splits, bar = mean)", fontsize=9); fig.tight_layout()
    if path:
        fig.savefig(path, dpi=130)
    plt.show()


def plot_class_bars(res, path=None):
    """Per-class test recall of real data only against real + synthetic of every family (mean over splits)."""
    t = class_table(res)
    cols = [c for c in t.columns if c == "real only (A)" or c.endswith("real+syn")] if len(t) else []
    if len(cols) < 2:
        return
    rare = np.array(next(iter(res.values()))["rare"])
    fig, ax = plt.subplots(figsize=(1.6 * len(t) + 3, 4))
    w = 0.8 / len(cols)
    for k, c in enumerate(cols):
        ax.bar(np.arange(len(t)) + (k - (len(cols) - 1) / 2) * w, t[c], w, label=c, color="0.5" if k == 0 else f"C{k - 1}")
    ax.set_xticks(range(len(t)))
    ax.set_xticklabels([f"{n}\n({c:,} train){' *' if r else ''}" for n, c, r in zip(t.index, t["train curves"], rare)], fontsize=8)
    ax.set_ylabel("test recall %"); ax.legend(fontsize=7, ncol=2)
    ax.set_title("recall per class, classes from most to least common (* = rare tier)", fontsize=9); fig.tight_layout()
    if path:
        fig.savefig(path, dpi=130)
    plt.show()


def plot_family_samples(res, Z, path=None, seed=0):
    """One real curve per class and, next to it, the curve every family generated for the SAME condition
    (same star's phases, amplitude and class). Possible because all notebooks use the same conditions."""
    S = {fam: samples_of(r) for fam in MAIN for s, r in main_jobs(res, fam) if s == seed}
    S = {fam: s for fam, s in S.items() if s is not None}
    if not S:
        return
    idx = next(iter(S.values()))["idx"]
    S = {fam: s for fam, s in S.items() if np.array_equal(s["idx"], idx)}        # same conditions only
    classes = [str(c) for c in Z["classes"]]
    fig, ax = plt.subplots(len(classes), 1 + len(S), figsize=(2.9 * (1 + len(S)), 1.9 * len(classes)), sharex=True, squeeze=False)
    for c, name in enumerate(classes):
        rows = np.flatnonzero(Z["y"][idx] == c)
        if not len(rows):
            continue
        i, k = rows[0], idx[rows[0]]
        curves = [("real", Z["mag"][k], "0.2")] + [(FAMILIES[f], s["x"][i].astype(float), f"C{n}") for n, (f, s) in enumerate(S.items())]
        lim = 1.6 * float(np.abs(Z["mag"][k]).max()) + 1e-3
        for a, (label, y, color) in zip(ax[c], curves):
            a.plot(Z["phase"][k], y, ".", ms=3, color=color)
            a.set_ylim(lim, -lim)
            if c == 0:
                a.set_title(label, fontsize=9)
        ax[c, 0].set_ylabel(name, fontsize=8)
    fig.supxlabel("phase"); fig.suptitle(f"split {seed}: a real curve and the generated curve for its condition, per family", fontsize=10)
    fig.tight_layout()
    if path:
        fig.savefig(path, dpi=130)
    plt.show()


def fidelity_one_space(res, npz_path, work_dir, device="cpu", log=print):
    """FID, density, recall, coverage and precision of every job's saved samples, all measured with ONE feature
    CNN per split (the one in work_dir, trained there if missing). Each notebook trains its own feature CNN,
    so the fidelity columns of different notebooks are only roughly comparable; these are comparable exactly."""
    import torch
    from . import bench_data, bench_run
    from .bench_eval import Fidelity
    rows = []
    for seed in sorted({r["seed"] for r in res.values() if r["family"] != "real"}):
        data, fid = bench_data.load(npz_path, seed, device), None
        for job, r in sorted(res.items()):
            S = samples_of(r) if r["family"] != "real" and r["seed"] == seed else None
            if S is None or r["data_hash"] != data["hash"]:
                continue
            if fid is None:
                fid = Fidelity(data, bench_run.feature_cnn(Path(work_dir), data, device, log=log))
            f = fid(torch.from_numpy(S["x"].astype(np.float32)).to(device), torch.from_numpy(S["idx"]).to(device))
            rows.append({"job": job, "FID": round(f["fid"], 2), **{m: round(float(np.nanmean(f[m])), 3)
                         for m in ("density", "recall", "coverage", "precision")}, "feature CNN": data["feat_hash"]})
    return pd.DataFrame(rows)


## 1 · Data and split

The curves are those of notebook 01 (period folding, outlier filtering, 100 or 40 random points, median-centred), so any difference to its numbers comes from the protocol, not from the data. If a ready `catalina_L100.npz` or `ztf_L40.npz` is attached under *Add Input* it is used; otherwise Catalina is downloaded and pre-processed here (about 3 minutes). **Attach the same file to all four notebooks**: the results carry a hash of the data and split, and the comparison refuses to pair results with different hashes.

The table shows the curves per class and side for split 0, and which classes form the rare tier. The figure shows, per class, a real curve in magnitudes (top) and the amplitude-normalised curve the models are trained on (bottom).

In [ ]:
# ---- Get the pre-processed light curves, then split them over stars ----
from astra import catalina, ztf, bench_data
from astra.common import find_input

NPZ = WORK / "data" / f"{DATASET}_L{L}.npz"
if not NPZ.exists():
    ready = find_input(NPZ.name)
    if ready:
        shutil.copy2(ready, NPZ)
    elif DATASET == "catalina":                          # download + pre-process, about 3 minutes
        out, _ = catalina.preprocess(WORK / "data", L=L, class_range_k=CLASS_RANGE_K, seed=DATA_SEED)
        np.savez_compressed(NPZ, **out)
        (WORK / "data" / "AllVar.phot.gz").unlink(missing_ok=True)
    else:
        raise AssertionError("ZTF: use Add Input to attach the dataset with ztf_L40.npz")
Z = np.load(NPZ)
CLASSES = [str(c) for c in Z["classes"]]
table = pd.DataFrame({"class": CLASSES, "curves": np.bincount(Z["y"], minlength=len(CLASSES))})
for s in SEEDS:
    t = bench_data.split_table(NPZ, s)
    for side in ("train", "val", "test"):
        table[f"{side} (split {s})"] = t[side]
table["rare tier"] = bench_data.split_table(NPZ, SEEDS[0])["train"] < bench_data.RARE_BELOW
table.to_csv(TABLES / "split_counts.csv", index=False)
print(table.to_string(index=False))
d0 = bench_data.load(NPZ, SEEDS[0], "cpu")
stars = [set(d0["star"][d0[f"{s}_idx_np"]]) for s in ("train", "val", "test")]
assert not (stars[0] & stars[1]) and not (stars[0] & stars[2]) and not (stars[1] & stars[2]), "a star is on two sides"
print(f"\n{len(Z['y']):,} curves of {L} points, {len(np.unique(d0['star'])):,} stars | split {SEEDS[0]}: "
      f"train {len(d0['train_idx']):,}, val {len(d0['val_idx']):,}, test {len(d0['test_idx']):,} | no star on two sides "
      f"| data hash {d0['hash']}")

# one curve per class: magnitudes (top) and the model's target u = magnitude / amplitude (bottom)
fig, ax = plt.subplots(2, len(CLASSES), figsize=(3 * len(CLASSES), 5), sharex=True)
rng = np.random.default_rng(0)
for c, name in enumerate(CLASSES):
    i = rng.choice(np.flatnonzero(Z["y"] == c))
    ax[0, c].plot(Z["phase"][i], Z["mag"][i], ".", ms=3); ax[0, c].invert_yaxis()
    ax[0, c].set_title(f"{name} (amplitude {Z['amp'][i]:.2f} mag)", fontsize=9)
    ax[1, c].plot(Z["phase"][i], Z["mag"][i] / Z["amp"][i], ".", ms=3, c="C1"); ax[1, c].set_ylim(2.1, -2.1)
ax[0, 0].set_ylabel("centred magnitude"); ax[1, 0].set_ylabel("magnitude / amplitude")
fig.supxlabel("phase"); fig.tight_layout(); fig.savefig(FIGS / "data_examples.png", dpi=110); plt.show()
del d0

## 2 · Self-tests (seconds)

Small checks with a known answer, run before any GPU time is spent:
- balanced accuracy of a constant guess is 1 / (number of classes), and the paired bootstrap agrees;
- the model of this notebook: parameter counts, one training step, sampling repeats exactly for a fixed seed, and a check of the sampler against a case with a known answer.

In [ ]:
# ---- Self-tests: harness ----
import importlib
from astra import bench_eval, bench_report, bench_run
from astra.sampling import ClassResampler
fam = importlib.import_module(f"astra.bench_{FAMILY}")

y = np.array([0] * 60 + [1] * 40)
pred = np.stack([y.copy(), np.zeros(100, int)])                  # run 0 is perfect, run 1 always says class 0
s = bench_eval.summarise(y, pred, 2, rare=np.array([False, True]))
assert np.allclose(s["bal_acc_runs"], [1.0, 0.5]) and np.allclose(s["acc_runs"], [1.0, 0.6])
b = bench_report.bootstrap_bal_acc(y, np.arange(100), {"good": pred[:1], "bad": pred[1:]}, 2, n_boot=200)
assert np.allclose(b["good"], 1.0) and np.allclose(b["bad"], 0.5)
assert all(j.startswith("real_s") or bench_run.job_tag(j, FAMILY) in fam.GRID for j in JOBS_ALL), "unknown job"

# the resume path on THIS device: a saved random-number state, loaded the way a resumed job loads it, must
# give the same random numbers again (on a GPU the loaded state has to be moved back to the CPU first)
from astra.common import restore_rng, torch_save
torch_save({"rng_cpu": torch.get_rng_state(),
            "rng_cuda": torch.cuda.get_rng_state_all() if DEV == "cuda" else None}, WORK / "rng_test.pt")
first = (torch.rand(3), torch.rand(3, device=DEV))
assert restore_rng(torch.load(WORK / "rng_test.pt", map_location=DEV, weights_only=False), DEV)
again = (torch.rand(3), torch.rand(3, device=DEV))
assert all(torch.equal(a, b) for a, b in zip(first, again)), "a resumed job would not continue its random numbers"
(WORK / "rng_test.pt").unlink()

# ---- this notebook's model (CPU, real-size networks, a batch of 16) ----
data_cpu = bench_data.load(NPZ, SEEDS[0], "cpu")
cfg = bench_run.job_config(SEARCH_JOBS[0], fam, "search", tiny=True)   # the same config path as the workers
torch.manual_seed(0)
model = fam.MODEL(cfg, data_cpu, "cpu")
rs = ClassResampler(data_cpu["y_np"][data_cpu["train_idx_np"]], cfg["gamma"], seed=0, n_classes=len(CLASSES))
stats = model.step(lambda: data_cpu["train_idx"][torch.from_numpy(rs.sample(16))], 0.5)
assert torch.isfinite(stats).all()
idx = bench_data.balanced_idx(data_cpu, 16, seed=0)
opts = {"nfe": 4} if "nfe" in cfg else {}
x1, x2 = model.sample(idx, seed=3, **opts), model.sample(idx, seed=3, **opts)
assert x1.shape == (16, L) and torch.equal(x1, x2), "sampling must repeat exactly for a fixed seed"
N_PARAMS = model.n_params()
print(f"harness tests passed | {FAMILY}: parameters {N_PARAMS} | one step: "
      + ", ".join(f"{n} {v:.3f}" for n, v in zip(model.STATS, stats.tolist())))

In [ ]:
# ---- Self-tests: diffusion ----
ratio = N_PARAMS["net"] / bench_data.G_PARAMS[L]
assert abs(ratio - 1) < 0.10, f"the denoising network should match the GAN generator's size (ratio {ratio:.2f})"
ab = model.ab
assert ab[0] > 0.99 and ab[-1] < 1e-3 and (ab[1:] < ab[:-1]).all(), "signal share must fall from ~1 to ~0"
u0 = data_cpu["u"][idx]


class Oracle:                                 # a "network" that always returns the true noise for the curves u0
    n_classes = len(CLASSES)

    def __call__(self, x, tt, ph, camp, y):
        a = ab[(tt * model.T).round().long()][:, None]
        return (x - a.sqrt() * u0) / (1 - a).sqrt()


# known answer: with the true noise, the DDIM sampler must return the clean curves, for any number of steps
net, model.ema = model.ema, Oracle()
for nfe in (2, 8, 32):
    assert torch.allclose(model.sample(idx, seed=1, nfe=nfe), data_cpu["x"][idx], atol=2e-3), nfe
model.ema = net
print(f"diffusion tests passed | denoising network {N_PARAMS['net']:,} parameters "
      f"({100 * ratio:.0f}% of the GAN generator's {bench_data.G_PARAMS[L]:,})")
del data_cpu, model

## 3 · The feature classifier (one per split)

As in the paper (Sec. 4.3), one CNN is trained on the real training curves of a split, and the output of its last conv block is the feature space in which density, recall, coverage and FID are measured. It is trained once per split and then reused by every job on that split.

In [ ]:
# ---- Train (or reuse) the feature CNN of every split in the queues ----
from astra import classifier

fig, ax = plt.subplots(1, 2, figsize=(10, 3.2))
for s in SEEDS:
    data = bench_data.load(NPZ, s, DEV)
    feat = bench_run.feature_cnn(RUNS, data, DEV, max_epochs=3 if SMOKE else 20)
    te = bench_data.subset(data, data["test_idx"])
    p = classifier.predict_proba(feat, classifier.stack_input(te["x"], te["ph"]), amp=False)[0].argmax(1).cpu().numpy()
    sc = bench_eval.summarise(data["y_np"][data["test_idx_np"]], p, len(CLASSES))
    curve = np.array(json.load(open(RUNS / f"feature_cnn_s{s}_curve.json")))
    print(f"split {s}: feature CNN trained {len(curve)} epochs | this single CNN on the test stars: "
          f"balanced accuracy {100 * sc['bal_acc']:.1f}%, accuracy {100 * sc['acc']:.1f}%")
    ax[0].plot(curve[:, 0], label=f"split {s} train (balanced)"); ax[0].plot(curve[:, 1], "--", label=f"split {s} validation")
    ax[1].plot(curve[:, 2], label=f"split {s}")
    del data, feat
ax[0].set_xlabel("epoch"); ax[0].set_ylabel("cross-entropy"); ax[0].legend(fontsize=7)
ax[1].set_xlabel("epoch"); ax[1].set_ylabel("validation accuracy"); ax[1].legend(fontsize=7)
fig.tight_layout(); fig.savefig(FIGS / "feature_cnn_curves.png", dpi=110); plt.show()

In [ ]:
# ---- SMOKE only: real-size speeds -> the family's SPEED table, and whether two jobs per GPU pay off ----
def packing_test(n_proc_per_gpu, gpus, steps):
    """Training steps per second of the default configuration when n processes share each GPU."""
    procs = []
    for g in gpus:
        for _ in range(n_proc_per_gpu):
            env = dict(os.environ, CUDA_VISIBLE_DEVICES=g, PYTHONPATH=str(WORK))
            cmd = [sys.executable, "-m", "astra.bench_run", "--family", FAMILY, "--data", str(NPZ),
                   "--speedtest", str(steps)]
            procs.append(subprocess.Popen(cmd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, cwd=WORK, text=True))
    outs = [p.communicate()[0] for p in procs]
    try:
        return [json.loads(o.strip().splitlines()[-1])["it_per_s"] for o in outs]
    except Exception:
        print("!! packing test failed:\n", "\n".join(o[-1500:] for o in outs))
        return None


if SMOKE:
    data = bench_data.load(NPZ, 0, DEV)
    b = bench_run.benchmark(fam, data, DEV, quick=LOCAL_TEST)
    n_train = len(data["train_idx"])
    del data
    print({k: (round(v, 3) if isinstance(v, float) else v) for k, v in b.items() if k != "speeds"})
    print(f"\n{FAMILY}: training steps per second ALONE on one GPU, per grid configuration "
          f"(for information only: the search measures its own speeds; nothing to paste):")
    print("SPEED = {" + ", ".join(f'"{k}": {v:.1f}' for k, v in b["speeds"].items()) + "}")
    for k, v in b["speeds"].items():
        bs = {**fam.DEFAULTS, **fam.GRID[k]}["batch"]
        print(f"  {k:<14} {v:6.1f} steps/s -> 1 h = {v * 3600:,.0f} steps = {v * 3600 * bs / n_train:,.0f} passes "
              f"over the {n_train:,} training curves")
    steps = 20 if LOCAL_TEST else 300
    if not LOCAL_TEST:
        solo = packing_test(1, ["0"], steps)
        pair = packing_test(2, ["0"], steps)
        quad = packing_test(2, ["0", "1"], steps)
        if solo and pair and quad:
            gain2, gain4 = sum(pair) / solo[0], sum(quad) / (2 * solo[0])
            rec = 2 if min(gain2, gain4) >= 1.25 else 1
            print(f"\npacking test ({steps} steps each): alone {solo[0]:.1f} steps/s | two on one GPU "
                  f"{' + '.join(f'{v:.1f}' for v in pair)} = {gain2:.2f} x | two on each GPU, per GPU {gain4:.2f} x"
                  f"\n-> recommended SLOTS_PER_GPU = {rec} (2 only if sharing gives at least 1.25 x the work per GPU)")
            b["packing"] = {"solo": solo, "pair": pair, "quad": quad, "gain2": gain2, "gain4": gain4, "recommended": rec}
    E = bench_eval.EVAL
    clf_sel = 4 * b[f"clf_epoch_s_{E['sel_runs'] * E['ens']}cnn"] * E["n_sel"] / n_train
    eval_min = ((E["n_sel"] + 2 * E["n_mem"]) / b["curves_per_s"] + clf_sel + 20) / 60
    print(f"\nsampling: {b['curves_per_s']:,.0f} curves per second with the default settings | one checkpoint "
          f"evaluation ≈ {eval_min:.1f} min alone on a GPU | peak GPU memory {b['peak_gpu_gb']:.1f} GB")
    json.dump(b, open(TABLES / "benchmark.json", "w"), indent=1, default=str)

# ---- every real run: the speeds of its configurations (kept in runs_*/speeds.json for later sessions) ----
SPEEDS_FILE = RUNS / "speeds.json"
SPEEDS = json.loads(SPEEDS_FILE.read_text()) if SPEEDS_FILE.exists() else {}
need = sorted({bench_run.job_tag(j, FAMILY) for j in JOBS_ALL if not j.startswith("real_s")}
              - {t for t, v in fam.SPEED.items() if v} - set(SPEEDS))
if not SMOKE and not LOCAL_TEST and need:
    if STAGE == "final":
        raise AssertionError(f"no speed for {need}: attach the search output of this configuration, or put its "
                             "speed into astra/bench_grid.py and rebuild")
    data = bench_data.load(NPZ, 0, DEV)
    for t in need:
        SPEEDS[t], _ = bench_run.speedtest(fam, fam.GRID[t], data, DEV, n_steps=60)
    del data
    SPEEDS_FILE.write_text(json.dumps(SPEEDS, indent=1))
    print("measured now (alone on this GPU, kept for the next sessions):")
if SPEEDS:
    print(f"speeds in {SPEEDS_FILE.name}: SPEED[\"{FAMILY}\"] = " + json.dumps({k: round(v, 2) for k, v in SPEEDS.items()}))

## 4 · Training: the workers

Every GPU runs `SLOTS_PER_GPU` workers (`python -m astra.bench_run`), each working down its own queue:
- a model job trains one configuration and takes a checkpoint after 5, 10, 20, 30, 45, 60, 90, 120, 180 minutes, then every hour (every 2 hours after 10 h; the final stage also after 1, 2, 3, 7.5 and 15 minutes), counted in hours of training on one T4 alone. At each checkpoint it generates 20,000 class-balanced curves and measures fidelity, memorisation (ratio and copy share), the model's own loss on training and validation stars, and the validation TSTR balanced accuracy (3 runs × 5 CNNs);
- it stops when 4 checkpoints in a row bring no new best validation score (after at least 1 h), or at the cap: 6 h in the search, 20 h in the final stage;
- **search stage**: the run ends with its best validation score and keeps its state;
- **final stage**: a fresh run per split (in its own folder) gets the test scores of its best checkpoint: one synthetic curve per training condition, TSTR and real + synthetic (15 runs × 5 CNNs each), the other sampling settings, the TSTR test score of the best checkpoint within 5, 10, 15, 30, 60 minutes and 2, 5, 10 h, and TSTR for conditions no training star has, at the selected and at the last checkpoint;
- a `real_s<k>` job trains the reference classifiers on the real training curves of split k (rows A–D).

In a real run, a short *canary* first runs a tiny version of each slot's first model job, so a broken setup fails early. The cell prints a status line per slot every 10 minutes, returns as soon as all workers are done, and stops a worker that is still running at the time limit (its saved state is kept). Full logs: `runs_*/<job>/log.txt`.

In [ ]:
# ---- Launch the workers (SLOTS_PER_GPU per GPU) and wait ----
RESULTS_RESERVE_H = 0.5          # kept free for the results cells and Kaggle's output saving
deadline = RUN_STARTED + SESSION_HOURS * 3600 - RESULTS_RESERVE_H * 3600


def launch(queues, runs_dir, results_dir, smoke, stage):
    procs = {}
    for slot, q in queues.items():
        env = dict(os.environ, CUDA_VISIBLE_DEVICES=slot[0], PYTHONPATH=str(WORK))     # slot "1b" runs on GPU 1
        f = open(runs_dir / f"worker_{slot}.log", "a")
        cmd = [sys.executable, "-m", "astra.bench_run", "--family", FAMILY, "--stage", stage, "--queue", ",".join(q),
               "--deadline", str(deadline), "--data", str(NPZ), "--dataset", DATASET, "--runs-dir", str(runs_dir),
               "--results-dir", str(results_dir), "--speeds", str(SPEEDS_FILE)] + (["--smoke"] if smoke else [])
        procs[slot] = (subprocess.Popen(cmd, env=env, stdout=f, stderr=subprocess.STDOUT, cwd=WORK), q)
    return procs


def state(job, runs_dir):
    p = runs_dir / job / "status.json"
    return json.loads(p.read_text()) if p.exists() else {}


def last_line(job, runs_dir):
    p = runs_dir / job / "log.txt"
    lines = p.read_text(encoding="utf-8").strip().splitlines() if p.exists() else []
    return lines[-1][:170] if lines else "(not started)"


def finished(job, runs_dir, stage):
    s = state(job, runs_dir)
    return s.get("state") == "failed" or (s.get("state") == "done" and (job.startswith("real_s") or s.get("stage") in ("final", stage)))


def wait(procs, runs_dir, every_s, stop_at, stage):
    """Returns as soon as every worker has exited; prints a status line per slot every `every_s` seconds.
    A worker pauses its job by itself 15 minutes before the deadline, so one that is still alive at `stop_at`
    is stuck: it is stopped here (its last saved state is kept) and the results cells still run."""
    t_print = time.time()
    while any(p.poll() is None for p, _ in procs.values()):
        time.sleep(5)
        if time.time() > stop_at:
            for slot, (p, _) in procs.items():
                if p.poll() is None:
                    p.kill()
                    p.wait()
                    print(f"!! the worker of slot {slot} was still running at the time limit and was stopped "
                          f"(see {runs_dir.name}/worker_{slot}.log); its last saved state is kept", flush=True)
        if all(p.poll() is not None for p, _ in procs.values()) or time.time() - t_print >= every_s:
            t_print = time.time()
            h = (time.time() - RUN_STARTED) / 3600
            for slot, (p, q) in procs.items():
                cur = next((j for j in q if not finished(j, runs_dir, stage)), q[-1])
                status = "running" if p.poll() is None else "exited "
                print(f"[{h:5.2f} h] slot {slot} {status} | {cur}: {last_line(cur, runs_dir)}", flush=True)
    return {slot: p.returncode for slot, (p, _) in procs.items()}


def run_stage(stage, queues, smoke):
    runs = STAGE_RUNS[stage]                 # final runs start fresh in their own folder (since 2026-10-22)
    runs.mkdir(parents=True, exist_ok=True)
    for f in RUNS.glob("feature_cnn_s*"):    # SMOKE runs both stages: the final folder needs the feature CNNs too
        if not (runs / f.name).exists():
            shutil.copy2(f, runs / f.name)
    todo = {s: [j for j in q if not finished(j, runs, stage)] for s, q in queues.items()}
    todo = {s: q for s, q in todo.items() if q}
    if not todo:
        print(f"{stage} stage: every job is already done")
        return {}
    if not smoke:      # canary: a tiny version of each slot's first model job, in a throw-away folder
        canary = WORK / "canary"
        shutil.rmtree(canary, ignore_errors=True)
        canary.mkdir()
        for f in runs.glob("feature_cnn_s*"):
            shutil.copy2(f, canary / f.name)
        first = {s: [next(j for j in q if not j.startswith("real_s"))] for s, q in todo.items()
                 if any(not j.startswith("real_s") for j in q)}
        codes = wait(launch(first, canary, canary / "results", True, "search"), canary, 20, time.time() + 1800, "search")
        states = {j: state(j, canary).get("state") for q in first.values() for j in q}
        assert all(c == 0 for c in codes.values()) and all(s == "done" for s in states.values()), \
            f"canary failed: {codes} {states} (see {canary})"
        shutil.rmtree(canary)
        print("canary passed\n")
    codes = wait(launch(todo, runs, RESULTS, smoke, stage), runs, 30 if smoke else 600, deadline, stage)
    print(f"{stage} stage: workers finished with exit codes", codes)
    return codes


# ---- how many jobs share a GPU: measured once (SMOKE cell or here), kept in runs_*/packing.json ----
PACK_FILE = RUNS / "packing.json"
if SLOTS_PER_GPU != "auto":
    SPG = int(SLOTS_PER_GPU)
elif LOCAL_TEST:
    SPG = 1
elif PACK_FILE.exists():
    SPG = json.loads(PACK_FILE.read_text())["recommended"]
else:
    pk = (json.load(open(TABLES / "benchmark.json")).get("packing") if (TABLES / "benchmark.json").exists() else None)
    if not pk:
        solo, pair = packing_test(1, ["0"], 300), packing_test(2, ["0"], 300)
        gain = sum(pair) / solo[0] if solo and pair else 1.0
        pk = {"solo": solo, "pair": pair, "gain2": gain, "recommended": 2 if gain >= 1.25 else 1}
    PACK_FILE.write_text(json.dumps(pk, indent=1))
    SPG = pk["recommended"]
print(f"jobs per GPU: {SPG}" + ("" if SLOTS_PER_GPU != "auto" else f" (auto; {PACK_FILE.name})"))
if not SMOKE and not LOCAL_TEST:   # fails here, in minutes, if a job has no speed (instead of in every worker)
    for j in JOBS_ALL:
        if not j.startswith("real_s"):
            bench_run.job_config(j, fam, STAGE, False, SPEEDS)

if SMOKE:              # a tiny search, then a tiny final stage (fresh runs of the first configuration, splits 0 and 1)
    run_stage("search", make_queues(SEARCH_JOBS, SPG), True)
    smoke_final = [SEARCH_JOBS[0], SEARCH_JOBS[0][:-1] + "1"]
    smoke_jobs = smoke_final + ["real_s0"]
    run_stage("final", make_queues(smoke_jobs, SPG), True)
    SHOWN = [(j, STAGE_RUNS["search"]) for j in SEARCH_JOBS] + [(j, STAGE_RUNS["final"]) for j in smoke_jobs]
else:
    run_stage(STAGE, make_queues(JOBS_ALL, SPG), False)
    SHOWN = [(j, RUNS) for j in JOBS_ALL]
for j, runs in SHOWN:                   # (job, its runs folder): in SMOKE the first job runs in both stages
    print(f"  {runs.name + '/' + j:<44} {state(j, runs)}")

## 5 · Results

**Search stage** (split 0, validation stars only): one row per configuration, best first, with the hour of training at which it peaked, why it stopped (no gain or the cap), and its rare-tier copies. The best row becomes `FINAL` in `astra/bench_grid.py`. Test scores are not computed in the search.

**Final stage**
- *Reference rows on real data* (A = oversampling, B–D = soft warping), per split.
- *One row per split*: the selected checkpoint and when it was reached, TSTR and real + synthetic balanced accuracy (± over the 15 classifier runs), the difference to row A of the same split with a 95% paired-bootstrap interval, **the rare-tier copies and memorisation ratio next to them**, fidelity and sampling time. An interval that excludes 0 is a difference the test stars can support.
- *The compute-matched view*: the TSTR test score of the best checkpoint within 5, 10, 15, 30, 60 minutes and 2, 5, 10 h of training, and at the family's own best point.
- *Conditions no training star has*: TSTR (3 classifier runs) at the selected and at the last checkpoint, for the training stars' own conditions and for new ones. A model that only copies its training curves scores well on its own conditions and badly on new ones.
- *Per-class recall*, classes from most to least common: the question of the project is the bottom rows.
- *Other sampling settings* of the selected checkpoints (3 classifier runs each): sensitivity checks, never picked by their test score.

**Figures:** validation score, FID, recall and density, rare-tier copies and the model's own losses against hours of training; the budget figure; real and generated curves with the same condition; the final classifiers' loss curves. **The four-way comparison** appears once the final outputs of the other benchmark notebooks are attached.

In [ ]:
# ---- Tables and figures from everything finished so far ----
import traceback
from astra import bench_report as rep

pd.set_option("display.width", 250)


def safely(what, fn):
    """A failing table or figure must not cost the session's output: report it and go on."""
    try:
        return fn()
    except Exception:
        print(f"\n!! {what} failed:\n{traceback.format_exc()}")


def show(title, make, name, index=False):
    table = safely(title, make)
    if table is not None and len(table):
        table.to_csv(TABLES / name, index=index)
        print(f"\n{title}\n{table.to_string(index=index)}")


mine = lambda d: {j: r for j, r in d.items() if Path(r["_from"]).resolve() == RESULTS.resolve()}
res = rep.collect([WORK, INPUT], RESULTS.name)                          # final stage: this notebook's + attached
search = rep.collect([WORK, INPUT], RESULTS.name, stage="search")       # search stage (validation only)
own, own_search = mine(res), mine(search)
models = {j: r for j, r in own.items() if r["family"] != "real"}
if not own and not own_search:
    print("(no finished jobs yet)")

# ---- the search: every configuration on split 0, validation stars only ----
show(f"{rep.FAMILIES[FAMILY]}: the search on split 0 (validation TSTR balanced accuracy %, best first)",
     lambda: rep.search_table(own_search), "search.csv")
safely("search curves", lambda: rep.plot_search(own_search, FIGS / "search_curves.png"))

# ---- the final stage: the winner on five splits, test scores ----
show(f"Reference rows on real data ({DATASET}), test balanced accuracy %", lambda: rep.real_table(own), "real_rows.csv")
show(f"{rep.FAMILIES[FAMILY]} on {DATASET}: selected checkpoints, test scores % (copies = memorisation)",
     lambda: rep.job_table(own), "jobs.csv")
show("If training had stopped after 5, 10, 15, 30, 60 min or 2, 5, 10 h on one T4 (best checkpoint within, test TSTR)",
     lambda: rep.budget_table(own), "budgets.csv")
show("Conditions no training star has (one star's phases, another's amplitude), test TSTR, 3 runs each",
     lambda: rep.novel_table(own), "novel_conditions.csv")
show("Per-class test recall %, mean over the splits", lambda: rep.class_table(own), "per_class_recall.csv", True)
show("Other sampling settings of the selected checkpoints (3 runs each)", lambda: rep.sweep_table(own), "sampling_sweeps.csv")

safely("training curves", lambda: rep.plot_training(models, FIGS / "training_curves.png"))
for j in sorted(models):
    safely(f"samples of {j}", lambda: rep.plot_samples(
        STAGE_RUNS["final"] / j, Z, f"{j}: real vs generated, same condition (selected checkpoint)",
        FIGS / f"samples_{j}.png"))
safely("classifier curves", lambda: rep.plot_classifiers(own, FIGS / "classifier_curves.png"))
safely("sampling sweeps", lambda: rep.plot_sweeps(models, FIGS / "sampling_sweeps.png"))
safely("budget figure", lambda: rep.plot_budget(own, FIGS / "budgets.png"))
safely("headline figure", lambda: rep.plot_headline(own, FIGS / "headline.png"))

# ---- the four-way comparison (needs the other benchmark notebooks' outputs under Add Input) ----
others = sorted({r["family"] for r in res.values()} - {FAMILY, "real"})
if others:
    print(f"\n==== comparison with attached results: {[rep.FAMILIES.get(f, f) for f in others]} ====")
    hashes = sorted({(r["seed"], r["data_hash"]) for r in res.values()})
    if len(hashes) != len({s for s, _ in hashes}):
        print(f"!! the attached results do not all use the same data and split (split, hash): {hashes}\n"
              "   rows are still listed, but results with different hashes are not paired")
    feats = sorted({(r["seed"], r.get("feat_hash")) for r in res.values() if r.get("feat_hash")})
    if len(feats) != len({s for s, _ in feats}):
        print("note: the notebooks trained their own feature CNNs and these differ, so FID, density and recall of "
              "different notebooks are measured in slightly different feature spaces. "
              "The classifier scores (same test stars) are not affected.")
    show("All families: mean ± std over the splits", lambda: rep.family_table(res), "comparison_families.csv")
    show("All families: the compute-matched view", lambda: rep.budget_table(res), "comparison_budgets.csv")
    show("All families: conditions no training star has", lambda: rep.novel_table(res), "comparison_novel_conditions.csv")
    show("Per-class test recall %, all families", lambda: rep.class_table(res), "comparison_per_class.csv", True)
    show("Paired differences between families (balanced accuracy, % points)", lambda: rep.family_pairs(res),
         "comparison_pairs.csv")
    split0 = {j: r for j, r in res.items() if r["family"] != "real" and r["seed"] == 0}
    safely("comparison curves", lambda: rep.plot_training(split0, FIGS / "comparison_training_split0.png"))
    safely("comparison budgets", lambda: rep.plot_budget(res, FIGS / "comparison_budgets.png"))
    safely("comparison headline", lambda: rep.plot_headline(res, FIGS / "comparison_headline.png"))
    safely("comparison per class", lambda: rep.plot_class_bars(res, FIGS / "comparison_per_class.png"))
    safely("comparison samples", lambda: rep.plot_family_samples(res, Z, FIGS / "comparison_samples_split0.png"))
    show("Fidelity of every job's saved samples in ONE feature space (this notebook's feature CNNs)",
         lambda: rep.fidelity_one_space(res, NPZ, RUNS, DEV, log=lambda *a: None), "comparison_fidelity_one_space.csv")
elif STAGE == "final" or SMOKE:
    print("\n(four-way comparison: attach the other benchmark notebooks' outputs to see it)")

In [ ]:
# ---- Deliverables check: is everything saved that the comparison and the report need? ----
load = lambda p: json.loads(p.read_text()) if p.exists() else None
need = [NPZ, TABLES / "split_counts.csv", FIGS / "data_examples.png", FIGS / "feature_cnn_curves.png"]
need += [RUNS / f"feature_cnn_s{s}.pt" for s in SEEDS]
lacking = []                                    # results that exist but miss a part
summary = {"session": SESSION, "smoke": SMOKE, "stage": STAGE, "family": FAMILY, "dataset": DATASET,
           "slots_per_gpu": SPG, "speeds": SPEEDS, "hours": round((time.time() - RUN_STARTED) / 3600, 2), "jobs": {}}
for j, runs in SHOWN:
    st = load(runs / j / "status.json") or {"state": "not started"}
    summary["jobs"][f"{runs.name}/{j}"] = st
    if st["state"] != "done":
        continue
    need += [RESULTS / f"{j}.json", runs / j / "config.json", runs / j / "log.txt"]
    if j.startswith("real_s"):
        need.append(RESULTS / f"{j}_pred.npz")
    elif st.get("stage") == "search":
        need += [runs / j / f for f in ("search.json", "metrics.json", "ckpts.json", "last.pt")]
    else:
        need += [runs / j / f for f in ("final.json", "metrics.json", "samples.npz", "train_curve.csv")]
        need += [RESULTS / f"{j}_pred.npz", RESULTS / f"{j}_samples.npz"]
        if "novel_selected" not in (load(runs / j / "final.json") or {}).get("parts", {}):
            lacking.append(f"{runs.name}/{j}: the novel-conditions scores")
states = [(j.split("/")[-1], s.get("stage"), s["state"]) for j, s in summary["jobs"].items()]
if any(st == "done" and stage == "search" for j, stage, st in states):
    need.append(TABLES / "search.csv")
if any(st == "done" and j.startswith("real_s") for j, _, st in states):
    need.append(TABLES / "real_rows.csv")
if any(st == "done" and stage == "final" and not j.startswith("real_s") for j, stage, st in states):
    need += [TABLES / "jobs.csv", TABLES / "budgets.csv", FIGS / "training_curves.png"]
missing = [str(p.relative_to(WORK)) for p in need if not p.exists()] + lacking
json.dump(summary, open(TABLES / f"session{SESSION}_{STAGE}_summary.json", "w"), indent=1, default=str)
print(json.dumps(summary["jobs"], indent=1, default=str))
out_gb = sum(f.stat().st_size for f in WORK.rglob("*") if f.is_file()) / 1e9
print(f"\noutput size {out_gb:.2f} GB (Kaggle keeps up to 20 GB)")
print("ALL DELIVERABLES PRESENT" if not missing else f"MISSING: {missing}")
left = [j for j, stage, st in states if st != "done" or (not SMOKE and not j.startswith("real_s") and stage not in (STAGE, "final"))]
print(f"next session: attach this version's output, set SESSION = {SESSION + 1} | {STAGE} stage unfinished:",
      left if left else "none")

# ---- one small file to send back after every session: results, tables, figures, logs (no model weights) ----
import zipfile
BUNDLE = WORK / f"report_{FAMILY}_{'smoke' if SMOKE else STAGE}_session{SESSION}.zip"
try:
    files = [f for root in (RESULTS, TABLES, FIGS) for f in root.rglob("*") if f.is_file()]
    files += [f for runs in STAGE_RUNS.values() for pat in ("*/*.json", "*/log.txt", "*/train_curve.csv", "*.json", "*.log")
              for f in runs.glob(pat)]
    with zipfile.ZipFile(BUNDLE, "w", zipfile.ZIP_DEFLATED) as z:
        for f in sorted(set(files)):
            z.write(f, f.relative_to(WORK))
    print(f"\nSEND BACK: {BUNDLE.name} ({BUNDLE.stat().st_size / 1e6:.1f} MB, in the Output tab) and this session's log")
except Exception as e:
    print(f"\n(the report bundle could not be written: {type(e).__name__}: {e}; send the folders instead)")

## 6 · Observations (to fill in after the real run)

**Does it beat real data only?** *(Placeholder: TSTR and real + synthetic balanced accuracy against row A, per split, with the paired intervals. Does an interval exclude 0 on all five splits? Which mix of real and synthetic curves did validation choose?)*

**Rare classes.** *(Placeholder: recall of the rare-tier classes for row A, TSTR and real + synthetic. This is the question ASTRA is about. Do the common classes pay for it?)*

**Fidelity and diversity.** *(Placeholder: density / precision against recall / coverage per class; FID; the amplitude error. Which classes are covered worst?)*

**Over/under-fitting and memorisation.** *(Placeholder: does the validation TSTR score peak and then fall? Does the model's loss on validation stars rise above its loss on training stars? Do the rare-tier copies rise above 5%, and when? Did selection pick an early checkpoint? Classifier curves: training against validation loss.)*

**New conditions.** *(Placeholder: TSTR for new conditions against the training stars' own, at the selected and at the last checkpoint. Does the last checkpoint, which copies, still generate useful curves for conditions it has not seen?)*

**Diffusion-specific.** *(Placeholder: score against the number of sampling steps (4, 8, 16, 32, 64) and the sampling time; does guidance (1.5, 2) help the classifier, and what does it cost in recall? Noise prediction against v-prediction and the learning rate in the search.)*

**Training length and compute.** *(Placeholder: did the runs stop by 'no gain' or at the 20 h cap? At how many hours of training on one T4 did each split peak? The compute-matched view: which family is ahead at 5, 10, 15, 30, 60 minutes and 2, 5, 10 h, and does the order change?)*

**The search.** *(Placeholder: the configurations on split 0, validation only: how far apart are they, compared with the ± of one checkpoint's score? Which knob mattered?)*

**Split-to-split spread.** *(Placeholder: how much do the scores move between splits 0–4, compared with the differences between methods?)*

**Deviations and their possible effect.** *(Placeholder: the amplitude-normalised target, the stopping rule and its caps, 3 selection runs per checkpoint, a search of six configurations on one split, mixed precision, the "ours" choices of the harness.)*